# Applicazione video: sei gesti YOLOv8 + tracce esistenti

Applica il detector gia addestrato ai frame video e legge i CSV esistenti per aggiungere contesto palla/giocatori. Non rilancia il tracking, YOLO palla/giocatori o OCR. Salva detection frame-per-frame, episodi deduplicati, report di copertura e configurazione in una directory univoca; il video annotato e opzionale e la sua indisponibilita e segnalata nel report.

Modalita contestuale opzionale: usa gli score grezzi reali delle sei classi per ripesare candidati vicini con prior temporali morbidi. Preserva sempre `class_raw`, score per classe, classe contestuale, score contestuali non calibrati, regola applicata e motivazione. Corregge solo episodi di piu frame con track/associazione affidabili, score alternativo realmente disponibile e margine raw limitato. Se il backend espone solo classe top/confidenza, il notebook dichiara `top_only`, lascia gli altri score mancanti e non corregge.

Lato geometrico non significa squadra o ruolo di rotazione. Vicinanza palla non prova contatto; un muro puo avvenire senza toccare la palla. Intervalli senza detection non dimostrano ace, errore o tocco mancato. Non dedurre prima/seconda linea dai box o battute fallose dalla posizione del contatto. Elaborazione offline, non real-time; correzioni di episodi usano anche frame futuri. Nessun download, installazione, training o elaborazione lunga all'apertura.

Valutazione opzionale separata dalle predizioni: confronta gli eventi con il CSV dei tocchi del match selezionato. Lo scout e solo ground truth, mai input a inferenza o prior. Report per la clip, non cross-validation; l'eventuale sovrapposizione con il dataset di training esterno non e verificata.

Regola di sequenza monouso: una battuta affidabile governa soltanto il primo episodio raw `ricezione`/`difesa` che la segue senza episodi gestuali intermedi rilevati. Un episodio intermedio consuma il contesto, ma non viene per questo dichiarato contatto: il CSV conserva le evidenze palla/osservazione e le relative incertezze. Raw e contestuale restano separati.

I nomi finali sono `proposte_DOM.csv`/`embedding_DOM.csv`, `proposte_SRB.csv`/`embedding_SRB.csv` e `proposte_POL.csv`/`embedding_POL.csv`, una coppia per esecuzione selezionando `OPPONENT`. Le righe rappresentano eventi stimati raggruppando le detection nel tempo e scegliendo picchi di confidence, non contatti confermati. Il modello gia addestrato viene applicato cosi com'e, senza riaddestramento o cross-validation. Le proposte usano sei score raw normalizzati a somma uno (non calibrati e senza correzioni contestuali); `conf_tocco` e la confidenza YOLO del frame scelto, non una probabilita calibrata di contatto. Squadra resta vuota perche il tracker fornisce solo il lato geometrico. Coordinate giocatore sono esportate solo da posizioni piedi calibrate e in campo; quelle palla sono proiezioni sul piano del campo di posizioni osservate, non misure certe della posizione a terra.

Revisione lato campo: durante la sola valutazione, il mapping squadra→metà campo può essere ricavato da servizi scout non dubbi e tracce dei piedi stabili presso la linea di fondo, validate con la calibrazione. Non si usa il giocatore associato alla predizione del gesto come ground truth. Servizi senza un unico track stabile, posizioni ambigue o campione incompleto non contribuiscono; se mancano ancore ripetute per un set, i lati restano sconosciuti. Lo scout non entra in inferenza o nelle correzioni contestuali.

Revisione v7: la valutazione near/far usa solo righe scout di battuta non dubbie e allineate per costruire ancore; le squadre presenti per set sono ricavate separatamente da tutte le annotazioni, senza riutilizzare tutte le righe come se fossero battute. Se le ancore geometriche ripetute non sono concordi, il lato resta sconosciuto e le metriche per metà campo non sono calcolate come se il mapping fosse disponibile.

Tolleranze di matching ground-truth configurabili: battuta annotata 1,0 s; tutti gli altri gesti 0,5 s. Il matching uno-a-uno massimizza le coppie e poi minimizza lo scarto assoluto, applicando la tolleranza alla classe del tocco annotato; una detection non puo essere abbinata due volte anche se ricade in due finestre temporali sovrapposte.
Contesto battuta-ricezione: finestra massima 3 s e correzione raw ricezione/difesa solo se battuta e risposta hanno associazioni affidabili su lati geometrici opposti e non e stato rilevato un episodio intermedio. I lati geometrici non identificano squadre o ruoli.
Attacco→muro: correzione contestuale solo quando un candidato raw attacco ha score raw muro disponibile e supportato, episodio/player associato affidabile, e un episodio attacco affidabile di un track sul lato opposto e quasi simultaneo. Il prior e morbido; senza queste evidenze o margine score utile resta raw. Non prova contatto, rete calibrata o identificazione certa del gesto.

## 1. Configurazione

Imposta `OPPONENT` a `POL`, `SRB` o `DOM`: il dizionario seleziona gli input reali e gli intervalli coperti dai rispettivi tracking. `END_SECONDS` è esclusivo. Gli intervalli preimpostati sono lunghi: prima di avviare l'inferenza, modifica l'intervallo della partita selezionata se vuoi una prova breve. Il checkpoint gesti è condiviso; le run sono salvate in cartelle univoche.

Per DOM l'utente ha confermato che `ITA_DOM_Parigi2024_30fps.mp4` nello scout e `Test_demo_ITA_DOM_Parigi2024_30fps.mp4` sono lo stesso video senza offset; il CSV originale non viene modificato. Altri nomi rimangono soggetti al controllo esatto.


In [ ]:
import os
os.environ['YOLO_AUTOINSTALL'] = 'false'
import sys
import math
import json
import csv
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from collections import defaultdict, Counter

WORKSPACE_DIR = next((p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / 'Annotazioni').is_dir() and (p / 'runs').is_dir()), None)
if WORKSPACE_DIR is None:
    raise FileNotFoundError('Avvia il notebook dalla cartella Scout_Volley_CV&NLP o imposta WORKSPACE_DIR.')
# Seleziona un solo avversario. Modifica i percorsi o l'intervallo
# nel relativo dizionario se scegli tracking o clip diversi.
OPPONENT = 'POL'  # Deliverable prioritario; 'POL' e 'SRB' sono facoltativi
MATCH_CONFIGS = {
    'POL': {
        'video': WORKSPACE_DIR / 'Annotazioni' / 'ITA_POL_VNL2024_30fps.mp4',
        'calibration': WORKSPACE_DIR / 'zone_campo_output' / 'calibrazione_ITA_POL_VNL2024_30fps.json',
        'ball_csvs': [
            WORKSPACE_DIR / 'runs' / 'ball_tracking' / 'ITA_POL_VNL2024_30fps_20261116_081527' / 'tracce.csv',
            WORKSPACE_DIR / 'runs' / 'ball_tracking' / 'ITA_POL_VNL2024_30fps_20261117_081248' / 'tracce.csv',
        ],
        'player_csv': WORKSPACE_DIR / 'runs' / 'player_tracking_jersey' / 'ITA_POL_VNL2024_30fps_20261119_084910_9bb409' / 'osservazioni.csv',
        'touch_csv': WORKSPACE_DIR / 'Annotazioni' / 'ITA_POL_VNL2024_30fps_FP_tocchi.csv',
        'start_seconds': 1150, #430,
        'end_seconds': 1200, #4948,
    },
    'SRB': {
        'video': WORKSPACE_DIR / 'Annotazioni' / 'ITA_SRB_Parigi2024_30fps.mp4',
        'calibration': WORKSPACE_DIR / 'zone_campo_output' / 'calibrazione_ITA_SRB_Parigi2024_30fps.json',
        'ball_csvs': [
            WORKSPACE_DIR / 'runs' / 'ball_tracking' / 'ITA_SRB_Parigi2024_30fps_20261118_105246' / 'tracce.csv',
        ],
        'player_csv': WORKSPACE_DIR / 'runs' / 'player_tracking_jersey' / 'ITA_SRB_Parigi2024_30fps_20261119_113428_d4dd80' / 'osservazioni.csv',
        'touch_csv': WORKSPACE_DIR / 'Annotazioni' / 'ITA_SRB_Parigi2024_30fps_FP_tocchi.csv',
        'start_seconds': 950.0,
        'end_seconds': 5982.0,
    },
    'DOM': {
        'video': WORKSPACE_DIR / 'Annotazioni' / 'Test_demo_ITA_DOM_Parigi2024_30fps.mp4',
        'calibration': WORKSPACE_DIR / 'zone_campo_output' / 'calibrazione_Test_demo_ITA_DOM_Parigi2024_30fps.json',
        'ball_csvs': [
            WORKSPACE_DIR / 'runs' / 'ball_tracking' / 'Test_demo_ITA_DOM_Parigi2024_30fps_20261118_140349' / 'tracce.csv',
        ],
        'player_csv': WORKSPACE_DIR / 'runs' / 'player_tracking_jersey' / 'Test_demo_ITA_DOM_Parigi2024_30fps_20261119_135930_a08032' / 'osservazioni.csv',
        'touch_csv': WORKSPACE_DIR / 'Annotazioni' / 'Test_demo_ITA_DOM_Parigi2024_30fps_FP_tocchi.csv',
        'start_seconds': 680.0,
        'end_seconds': 7350.0,
    },
}
if OPPONENT not in MATCH_CONFIGS:
    raise ValueError(f'Avversario {OPPONENT!r} non valido; scegliere tra {", ".join(MATCH_CONFIGS)}.')
match_config = MATCH_CONFIGS[OPPONENT]
VIDEO_PATH = match_config['video']
# L'utente ha confermato che questi due nomi DOM indicano lo stesso video
# senza tagli né offset. L'alias non si applica ad altri video o avversari.
SCOUT_VIDEO_NAME = VIDEO_PATH.name
if (OPPONENT == 'DOM'
        and VIDEO_PATH.name == 'Test_demo_ITA_DOM_Parigi2024_30fps.mp4'):
    SCOUT_VIDEO_NAME = 'ITA_DOM_Parigi2024_30fps.mp4'
CAMERA_CALIBRATION_JSON = match_config['calibration']
BALL_CSV_PATHS = list(match_config['ball_csvs'])
BALL_CONFIG_PATHS = [path.parent / 'configurazione.json' for path in BALL_CSV_PATHS]
PLAYER_CSV = match_config['player_csv']
PLAYER_CONFIG = PLAYER_CSV.parent / 'configurazione.json'
SIDE_MAPPING_PLAYER_CSV = PLAYER_CSV
SIDE_MAPPING_PLAYER_CONFIG = PLAYER_CONFIG
TOUCH_CSV = match_config['touch_csv']
START_SECONDS = match_config['start_seconds']
END_SECONDS = match_config['end_seconds']

if not BALL_CSV_PATHS:
    raise ValueError(f'Nessun CSV palla configurato per {OPPONENT}.')
required_match_inputs = {
    'video': VIDEO_PATH, 'calibrazione campo': CAMERA_CALIBRATION_JSON,
    'CSV giocatori': PLAYER_CSV, 'configurazione giocatori': PLAYER_CONFIG,
    'CSV tocchi scout': TOUCH_CSV,
}
required_match_inputs.update({f'CSV palla {i}': path for i, path in enumerate(BALL_CSV_PATHS, 1)})
required_match_inputs.update({f'configurazione palla {i}': path for i, path in enumerate(BALL_CONFIG_PATHS, 1)})
missing_match_inputs = [f'{label}: {path}' for label, path in required_match_inputs.items() if not path.is_file()]
if missing_match_inputs:
    raise FileNotFoundError(f'Input richiesti mancanti per {OPPONENT}:\n' + '\n'.join(missing_match_inputs))

# Verifica il nome video delle annotazioni prima di avviare l'inferenza.
with TOUCH_CSV.open(encoding='utf-8-sig', newline='') as stream:
    scout_reader = csv.DictReader(stream)
    if not scout_reader.fieldnames or 'video' not in scout_reader.fieldnames:
        raise ValueError(f'Colonna video mancante nel CSV tocchi {TOUCH_CSV}.')
    scout_video_names = sorted({
        str(row.get('video') or '').strip() for row in scout_reader
        if str(row.get('video') or '').strip()
    })
if SCOUT_VIDEO_NAME.lower() not in {name.lower() for name in scout_video_names}:
    raise ValueError(
        f'Il CSV tocchi {TOUCH_CSV.name} riporta video {scout_video_names}, '
        f'mentre il nome scout atteso per {VIDEO_PATH.name} e {SCOUT_VIDEO_NAME}. '
        'Verificare che siano la stessa registrazione e allineati prima di procedere; '
        'nessuna inferenza e stata avviata.'
    )

GESTURE_CHECKPOINT = WORKSPACE_DIR / 'runs' / 'sei_gesti_dataset' / '20261120_223420_d3008fd1' / 'training' / 'weights' / 'best.pt'
EXPECTED_CHECKPOINT_SHA256 = 'eca3b8ed4f34d25e431e49409fe1b7cf090c11a6b055ae7cae253684fccbec44'
GROUND_TRUTH_TEAM_SIDE_BY_SET = {}
# Configura mapping squadra/set solo se verificati indipendentemente.
PREDICTION_TRACK_SIDE_TO_CAMERA_HALF = {}
SIDE_ANCHOR_LOOKBACK_SECONDS = 1.0
SIDE_ANCHOR_MIN_FRAMES = 5
SIDE_ANCHOR_MIN_DURATION_SECONDS = 0.15
SIDE_ANCHOR_MAX_FOOT_Y_RANGE_METERS = 0.5
SIDE_ANCHOR_MAX_LAST_OBSERVATION_AGE_SECONDS = 0.5
SIDE_ANCHOR_MIN_REPEATED_SERVES_PER_TEAM_SET = 2
MATCH_TOLERANCE_SECONDS = 0.5
SERVE_MATCH_TOLERANCE_SECONDS = 1.0
MATCH_TOLERANCE_BY_CLASS = {
    'battuta': SERVE_MATCH_TOLERANCE_SECONDS,
    **{name: MATCH_TOLERANCE_SECONDS for name in ('ricezione', 'alzata', 'attacco', 'muro', 'difesa')},
}
print(MATCH_TOLERANCE_BY_CLASS)
RALLY_WINDOW_MARGIN_SECONDS = 0.5
OUTPUT_ROOT = WORKSPACE_DIR / 'runs' / 'gesture_video'

# Intervallo sorgente [START_SECONDS, END_SECONDS); verificare prima di eseguire.
FRAME_STRIDE = 1
IMAGE_SIZE = 640
DETECTION_CONFIDENCE = 0.25
NMS_IOU = 0.50
MAX_DETECTIONS = 100
REQUEST_GPU = True
RUN_INFERENCE = True  # Non eseguire durante la configurazione; impostare False per ispezionare senza inferenza
EMBEDDING_EXPORT_ENABLED = True

# Associazione spaziale e prior temporali solo per diagnostica/ordinamento.
PLAYER_IOU_MIN = 0.05
PLAYER_CENTER_MAX_HEIGHTS = 0.75
PLAYER_AMBIGUITY_RATIO = 1.20
BALL_ASSOCIATION_MAX_PX = 120.0
EPISODE_GAP_SECONDS = 1.0
# Euristiche non tarate sullo scout: gap tra detection e separazione/prominenza dei picchi.
TOUCH_GROUP_MAX_GAP_SECONDS = 0.20
TOUCH_PEAK_MIN_DISTANCE_SECONDS = 0.30
TOUCH_PEAK_MIN_PROMINENCE = 0.05
SERVE_RESPONSE_WINDOW_SECONDS = 3.0
MIN_SERVE_CONFIDENCE_FOR_RECEIVE_DIG = 0.50
SAME_SIDE_ATTACK_WINDOW_SECONDS = 2.0
ATTACK_RESPONSE_WINDOW_SECONDS = 2.0
RECEIVE_DIG_TO_SET_FACTOR = 1.08
ATTACK_TO_DEFENSE_FACTOR = 1.05
SIMULTANEOUS_WINDOW_SECONDS = 0.20
ATTACK_TO_BLOCK_FACTOR = 1.15
SOFT_RULE_WEIGHT = 0.15

# Correzione di classe solo offline e solo se raw head fornisce i sei score.
ENABLE_CONTEXTUAL_CLASS_CORRECTION = True
CONTEXT_CANDIDATE_SCORE_FLOOR = 0.05
CONTEXT_MAX_RAW_SCORE_GAP = 0.08
MIN_EPISODE_DETECTIONS_FOR_CORRECTION = 2
SERVE_RESPONSE_RECEIVE_FACTOR = 1.10
SERVE_RESPONSE_DIG_FACTOR = 1.03
SET_TO_ATTACK_FACTOR = 1.08
SAME_SIDE_ATTACK_FACTOR = 0.92
CONTACT_PROXIMITY_DIAGNOSTIC_PX = 35.0

SIDE_ANCHOR_MIN_FRAMES = 5
SIDE_ANCHOR_MIN_DURATION_SECONDS = 0.15
SIDE_ANCHOR_MAX_FOOT_Y_RANGE_METERS = 0.5
SIDE_ANCHOR_MAX_LAST_OBSERVATION_AGE_SECONDS = 1.0
SIDE_ANCHOR_LOOKBACK_SECONDS = 10.0

CLASS_NAMES = ['battuta', 'ricezione', 'alzata', 'attacco', 'muro', 'difesa']
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / f'{VIDEO_PATH.stem}_{RUN_ID}'
print('Avversario:', OPPONENT)
print('Workspace:', WORKSPACE_DIR)
print('Video:', VIDEO_PATH)
print('Checkpoint:', GESTURE_CHECKPOINT)
print('Tracce palla (merge in memoria):', BALL_CSV_PATHS)
print('Tracce giocatori:', PLAYER_CSV)
print(f'Intervallo: [{START_SECONDS}, {END_SECONDS}) secondi; stride={FRAME_STRIDE}; run={RUN_ID}')

{'battuta': 1.0, 'ricezione': 0.5, 'alzata': 0.5, 'attacco': 0.5, 'muro': 0.5, 'difesa': 0.5}
Avversario: DOM
Workspace: C:\Users\SavarDar\OneDrive - Electrolux\Desktop\Scout_Volley_CV&NLP
Video: C:\Users\SavarDar\OneDrive - Electrolux\Desktop\Scout_Volley_CV&NLP\Annotazioni\Test_demo_ITA_DOM_Parigi2024_30fps.mp4
Checkpoint: C:\Users\SavarDar\OneDrive - Electrolux\Desktop\Scout_Volley_CV&NLP\runs\sei_gesti_dataset\20261120_223420_d3008fd1\training\weights\best.pt
Tracce palla (merge in memoria): [WindowsPath('C:/Users/SavarDar/OneDrive - Electrolux/Desktop/Scout_Volley_CV&NLP/runs/ball_tracking/Test_demo_ITA_DOM_Parigi2024_30fps_20261118_140349/tracce.csv')]
Tracce giocatori: C:\Users\SavarDar\OneDrive - Electrolux\Desktop\Scout_Volley_CV&NLP\runs\player_tracking_jersey\Test_demo_ITA_DOM_Parigi2024_30fps_20261119_135930_a08032\osservazioni.csv
Intervallo: [680.0, 7350.0) secondi; stride=1; run=20261121_184436_4aba0e59


## 2. Preflight: modello, video, CSV e sincronizzazione

Controlla versione classi e hash checkpoint, apertura e metadata video, schema/valori CSV, riferimenti video nei file di configurazione tracking, FPS/frame-index/timestamp, intervallo coperto e frame-size quando dichiarato. Le righe palla `predetta` restano marcate come stime Kalman, mai promosse a osservazioni. Per i giocatori sono utilizzate solo righe associate a detection correnti; nessuna previsione di traccia viene trattata come osservazione.

Se l'intervallo scelto non e coperto da entrambe le tracce, la cella si ferma: non inventa un'associazione né presume copertura.

In [2]:
import importlib.util
missing = [name for name in ('torch', 'torchvision', 'ultralytics', 'cv2', 'numpy', 'pandas')
           if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f'Dipendenze mancanti in {sys.executable}: {missing}. Nessuna installazione automatica.')
import torch
import torchvision
import ultralytics
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import inspect
try:
    from ultralytics.utils.nms import non_max_suppression as YOLO_NMS
    from ultralytics.utils.ops import scale_boxes
    _nms_parameters = inspect.signature(YOLO_NMS).parameters
    RAW_SCORE_VECTOR_SUPPORTED = (hasattr(YOLO, '_smart_load')
                                  and 'return_idxs' in _nms_parameters
                                  and 'multi_label' in _nms_parameters)
    RAW_SCORE_FALLBACK_REASON = None if RAW_SCORE_VECTOR_SUPPORTED else (
        'Ultralytics NMS non espone return_idxs/multi_label richiesti per collegare lo score vector.')
except ImportError as exc:
    YOLO_NMS = None
    scale_boxes = None
    RAW_SCORE_VECTOR_SUPPORTED = False
    RAW_SCORE_FALLBACK_REASON = f'API score raw indisponibile: {type(exc).__name__}: {exc}'
from ultralytics.utils import AUTOINSTALL
if AUTOINSTALL:
    raise RuntimeError('Ultralytics ha AUTOINSTALL attivo. Riavvia il kernel; questo notebook non installa pacchetti.')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_file(path, label):
    if not Path(path).is_file():
        raise FileNotFoundError(f'{label} mancante: {path}')

for path, label in ((VIDEO_PATH, 'Video'), (GESTURE_CHECKPOINT, 'Checkpoint'),
                    (PLAYER_CSV, 'CSV giocatori'), (PLAYER_CONFIG, 'Configurazione giocatori')):
    require_file(path, label)
if not BALL_CSV_PATHS or len(BALL_CSV_PATHS) != len(BALL_CONFIG_PATHS):
    raise ValueError('Configurare almeno un CSV palla e un file di configurazione corrispondente.')
for index, (csv_path, config_path) in enumerate(zip(BALL_CSV_PATHS, BALL_CONFIG_PATHS), start=1):
    require_file(csv_path, f'CSV palla {index}')
    require_file(config_path, f'Configurazione palla {index}')
CHECKPOINT_SHA256 = sha256_file(GESTURE_CHECKPOINT)
if EXPECTED_CHECKPOINT_SHA256 and CHECKPOINT_SHA256 != EXPECTED_CHECKPOINT_SHA256:
    raise ValueError(f'Hash checkpoint inatteso: {CHECKPOINT_SHA256}. Verificare e aggiornare esplicitamente la configurazione.')
MODEL = YOLO(str(GESTURE_CHECKPOINT))
MODEL_NAMES = {int(key): str(value) for key, value in MODEL.names.items()}
if MODEL.task != 'detect' or MODEL_NAMES != dict(enumerate(CLASS_NAMES)):
    raise ValueError(f'Checkpoint/task/classi inattese: task={MODEL.task}, names={MODEL_NAMES}')

ball_cfgs = [json.loads(path.read_text(encoding='utf-8')) for path in BALL_CONFIG_PATHS]
player_cfg = json.loads(PLAYER_CONFIG.read_text(encoding='utf-8'))
player_settings = player_cfg.get('settings', {})
video_name = VIDEO_PATH.name.lower()
for index, cfg in enumerate(ball_cfgs, start=1):
    if Path(cfg.get('video', '')).resolve() != VIDEO_PATH.resolve():
        raise ValueError(f'Il segmento palla {index} non appartiene esattamente al video selezionato.')
    if cfg.get('source_fps') is not None and not math.isclose(
            float(cfg['source_fps']), float(ball_cfgs[0].get('source_fps', cfg['source_fps'])),
            abs_tol=0.01):
        raise ValueError('I segmenti CSV palla hanno FPS sorgente incompatibili.')
ball_starts = [float(cfg.get('start_seconds', cfg.get('start')))
               for cfg in ball_cfgs if cfg.get('start_seconds', cfg.get('start')) is not None]
ball_ends = [float(cfg.get('end_seconds', cfg.get('end')))
             for cfg in ball_cfgs if cfg.get('end_seconds', cfg.get('end')) is not None]
ball_cfg = {
    'video': ball_cfgs[0].get('video'),
    'source_fps': ball_cfgs[0].get('source_fps'),
    'start_seconds': min(ball_starts) if ball_starts else None,
    'end_seconds': max(ball_ends) if ball_ends else None,
    'segments': ball_cfgs,
}
if Path(player_settings.get('video', '')).resolve() != VIDEO_PATH.resolve():
    raise ValueError('Il CSV giocatori non appartiene esattamente al video selezionato.')
if player_cfg.get('status') not in (None, 'complete'):
    raise ValueError(f"La run giocatori non e completa: {player_cfg.get('status')}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise OSError(f'Video non apribile: {VIDEO_PATH}')
VIDEO_FPS = float(cap.get(cv2.CAP_PROP_FPS))
VIDEO_FRAME_COUNT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
if not math.isfinite(VIDEO_FPS) or VIDEO_FPS <= 0 or VIDEO_FRAME_COUNT <= 0:
    raise ValueError(f'Metadata video non validi: fps={VIDEO_FPS}, frames={VIDEO_FRAME_COUNT}')
if FRAME_STRIDE < 1 or int(FRAME_STRIDE) != FRAME_STRIDE:
    raise ValueError('FRAME_STRIDE deve essere un intero positivo.')
if not (math.isfinite(START_SECONDS) and math.isfinite(END_SECONDS)
        and 0 <= START_SECONDS < END_SECONDS <= VIDEO_FRAME_COUNT / VIDEO_FPS):
    raise ValueError(f'Intervallo [inizio,fine) fuori durata video ({VIDEO_FRAME_COUNT/VIDEO_FPS:.3f}s).')
if ball_cfg.get('source_fps') is not None and not math.isclose(float(ball_cfg['source_fps']), VIDEO_FPS, abs_tol=0.01):
    raise ValueError('FPS del CSV palla diverso dai metadata video.')
if player_settings.get('source_fps') is not None and not math.isclose(float(player_settings['source_fps']), VIDEO_FPS, abs_tol=0.01):
    raise ValueError('FPS del CSV giocatori diverso dai metadata video.')
for settings, label in ((player_cfg, 'giocatori'), (player_settings, 'giocatori'), (ball_cfg, 'palla')):
    size = settings.get('frame_size')
    if size and tuple(map(int, size)) != (VIDEO_WIDTH, VIDEO_HEIGHT):
        raise ValueError(f'Frame size {label} {size} incompatibile con video {(VIDEO_WIDTH, VIDEO_HEIGHT)}.')
calibration_path = Path(CAMERA_CALIBRATION_JSON)
if not calibration_path.is_file():
    raise FileNotFoundError(f'Calibrazione campo non trovata: {calibration_path}')
calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
if calibration.get('confirmed_by_user') is not True:
    raise ValueError('La calibrazione campo non risulta confermata.')
if Path(calibration['video']).resolve() != VIDEO_PATH.resolve():
    raise ValueError('La calibrazione campo appartiene a un altro video.')
if (calibration.get('image_width_px'), calibration.get('image_height_px')) != (VIDEO_WIDTH, VIDEO_HEIGHT):
    raise ValueError('La risoluzione della calibrazione non coincide con quella del video.')
if not np.allclose([calibration.get('court_width_m'), calibration.get('court_length_m')], [9., 18.]):
    raise ValueError('La calibrazione deve usare larghezza 9 m e lunghezza 18 m.')
CAMERA_H_IMAGE_TO_WORLD = np.asarray(calibration.get('H_image_to_world'), dtype=np.float64)
if CAMERA_H_IMAGE_TO_WORLD.shape != (3, 3) or not np.isfinite(CAMERA_H_IMAGE_TO_WORLD).all():
    raise ValueError('H_image_to_world non valida nella calibrazione.')
_center_denominator = float(CAMERA_H_IMAGE_TO_WORLD[2] @ np.array([VIDEO_WIDTH / 2, VIDEO_HEIGHT / 2, 1.]))
if not math.isfinite(_center_denominator) or abs(_center_denominator) < 1e-8:
    raise ValueError('Il centro del frame cade sull’orizzonte proiettivo della calibrazione.')
CAMERA_H_IMAGE_TO_WORLD = CAMERA_H_IMAGE_TO_WORLD / _center_denominator
_calibration_corners = np.asarray(calibration.get('corners_image_px'), dtype=np.float64)
if _calibration_corners.shape != (4, 2) or not np.isfinite(_calibration_corners).all():
    raise ValueError('I corner immagine della calibrazione non sono validi.')
_corner_h = (CAMERA_H_IMAGE_TO_WORLD @ np.column_stack([_calibration_corners, np.ones(4)]).T).T
if (np.any(_corner_h[:, 2] < 1e-4)
        or not np.allclose(_corner_h[:, :2] / _corner_h[:, 2, None],
                           [[0., 0.], [9., 0.], [9., 18.], [0., 18.]], atol=0.05)):
    raise ValueError('I corner non sono coerenti con il semispazio o gli assi del campo attesi.')
if not 0 <= DETECTION_CONFIDENCE <= 1 or not 0 < NMS_IOU <= 1:
    raise ValueError('Soglie confidenza/IoU fuori range.')

BALL_COLUMNS = {'timestamp_seconds', 'frame_index', 'track_id', 'confidence', 'x_px', 'y_px', 'position_type',
                'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'}
PLAYER_COLUMNS = {'frame_index', 'timestamp_seconds', 'row_type', 'class_name', 'confidence', 'bbox_x1',
                  'bbox_y1', 'bbox_x2', 'bbox_y2', 'track_id', 'association_status', 'output_included', 'team_side'}
ball_parts = []
BALL_SEGMENT_AUDIT = []
for csv_path, cfg in zip(BALL_CSV_PATHS, ball_cfgs):
    part = pd.read_csv(csv_path)
    if not BALL_COLUMNS.issubset(part.columns):
        raise ValueError(f'Colonne mancanti in {csv_path}: {sorted(BALL_COLUMNS - set(part.columns))}')
    if part.empty:
        raise ValueError(f'CSV palla vuoto: {csv_path}')
    for column in ('frame_index', 'timestamp_seconds'):
        values = pd.to_numeric(part[column], errors='coerce')
        if values.isna().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
            raise ValueError(f'{csv_path}: {column} contiene valori non validi.')
        part[column] = values
    alignment = (part['timestamp_seconds'].to_numpy() * VIDEO_FPS
                 - part['frame_index'].to_numpy())
    if np.quantile(np.abs(alignment), 0.99) > 0.51:
        raise ValueError(f'{csv_path}: frame e timestamp non sincronizzati con FPS={VIDEO_FPS}.')
    part['_source_csv'] = str(csv_path.resolve())
    BALL_SEGMENT_AUDIT.append({
        'csv': str(csv_path.resolve()), 'rows': int(len(part)),
        'config_start_seconds': cfg.get('start_seconds', cfg.get('start')),
        'config_end_seconds_exclusive': cfg.get('end_seconds', cfg.get('end')),
        'min_frame_with_row': int(part['frame_index'].min()),
        'max_frame_with_row': int(part['frame_index'].max()),
        'min_timestamp_with_row': float(part['timestamp_seconds'].min()),
        'max_timestamp_with_row': float(part['timestamp_seconds'].max()),
        'position_types': part['position_type'].value_counts().to_dict(),
    })
    ball_parts.append(part)
ball_df = pd.concat(ball_parts, ignore_index=True)
_ball_rows_before_exact_dedup = len(ball_df)
ball_df = ball_df.drop_duplicates(subset=sorted(BALL_COLUMNS), keep='first')
BALL_EXACT_DUPLICATE_ROWS_DROPPED = _ball_rows_before_exact_dedup - len(ball_df)
ball_df = ball_df.sort_values(['frame_index', 'timestamp_seconds'], kind='stable').reset_index(drop=True)
player_df = pd.read_csv(PLAYER_CSV, low_memory=False)
if not PLAYER_COLUMNS.issubset(player_df.columns):
    raise ValueError(f'Colonne CSV giocatori mancanti: {sorted(PLAYER_COLUMNS - set(player_df.columns))}')
if ball_df.empty or player_df.empty:
    raise ValueError('Uno dei CSV tracce e vuoto.')
for name, frame in (('palla', ball_df), ('giocatori', player_df)):
    for col in ('frame_index', 'timestamp_seconds'):
        values = pd.to_numeric(frame[col], errors='coerce')
        if values.isna().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
            raise ValueError(f'{name}: {col} contiene valori non validi.')
    delta = (pd.to_numeric(frame['timestamp_seconds']).to_numpy() * VIDEO_FPS
             - pd.to_numeric(frame['frame_index']).to_numpy())
    if np.quantile(np.abs(delta), 0.99) > 0.51:
        raise ValueError(f'{name}: frame_index e timestamp_seconds non sono sincronizzati con FPS={VIDEO_FPS}.')
if not player_df['row_type'].eq('detection_yolo').all():
    raise ValueError('Schema giocatori inatteso: sono ammesse solo righe detection_yolo osservate.')
if not ball_df['position_type'].isin(['osservata', 'predetta']).all():
    raise ValueError('position_type palla non riconosciuto: non classificare righe senza evidenza osservata/predetta.')

def compute_frame_range(start, end, fps, frame_count, stride):
    values = (start, end, fps)
    if not all(math.isfinite(float(value)) for value in values) or fps <= 0:
        raise ValueError('Intervallo/FPS non validi.')
    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 1:
        raise ValueError('Stride deve essere intero positivo.')
    if not 0 <= start < end <= frame_count / fps:
        raise ValueError('Intervallo oltre durata video.')
    first = int(math.ceil(start * fps - 1e-9))
    exclusive = int(math.ceil(end * fps - 1e-9))
    if first >= exclusive:
        raise ValueError('Intervallo privo di frame sorgente.')
    return first, exclusive

START_FRAME, END_FRAME_EXCLUSIVE = compute_frame_range(
    START_SECONDS, END_SECONDS, VIDEO_FPS, VIDEO_FRAME_COUNT, FRAME_STRIDE)
if START_FRAME >= END_FRAME_EXCLUSIVE:
    raise ValueError('L’intervallo selezionato non contiene frame alla FPS sorgente.')
for config, label in ((ball_cfg, 'palla'), (player_settings, 'giocatori')):
    start_meta, end_meta = config.get('start_seconds', config.get('start')), config.get('end_seconds', config.get('end'))
    if start_meta is not None and START_SECONDS < float(start_meta) - 1 / VIDEO_FPS:
        message = f'Intervallo oltre inizio copertura CSV {label}: {start_meta}s.'
        if label == 'palla':
            warnings.warn(message + ' Il contesto palla mancante resta non osservato; le proposte raw non sono filtrate.')
        else:
            raise ValueError(message)
    if end_meta is not None and END_SECONDS > float(end_meta) + 1 / VIDEO_FPS:
        message = f'Intervallo oltre fine copertura CSV {label}: {end_meta}s.'
        if label == 'palla':
            warnings.warn(message + ' Il contesto palla mancante resta non osservato; le proposte raw non sono filtrate.')
        else:
            raise ValueError(message)
expected_first = player_cfg.get('first_frame_expected')
expected_last = player_cfg.get('last_frame_expected')
if expected_first is not None and START_FRAME < int(expected_first):
    raise ValueError(f'Il CSV giocatori non copre il frame iniziale {START_FRAME}.')
if expected_last is not None and END_FRAME_EXCLUSIVE - 1 > int(expected_last):
    raise ValueError(f'Il CSV giocatori non copre l’intervallo fino al frame {END_FRAME_EXCLUSIVE-1}.')
for frame, label in ((ball_df, 'palla'), (player_df, 'giocatori')):
    relevant = frame[(frame['frame_index'] >= START_FRAME) & (frame['frame_index'] < END_FRAME_EXCLUSIVE)]
    if relevant.empty:
        raise ValueError(f'Nessuna riga CSV {label} nell’intervallo selezionato.')
    if relevant['frame_index'].min() > START_FRAME + 1 or relevant['frame_index'].max() < END_FRAME_EXCLUSIVE - 2:
        warnings.warn(f'Copertura {label} incompleta vicino ai bordi dell’intervallo; controllare il report.')

DEVICE = 0 if REQUEST_GPU and torch.cuda.is_available() else 'cpu'
if RAW_SCORE_VECTOR_SUPPORTED:
    print('Score per-classe: estrazione abilitata dal tensor raw dell\'head YOLO; NMS restituisce gli indici anchor.')
else:
    print('FALLBACK TOP-ONLY esplicito:', RAW_SCORE_FALLBACK_REASON,
          '| le correzioni contestuali saranno disabilitate.')
print(f'Checkpoint detect {MODEL_NAMES}; SHA256={CHECKPOINT_SHA256}')
print(f'Video {VIDEO_WIDTH}x{VIDEO_HEIGHT}, FPS={VIDEO_FPS:g}, frame={VIDEO_FRAME_COUNT}, durata={VIDEO_FRAME_COUNT/VIDEO_FPS:.3f}s')
BALL_SEGMENT_AUDIT.sort(key=lambda item: (
    math.inf if item['config_start_seconds'] is None else float(item['config_start_seconds']),
    item['csv']))
BALL_SEGMENT_BOUNDARY_AUDIT = []
for previous, current in zip(BALL_SEGMENT_AUDIT, BALL_SEGMENT_AUDIT[1:]):
    previous_end = previous['config_end_seconds_exclusive']
    current_start = current['config_start_seconds']
    if previous_end is not None and current_start is not None:
        interval_delta = float(current_start) - float(previous_end)
        if interval_delta < -1 / VIDEO_FPS:
            warnings.warn(f'Intervalli configurati CSV palla sovrapposti di {-interval_delta:.3f}s.')
        elif interval_delta > 1 / VIDEO_FPS:
            warnings.warn(f'Gap tra intervalli configurati CSV palla: {interval_delta:.3f}s.')
    row_gap = current['min_frame_with_row'] - previous['max_frame_with_row'] - 1
    BALL_SEGMENT_BOUNDARY_AUDIT.append({
        'previous_csv': previous['csv'], 'next_csv': current['csv'],
        'config_interval_delta_seconds': interval_delta if previous_end is not None and current_start is not None else None,
        'row_gap_frames': max(0, int(row_gap)),
        'row_ranges_overlap': bool(row_gap < 0),
    })
print(f'CSV palla uniti: {len(ball_df)} righe da {len(BALL_CSV_PATHS)} file; '
      f'tipi={ball_df.position_type.value_counts().to_dict()}; '
      f'duplicati identici rimossi={BALL_EXACT_DUPLICATE_ROWS_DROPPED}')
print('Audit segmenti palla:', json.dumps(BALL_SEGMENT_AUDIT, ensure_ascii=False))
print('Audit confini palla:', json.dumps(BALL_SEGMENT_BOUNDARY_AUDIT, ensure_ascii=False))
print(f'CSV giocatori: {len(player_df)} righe, classi={player_df.class_name.value_counts().to_dict()}')
print(f'Frame richiesti [{START_FRAME},{END_FRAME_EXCLUSIVE}), stride={FRAME_STRIDE}, device={DEVICE}')
print('Lettura/merge CSV palla solo in memoria; nessuna inferenza video/tracking eseguita in questa cella.')

C:\Users\SavarDar\AppData\Local\Temp\ipykernel_27484\4123966070.py:239: UserWarning: Copertura palla incompleta vicino ai bordi dell’intervallo; controllare il report.
  warnings.warn(f'Copertura {label} incompleta vicino ai bordi dell’intervallo; controllare il report.')


Score per-classe: estrazione abilitata dal tensor raw dell'head YOLO; NMS restituisce gli indici anchor.
Checkpoint detect {0: 'battuta', 1: 'ricezione', 2: 'alzata', 3: 'attacco', 4: 'muro', 5: 'difesa'}; SHA256=eca3b8ed4f34d25e431e49409fe1b7cf090c11a6b055ae7cae253684fccbec44
Video 1280x720, FPS=30, frame=221230, durata=7374.333s
CSV palla uniti: 103487 righe da 1 file; tipi={'osservata': 85437, 'predetta': 18050}; duplicati identici rimossi=0
Audit segmenti palla: [{"csv": "C:\\Users\\SavarDar\\OneDrive - Electrolux\\Desktop\\Scout_Volley_CV&NLP\\runs\\ball_tracking\\Test_demo_ITA_DOM_Parigi2024_30fps_20261118_140349\\tracce.csv", "rows": 103487, "config_start_seconds": 680, "config_end_seconds_exclusive": 7350, "min_frame_with_row": 20585, "max_frame_with_row": 220500, "min_timestamp_with_row": 686.1666666666666, "max_timestamp_with_row": 7350.0, "position_types": {"osservata": 85437, "predetta": 18050}}]
Audit confini palla: []
CSV giocatori: 1921965 righe, classi={'player': 192196

## 3. Associazioni, score raw e prior contestuali

Per frame esatto si associano detection gesti a detection giocatore con IoU e distanza centri; match concorrenti diventano `ambigua`. Si allega il punto palla temporalmente piu vicino, distinguendo `osservata` da `predetta`; la distanza resta un indizio, non prova un contatto.

Il modello espone tramite il raw head sei score di classe per anchor e l'indice dell'anchor mantenuto dalla NMS top-class. Gli score sono **non calibrati**, non probabilita. Un'alternativa e candidata solo se ha punteggio raw sufficiente e si ripete in una parte sostanziale dei frame dell'episodio. Priors limitati possono ripesare ricezione/difesa dopo una battuta affidabile dell'altro lato o attacco dopo alzata affidabile dello stesso lato. La correzione avviene solo per episodio multi-frame, track associato, classe alternativa raw disponibile, prior esplicito e gap raw piccolo. CSV e video conservano sempre classe raw e contestuale, vettore raw e ripesato, regole e motivo.

Se l'API non espone il tensor raw compatibile, il fallback salva solo classe top/confidenza, lascia score alternativi vuoti e impedisce la correzione. Nessun ace/errore viene dedotto da assenza di risposta. Sequenza same-side di attacchi e un segnale di anomalia con prior debole, non un divieto o motivo sufficiente da solo per cambiare classe. Un attacco affidabile del lato opposto quasi simultaneo puo fornire un prior morbido per muro se gli score raw sostengono l'alternativa; senza calibrazione rete e contatto resta un indizio, non una conferma. Non inferire ruolo/rotazione/linea da bbox o salto; la battuta in salto non e scartata. Muro puo essere eseguito senza tocco: prossimita palla osservata e solo candidato contatto non confermato, stima Kalman non prova contatto.

Funzioni e controesempi sintetici seguono: cambio reale da score vicino + prior, fallback top-only, score lontano, episodio breve, nessun servizio precedente/possibile ace, attacco da altro lato e muro senza rete calibrata.

**Regola di precedenza monouso:** una battuta raw affidabile (almeno tre frame, track e associazione giocatore univoci, lato disponibile, confidenza picco >= 0.60) entro `SERVE_RESPONSE_WINDOW_SECONDS` (3s configurati) abilita `ricezione` solo per il primo episodio raw `ricezione`/`difesa` successivo. Qualunque altro episodio gestuale rilevato iniziato dopo la battuta e prima del candidato consuma/interrompe il token: il contesto non supporta una correzione e la classe raw resta invariata, senza riutilizzare una battuta vecchia. Il token si consuma come barriera di sequenza, **non** come affermazione che l'episodio intermedio abbia toccato la palla. Stato palla osservata, Kalman o assente resta una distinta evidenza: prossimita osservata non conferma contatto e un muro puo essere senza tocco. Non si aggiungono soglie score-gap e non si creano episodi; `class_raw` resta invariata, anche in top-only (senza inventare score).

Se manca una battuta affidabile, il contesto e incompleto o un episodio intermedio interrompe la precedenza, la classe raw YOLO viene mantenuta: non si forza `difesa`. La correzione a `ricezione` avviene solo per una battuta affidabile senza episodi intermedi rilevati; altrimenti raw, motivo e stato di correzione restano espliciti. Un evento non rilevato puo ancora rendere insufficiente il contesto. Nessuna di queste incertezze usa scout/CSV ground truth durante l'inferenza.

La regola raw battuta→ricezione usa ora una finestra di 3 s e richiede lati geometrici opposti con battuta/risposta affidabilmente associate; lati mancanti, uguali, track incerti o episodi intermedi mantengono la classe raw. Per un candidato raw attacco, un attacco affidabile quasi simultaneo dell'altro lato puo aumentare debolmente lo score muro, ma si corregge solo se il vero score raw muro e ripetutamente supportato e il gap/margine consente la modifica. Il confronto usa episodi video, anche successivi nella stessa elaborazione offline; non usa scout. La simultaneita non dimostra muro, contatto, rete o squadra certa. Muri senza contatto restano possibili.

Ulteriori priors morbidi: ricezione/difesa affidabile favorisce alzata dello stesso lato; attacco affidabile favorisce difesa del lato opposto entro la finestra configurata e muro opposto quasi simultaneo. Un muro precedente non impone un lato per l'azione successiva. Sono solo pesi contestuali e non divieti; il candidato alternativo deve esistere negli score raw reali, l'episodio deve essere affidabile e il gap raw deve consentire la correzione.

In [3]:
def compute_frame_range(start, end, fps, frame_count, stride):
    values = (start, end, fps)
    if not all(math.isfinite(float(value)) for value in values) or fps <= 0:
        raise ValueError('Intervallo/FPS non validi.')
    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 1:
        raise ValueError('Stride deve essere intero positivo.')
    if not 0 <= start < end <= frame_count / fps:
        raise ValueError('Intervallo oltre durata video.')
    first = int(math.ceil(start * fps - 1e-9))
    exclusive = int(math.ceil(end * fps - 1e-9))
    if first >= exclusive:
        raise ValueError('Intervallo privo di frame sorgente.')
    return first, exclusive

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    iw, ih = max(0., min(ax2, bx2) - max(ax1, bx1)), max(0., min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = max(0., ax2-ax1) * max(0., ay2-ay1) + max(0., bx2-bx1) * max(0., by2-by1) - inter
    return inter / union if union > 0 else 0.

def associate_player(gesture_box, candidates):
    x1, y1, x2, y2 = map(float, gesture_box)
    gx, gy = (x1+x2)/2, (y1+y2)/2
    gh = max(1., y2-y1)
    scored = []
    for row in candidates:
        pbox = [row['bbox_x1'], row['bbox_y1'], row['bbox_x2'], row['bbox_y2']]
        iou = box_iou(gesture_box, pbox)
        px, py = (pbox[0]+pbox[2])/2, (pbox[1]+pbox[3])/2
        distance = math.hypot(gx-px, gy-py)
        ph = max(1., pbox[3]-pbox[1])
        normalized = distance / max(gh, ph)
        eligible = iou >= PLAYER_IOU_MIN or distance <= PLAYER_CENTER_MAX_HEIGHTS * max(gh, ph)
        if eligible:
            # Score di ordinamento, non probabilita/calibrazione.
            score = 0.65 * iou + 0.35 / (1.0 + normalized)
            scored.append((score, row, iou, distance, normalized))
    scored.sort(key=lambda item: (-item[0], str(item[1].get('track_id'))))
    empty = {'player_track_id': None, 'player_team_side': None, 'player_association': 'non_associata',
             'player_x_m': math.nan, 'player_y_m': math.nan,
             'player_iou': math.nan, 'player_center_distance_px': math.nan, 'player_score': math.nan}
    if not scored:
        return empty
    best = scored[0]
    if len(scored) > 1 and best[0] <= scored[1][0] * PLAYER_AMBIGUITY_RATIO:
        return {**empty, 'player_association': 'ambigua', 'player_iou': best[2],
                'player_center_distance_px': best[3], 'player_score': best[0]}
    track_id = best[1].get('track_id')
    player_x_m = player_y_m = math.nan
    try:
        if as_bool(best[1].get('in_court')):
            # The tracking calibration stores width in floor_x_m and court length in floor_y_m.
            player_x_m = float(best[1]['floor_y_m'])
            player_y_m = float(best[1]['floor_x_m'])
            if not (math.isfinite(player_x_m) and math.isfinite(player_y_m)
                    and 0.0 <= player_x_m <= 18.0 and 0.0 <= player_y_m <= 9.0):
                player_x_m = player_y_m = math.nan
    except (KeyError, TypeError, ValueError):
        player_x_m = player_y_m = math.nan
    return {'player_track_id': int(track_id) if pd.notna(track_id) else None,
            'player_x_m': player_x_m, 'player_y_m': player_y_m,
            'player_team_side': best[1].get('team_side') if pd.notna(best[1].get('team_side')) else None,
            'player_association': 'associata' if pd.notna(track_id) else 'bbox_senza_track',
            'player_iou': best[2], 'player_center_distance_px': best[3], 'player_score': best[0]}

def nearest_ball(gesture_box, candidates):
    x1, y1, x2, y2 = map(float, gesture_box)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    valid = []
    for row in candidates:
        distance = math.hypot(cx-float(row['x_px']), cy-float(row['y_px']))
        if distance <= BALL_ASSOCIATION_MAX_PX:
            valid.append((distance, row))
    valid.sort(key=lambda item: item[0])
    if not valid:
        return {'ball_track_id': None, 'ball_position_type': 'assente_o_lontana',
                'ball_distance_px': math.nan, 'ball_confidence': math.nan}
    distance, row = valid[0]
    status = str(row['position_type'])
    return {'ball_track_id': int(row['track_id']) if pd.notna(row['track_id']) else None,
            'ball_position_type': status, 'ball_distance_px': distance,
            'ball_confidence': float(row['confidence']) if pd.notna(row['confidence']) else math.nan}

def observed_ball_ground_position(gesture_box, candidates, homography):
    """Return (court-length x, court-width y) from an observed ball pixel, or (None, None)."""
    try:
        h = np.asarray(homography, dtype=np.float64)
    except (TypeError, ValueError):
        return None, None
    if h.shape != (3, 3) or not np.isfinite(h).all():
        return None, None
    x1, y1, x2, y2 = map(float, gesture_box)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    eligible = []
    for row in candidates:
        if str(row.get('position_type', '')).strip().casefold() != 'osservata':
            continue
        try:
            px, py = float(row['x_px']), float(row['y_px'])
        except (KeyError, TypeError, ValueError):
            continue
        if not (math.isfinite(px) and math.isfinite(py)):
            continue
        distance = math.hypot(cx - px, cy - py)
        if distance <= BALL_ASSOCIATION_MAX_PX:
            eligible.append((distance, px, py))
    if not eligible:
        return None, None
    _, px, py = min(eligible, key=lambda item: item[0])
    projected = h @ np.array([px, py, 1.], dtype=np.float64)
    denominator = float(projected[2])
    if not np.isfinite(projected).all() or denominator < 1e-4:
        return None, None
    court_width_x, court_length_y = projected[:2] / denominator
    if not (0. <= court_width_x <= 9. and 0. <= court_length_y <= 18.):
        return None, None
    return float(court_length_y), float(court_width_x)

def make_episode_id(row, open_episodes, frame_index, fps):
    track = row['player_track_id']
    candidates = []
    for episode in open_episodes:
        if episode['class_id'] != row['class_id'] or (frame_index-episode['last_frame'])/fps > EPISODE_GAP_SECONDS:
            continue
        if track is not None and episode['player_track_id'] == track:
            candidates.append((1.0, episode))
        elif track is None and episode['player_track_id'] is None:
            overlap = box_iou(row['bbox'], episode['last_bbox'])
            if overlap >= 0.5:
                candidates.append((overlap, episode))
    if candidates:
        return max(candidates, key=lambda pair: pair[0])[1]['episode_id']
    return f'E{len(open_episodes)+1:06d}'

def attach_soft_context(row, history, simultaneous):
    reasons = []
    adjustment = 0.0
    side = row.get('player_team_side')
    if row['class_name'] in ('ricezione', 'difesa') and side:
        serves = [item for item in history if item['class_name'] == 'battuta'
                  and 0 < row['timestamp_seconds']-item['timestamp_seconds'] <= SERVE_RESPONSE_WINDOW_SECONDS]
        if serves and serves[-1].get('player_team_side') and serves[-1]['player_team_side'] != side:
            adjustment += SOFT_RULE_WEIGHT
            reasons.append('battuta_precedente_risposta_altro_lato_geometrico_indizio')
        # Un intervallo senza risposta rilevata NON prova un ace/errore/contatto mancato.
    if row['class_name'] == 'attacco' and side:
        prior = [item for item in history if item['class_name'] == 'attacco'
                 and 0 < row['timestamp_seconds']-item['timestamp_seconds'] <= SAME_SIDE_ATTACK_WINDOW_SECONDS]
        if prior and prior[-1].get('player_team_side') == side:
            adjustment -= SOFT_RULE_WEIGHT
            reasons.append('attacco_stesso_lato_ravvicinato_anomalia_morbida_non_divieto')
    if row['class_name'] == 'muro':
        opposed_attack = any(item['class_name'] == 'attacco'
                             and abs(row['timestamp_seconds']-item['timestamp_seconds']) <= SIMULTANEOUS_WINDOW_SECONDS
                             and side and item.get('player_team_side') and side != item.get('player_team_side')
                             for item in history + simultaneous)
        if opposed_attack:
            reasons.append('attacco_muro_quasi_simultanei_lati_opposti_rete_non_calibrata')
    if not side:
        reasons.append('lato_geometrico_assente')
    if row.get('player_association') != 'associata':
        reasons.append('associazione_giocatore_non_affidabile')
    if row.get('ball_position_type') == 'predetta':
        reasons.append('contesto_palla_kalman_predetto_non_osservato')
    if row.get('ball_position_type') == 'osservata':
        reasons.append('palla_osservata_vicinanza_non_prova_contatto')
    if not row.get('ball_position_type') or row.get('ball_position_type') == 'assente_o_lontana':
        reasons.append('palla_non_associata')
    row['soft_adjustment'] = adjustment
    row['ranking_score_diagnostic_only'] = min(1.0, max(0.0, row['confidence'] + adjustment))
    row['class_corrected'] = row['class_name']
    row['correction_reason'] = 'nessuna_modifica_automatica_della_classe'
    row['rule_evidence'] = '|'.join(dict.fromkeys(reasons)) or 'nessuna_regola_attivata'
    return row

LAST_CLASSIFIER_FEATURE_MAPS = {}

def classifier_feature_for_anchor(feature_maps, anchor_index):
    if not feature_maps or any(not isinstance(feature_map, torch.Tensor)
                               or feature_map.ndim != 4 or feature_map.shape[0] != 1
                               for feature_map in feature_maps):
        return None
    offset = 0
    for feature_map in feature_maps:
        height, width = feature_map.shape[2:]
        level_size = height * width
        if offset <= anchor_index < offset + level_size:
            local_index = anchor_index - offset
            y, x = divmod(local_index, width)
            vector = feature_map[0, :, y, x].detach().float().cpu().numpy()
            return vector.astype(np.float32) if np.isfinite(vector).all() else None
        offset += level_size
    return None

def normalize_raw_class_scores(raw_scores, class_count):
    if raw_scores is None:
        return None
    scores = np.asarray(raw_scores, dtype=np.float64)
    if (scores.shape != (class_count,) or not np.isfinite(scores).all()
            or np.any(scores < 0.0) or np.any(scores > 1.0)):
        return None
    total = float(scores.sum())
    return scores / total if total > 0.0 else None

_synthetic_feature_maps = [torch.arange(8, dtype=torch.float32).reshape(1, 2, 2, 2),
                           torch.arange(8, 20, dtype=torch.float32).reshape(1, 3, 2, 2)]
assert np.array_equal(classifier_feature_for_anchor(_synthetic_feature_maps, 1), np.array([1., 5.], dtype=np.float32))
assert np.array_equal(classifier_feature_for_anchor(_synthetic_feature_maps, 4), np.array([8., 12., 16.], dtype=np.float32))
assert classifier_feature_for_anchor(_synthetic_feature_maps, 8) is None
_synthetic_probabilities = normalize_raw_class_scores([0.6, 0.1, 0.1, 0.1, 0.05, 0.05], 6)
assert np.isclose(_synthetic_probabilities.sum(), 1.0) and np.argmax(_synthetic_probabilities) == 0
assert normalize_raw_class_scores(None, 6) is None
_synthetic_box = [0., 0., 10., 20.]
_synthetic_ball_rows = [
    {'position_type': 'predetta', 'x_px': 5., 'y_px': 10.},
    {'position_type': 'osservata', 'x_px': 4., 'y_px': 12.},
]
_synthetic_ball_xy = observed_ball_ground_position(_synthetic_box, _synthetic_ball_rows, np.eye(3))
assert np.allclose(_synthetic_ball_xy, (12., 4.))
assert observed_ball_ground_position(_synthetic_box, [_synthetic_ball_rows[0]], np.eye(3)) == (None, None)
assert observed_ball_ground_position(_synthetic_box, [], np.eye(3)) == (None, None)
assert observed_ball_ground_position(
    _synthetic_box, [{'position_type': 'osservata', 'x_px': 10., 'y_px': 10.}], np.eye(3)) == (None, None)
assert observed_ball_ground_position(_synthetic_box, _synthetic_ball_rows,
    np.array([[1., 0., 0.], [0., 1., 0.], [0., 0., 0.]])) == (None, None)

def predict_frame_candidates(frame):
    """Return standard top-class NMS boxes plus true per-anchor raw class scores if exposed."""
    global PREDICTOR, RAW_SCORE_VECTOR_SUPPORTED, RAW_SCORE_FALLBACK_REASON
    global LAST_CLASSIFIER_FEATURE_MAPS
    if RAW_SCORE_VECTOR_SUPPORTED:
        if PREDICTOR is None:
            predictor_type = MODEL._smart_load('predictor')
            PREDICTOR = predictor_type(overrides={
                'imgsz': IMAGE_SIZE, 'conf': DETECTION_CONFIDENCE, 'iou': NMS_IOU,
                'device': DEVICE, 'rect': True, 'mode': 'predict', 'task': 'detect',
                'verbose': False, 'max_det': MAX_DETECTIONS},
                _callbacks=MODEL.callbacks)
            PREDICTOR.setup_model(model=MODEL.model, verbose=False)
            PREDICTOR.setup_source([frame], stride=int(PREDICTOR.model.stride))
            detect_module = PREDICTOR.model.model.model[-1]
            def make_classifier_feature_hook(level):
                def capture_classifier_features(module, inputs):
                    global LAST_CLASSIFIER_FEATURE_MAPS
                    feature_input = inputs[0] if inputs else None
                    if isinstance(feature_input, torch.Tensor):
                        LAST_CLASSIFIER_FEATURE_MAPS[level] = feature_input.detach()
                return capture_classifier_features
            for level, classifier in enumerate(detect_module.cv3):
                classifier[-1].register_forward_pre_hook(make_classifier_feature_hook(level))
        LAST_CLASSIFIER_FEATURE_MAPS = {}
        input_tensor = PREDICTOR.preprocess([frame])
        with torch.inference_mode():
            prediction = PREDICTOR.inference(input_tensor)
        raw_tensor = prediction[0] if isinstance(prediction, (tuple, list)) else prediction
        expected_channels = 4 + len(CLASS_NAMES)
        if (not isinstance(raw_tensor, torch.Tensor) or raw_tensor.ndim != 3
                or raw_tensor.shape[0] != 1 or raw_tensor.shape[1] != expected_channels):
            RAW_SCORE_VECTOR_SUPPORTED = False
            RAW_SCORE_FALLBACK_REASON = (
                f'Output raw non compatibile con 4+{len(CLASS_NAMES)} canali per anchor: '
                f'{getattr(raw_tensor, "shape", type(raw_tensor).__name__)}')
            warnings.warn(RAW_SCORE_FALLBACK_REASON + '; uso solo classe top/confidenza YOLO.')
        else:
            detect_module = PREDICTOR.model.model.model[-1]
            classifier_feature_maps = [
                LAST_CLASSIFIER_FEATURE_MAPS.get(level)
                for level in range(len(detect_module.cv3))]
            if (any(not isinstance(level, torch.Tensor) or level.ndim != 4
                    for level in classifier_feature_maps)
                    or sum(level.shape[2] * level.shape[3] for level in classifier_feature_maps)
                    != raw_tensor.shape[2]):
                raise RuntimeError('Le feature pre-classificatore non corrispondono agli anchor raw del modello.')
            detections, keep_indices = YOLO_NMS(
                prediction, conf_thres=DETECTION_CONFIDENCE, iou_thres=NMS_IOU,
                multi_label=False, max_det=MAX_DETECTIONS, nc=len(CLASS_NAMES),
                return_idxs=True)
            rows = []
            if len(detections) != 1 or len(keep_indices) != 1:
                raise RuntimeError('NMS raw non ha restituito un batch singolo interpretabile.')
            det = detections[0]
            anchors = keep_indices[0]
            if len(det) != len(anchors):
                raise RuntimeError('NMS raw: box e indici anchor non allineati.')
            if len(det):
                scaled_boxes = scale_boxes(input_tensor.shape[2:], det[:, :4].clone(), frame.shape[:2])
                for index in range(len(det)):
                    anchor = int(anchors[index])
                    scores = raw_tensor[0, 4:expected_channels, anchor].detach().float().cpu().numpy()
                    class_id = int(det[index, 5])
                    confidence = float(det[index, 4])
                    if (scores.shape != (len(CLASS_NAMES),) or not np.isfinite(scores).all()
                            or not 0 <= class_id < len(CLASS_NAMES)
                            or not math.isclose(float(scores[class_id]), confidence, rel_tol=0.03, abs_tol=0.015)):
                        RAW_SCORE_VECTOR_SUPPORTED = False
                        RAW_SCORE_FALLBACK_REASON = 'Score vector raw non allineato alla classe/confidenza NMS.'
                        warnings.warn(RAW_SCORE_FALLBACK_REASON + '; uso solo classe top/confidenza YOLO.')
                        break
                    feature_vector = classifier_feature_for_anchor(classifier_feature_maps, anchor)
                    rows.append({'bbox': scaled_boxes[index].detach().float().cpu().numpy().tolist(),
                                 'class_id': class_id, 'confidence': confidence,
                                 'raw_class_scores': scores.tolist(), 'score_mode': 'raw_head_six_class',
                                 'embedding_feature': feature_vector})
                if RAW_SCORE_VECTOR_SUPPORTED:
                    return rows, 'raw_head_six_class'
            else:
                return rows, 'raw_head_six_class'
    # Safe fallback: standard Ultralytics Results expose only top class and confidence.
    result = MODEL.predict(source=frame, imgsz=IMAGE_SIZE, conf=DETECTION_CONFIDENCE,
                           iou=NMS_IOU, max_det=MAX_DETECTIONS, device=DEVICE, verbose=False)[0]
    rows = []
    if result.boxes is not None and len(result.boxes):
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        confidences = result.boxes.conf.detach().cpu().numpy()
        class_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
        for box, confidence, class_id in zip(boxes, confidences, class_ids):
            rows.append({'bbox': list(map(float, box)), 'class_id': int(class_id),
                         'confidence': float(confidence), 'raw_class_scores': None,
                         'score_mode': 'top_only'})
    return rows, 'top_only'

def resolve_receive_dig_class(raw_id, current_time, prior_episodes,
                              response_side=None, response_track_id=None,
                              response_reliable=False):
    receive_id, defense_id = CLASS_NAMES.index('ricezione'), CLASS_NAMES.index('difesa')
    serve_id = CLASS_NAMES.index('battuta')
    if raw_id not in (receive_id, defense_id):
        return None
    valid_serves, unreliable_serves = [], []
    for prior in prior_episodes:
        if prior.get('class_id') != serve_id:
            continue
        previous_time = prior.get('last_timestamp_seconds')
        if previous_time is None or not math.isfinite(float(previous_time)):
            continue
        gap = current_time - float(previous_time)
        if not (0 < gap <= SERVE_RESPONSE_WINDOW_SECONDS):
            continue
        if (prior.get('reliable')
                and prior.get('peak_confidence', 0.) >= MIN_SERVE_CONFIDENCE_FOR_RECEIVE_DIG):
            valid_serves.append((gap, prior))
        else:
            unreliable_serves.append((gap, prior))

    def preserve_raw(rule, reason, serve=None, intervening=()):
        return {
            'target_id': raw_id,
            'supported': False,
            'rule': rule,
            'reason': reason,
            'serve_episode_id': serve.get('episode_id') if serve else None,
            'intervening_episode_ids': [
                item.get('episode_id') for item in intervening
            ],
        }

    if valid_serves:
        gap, serve = min(valid_serves, key=lambda item: item[0])
        serve_start = float(serve.get(
            'first_timestamp_seconds', serve['last_timestamp_seconds']))
        intervening = []
        for prior in prior_episodes:
            if prior is serve or prior.get('class_id') == serve_id:
                continue
            event_start = prior.get('first_timestamp_seconds')
            if event_start is None or not math.isfinite(float(event_start)):
                continue
            if serve_start < float(event_start) < current_time:
                intervening.append(prior)
        if intervening:
            intervening.sort(key=lambda item: (
                float(item.get('first_timestamp_seconds', math.inf)),
                str(item.get('episode_id', ''))))
            events = ','.join(
                f"{item.get('episode_id', '?')}:{CLASS_NAMES[int(item['class_id'])]}"
                f"[{item.get('contact_evidence_status', 'contatto_non_verificato')}]"
                for item in intervening)
            return preserve_raw(
                'episodio_intermedio_contesto_insufficiente_raw_preservata',
                (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: episodio/i intermedio/i "
                 f"({events}) tra battuta e candidato; contatto non affermato e nessuna "
                 f"riclassificazione supportata; battuta {gap:.2f}s prima"),
                serve, intervening)
        if not response_reliable or response_track_id is None or not response_side:
            return preserve_raw(
                'risposta_senza_associazione_lato_affidabile_raw_preservata',
                (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: la risposta non ha "
                 "track/associazione/lato geometrico affidabile per verificare il lato "
                 "opposto della battuta"),
                serve)
        serve_side = serve.get('player_team_side')
        if not serve_side:
            return preserve_raw(
                'battuta_senza_lato_affidabile_raw_preservata',
                (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: la battuta entro {gap:.2f}s "
                 "non ha lato geometrico affidabile"),
                serve)
        if serve_side == response_side:
            return preserve_raw(
                'battuta_e_risposta_stesso_lato_raw_preservata',
                (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: battuta e risposta "
                 "risultano associate allo stesso lato geometrico; la regola richiede "
                 "lati opposti e il contesto non supporta la correzione"),
                serve)
        return {
            'target_id': receive_id,
            'supported': True,
            'rule': 'raw_ricezione_difesa_dopo_battuta_affidabile_lato_opposto_entro_3s',
            'reason': (f"classe raw {CLASS_NAMES[raw_id]} risolta a ricezione: battuta "
                       f"affidabile {gap:.2f}s prima, risposta associata al lato "
                       "geometrico opposto e nessun episodio intermedio rilevato; "
                       "il lato non identifica la squadra e cio non conferma un contatto"),
            'serve_episode_id': serve.get('episode_id'),
            'intervening_episode_ids': [],
        }

    initial_context_incomplete = (
        current_time - SERVE_RESPONSE_WINDOW_SECONDS < START_SECONDS
    )
    if unreliable_serves:
        gap, serve = min(unreliable_serves, key=lambda item: item[0])
        return preserve_raw(
            'battuta_non_affidabile_raw_preservata',
            (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: battuta rilevata {gap:.2f}s "
             f"prima ma non affidabile (track/associazione/durata/confidenza insufficienti); "
             "nessuna correzione contestuale supportata"),
            serve)
    if initial_context_incomplete:
        reason = (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: i {SERVE_RESPONSE_WINDOW_SECONDS:.2f}s "
                  "precedenti non sono tutti osservati dall'inizio della clip; contesto insufficiente")
        rule = 'contesto_iniziale_incompleto_raw_preservata'
    else:
        reason = (f"classe raw {CLASS_NAMES[raw_id]} mantenuta: nessuna battuta affidabile "
                  f"rilevata nei {SERVE_RESPONSE_WINDOW_SECONDS:.2f}s precedenti; "
                  "contesto insufficiente")
        rule = 'nessuna_battuta_affidabile_raw_preservata'
    return preserve_raw(rule, reason)

def apply_receive_dig_decision(result, decision, extra_rules=(), score_note=''):
    if decision is None:
        return
    target = CLASS_NAMES[decision['target_id']]
    result['class_contextual'] = target
    result['class_corrected'] = target
    result['correction_eligible'] = bool(decision.get('supported', True))
    result['correction_performed'] = target != result['class_raw']
    result['context_rules_applied'] = list(dict.fromkeys(
        list(extra_rules) + [decision['rule']]))
    result['correction_reason'] = decision['reason'] + score_note

def contextualize_episode(episode, prior_episodes, enabled=True, simultaneous_episodes=()):
    """Apply bounded soft priors only to real raw-head scores in a reliable episode."""
    raw_id = int(episode['class_id'])
    result = {'class_raw': CLASS_NAMES[raw_id], 'class_contextual': CLASS_NAMES[raw_id],
              'class_corrected': CLASS_NAMES[raw_id], 'score_mode': 'top_only',
              'raw_score_mean': None, 'context_score_adjusted': None,
              'context_rules_applied': [], 'correction_reason': 'fallback_raw_unchanged',
              'correction_performed': False, 'correction_eligible': False,
              'contact_evidence_status': 'non_applicabile'}
    if raw_id == CLASS_NAMES.index('muro'):
        nearby_observed = any(row.get('ball_position_type') == 'osservata'
                              and row.get('ball_distance_px') is not None
                              and math.isfinite(float(row['ball_distance_px']))
                              and float(row['ball_distance_px']) <= CONTACT_PROXIMITY_DIAGNOSTIC_PX
                              for row in episode.get('frame_evidence', []))
        nearby_predicted = any(row.get('ball_position_type') == 'predetta'
                               and row.get('ball_distance_px') is not None
                               and math.isfinite(float(row['ball_distance_px']))
                               and float(row['ball_distance_px']) <= CONTACT_PROXIMITY_DIAGNOSTIC_PX
                               for row in episode.get('frame_evidence', []))
        if nearby_observed:
            result['contact_evidence_status'] = 'palla_osservata_vicina_candidato_contatto_non_confermato'
        elif nearby_predicted:
            result['contact_evidence_status'] = 'solo_palla_kalman_vicina_contatto_non_confermato'
        else:
            result['contact_evidence_status'] = 'nessuna_evidenza_palla_vicina_muro_puo_essere_senza_contatto'

    current_episode_reliable = (
        episode.get('detection_count', 0) >= MIN_EPISODE_DETECTIONS_FOR_CORRECTION
        and episode.get('player_track_id') is not None
        and episode.get('association_statuses') == {'associata'}
        and episode.get('player_team_side') is not None)
    receive_dig_decision = (resolve_receive_dig_class(
        raw_id, float(episode['first_timestamp_seconds']), prior_episodes,
        response_side=episode.get('player_team_side'),
        response_track_id=episode.get('player_track_id'),
        response_reliable=current_episode_reliable) if enabled else None)
    vectors = episode.get('raw_score_vectors', [])
    if (not vectors or len(vectors) != episode.get('detection_count')
            or episode.get('score_mode_set') != {'raw_head_six_class'}):
        if vectors or episode.get('score_mode_set', set()) - {'top_only'}:
            result['score_mode'] = 'mixed_or_incomplete_raw_scores'
            result['correction_reason'] = 'vettore_raw_incompleto_misto_o_non_allineato; nessuna correzione'
        else:
            result['correction_reason'] = 'vettore_sei_classi_non_disponibile_classe_top_only_raw_preservata'
        apply_receive_dig_decision(
            result, receive_dig_decision, score_note='; fallback top-only, score alternativi non disponibili.')
        return result
    matrix = np.asarray(vectors, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[1] != len(CLASS_NAMES) or not np.isfinite(matrix).all():
        result['correction_reason'] = 'vettore_score_raw_non_valido_raw_preservata'
        apply_receive_dig_decision(
            result, receive_dig_decision, score_note='; vettore score raw non valido, nessuno score alternativo creato.')
        return result
    means = matrix.mean(axis=0)
    support = (matrix >= CONTEXT_CANDIDATE_SCORE_FLOOR).sum(axis=0)
    min_support = max(2, int(math.ceil(len(matrix) * 0.5)))
    available = (support >= min_support) & (means >= CONTEXT_CANDIDATE_SCORE_FLOOR)
    result['score_mode'] = 'raw_head_six_class'
    result['raw_score_mean'] = means.tolist()
    adjusted = means.copy()
    if not enabled:
        result['context_score_adjusted'] = adjusted.tolist()
        result['correction_reason'] = 'correzione_disattivata; score contestuali uguali alla media raw'
        return result
    rules, correctable_targets = [], set()
    side = episode.get('player_team_side')
    current_time = float(episode['first_timestamp_seconds'])
    reliable_priors = [prior for prior in prior_episodes
        if prior.get('reliable') and prior.get('peak_confidence', 0.) >= 0.60
        and prior.get('player_team_side') and side
        and prior.get('last_timestamp_seconds') is not None]

    if raw_id != 0:
        serves = [prior for prior in reliable_priors if prior['class_id'] == 0
                  and prior['player_team_side'] != side
                  and prior.get('player_track_id') != episode.get('player_track_id')
                  and 0 < current_time-prior['last_timestamp_seconds'] <= SERVE_RESPONSE_WINDOW_SECONDS]
        if serves and (available[1] or available[5]):
            adjusted[1] *= SERVE_RESPONSE_RECEIVE_FACTOR
            adjusted[5] *= SERVE_RESPONSE_DIG_FACTOR
            correctable_targets.update((1, 5))
            rules.append('battuta_precedente_risposta_altro_lato_prior_soft')
    sets = [prior for prior in reliable_priors if prior['class_id'] == 2
            and prior['player_team_side'] == side
            and 0 < current_time-prior['last_timestamp_seconds'] <= SERVE_RESPONSE_WINDOW_SECONDS]
    if raw_id != 2 and sets and available[3]:
        adjusted[3] *= SET_TO_ATTACK_FACTOR
        correctable_targets.add(3)
        rules.append('alzata_precedente_stesso_lato_prior_soft_attacco')
    same_side_attacks = [prior for prior in reliable_priors if prior['class_id'] == 3
        and prior['player_team_side'] == side
        and 0 < current_time-prior['last_timestamp_seconds'] <= SAME_SIDE_ATTACK_WINDOW_SECONDS]
    if raw_id == 3 and same_side_attacks:
        adjusted[3] *= SAME_SIDE_ATTACK_FACTOR
        rules.append('attacco_stesso_lato_ravvicinato_prior_morbido_non_divieto')
        # Sequenza potenzialmente incompleta: questa regola da sola non autorizza una nuova classe.
    def intervals_are_simultaneous(other):
        current_start = float(episode['first_timestamp_seconds'])
        current_end = float(episode['last_timestamp_seconds'])
        other_start = float(other.get('first_timestamp_seconds',
                                      other.get('last_timestamp_seconds', math.inf)))
        other_end = float(other.get('last_timestamp_seconds', other_start))
        gap = max(other_start - current_end, current_start - other_end, 0.0)
        return gap <= SIMULTANEOUS_WINDOW_SECONDS

    if raw_id == 3 and available[4] and side and episode.get('player_track_id') is not None:
        opposite_attacks = [
            other for other in simultaneous_episodes
            if other.get('episode_id') != episode.get('episode_id')
            and other.get('class_id') == CLASS_NAMES.index('attacco')
            and other.get('reliable')
            and other.get('peak_confidence', 0.) >= 0.60
            and other.get('player_track_id') is not None
            and other.get('player_track_id') != episode.get('player_track_id')
            and other.get('player_team_side')
            and other.get('player_team_side') != side
            and intervals_are_simultaneous(other)
        ]
        if opposite_attacks:
            adjusted[4] *= ATTACK_TO_BLOCK_FACTOR
            correctable_targets.add(4)
            rules.append('attacco_opposto_quasi_simultaneo_prior_morbido_attacco_a_muro')
    recent_same_side_receive_dig = [
        prior for prior in reliable_priors
        if prior['class_id'] in (CLASS_NAMES.index('ricezione'), CLASS_NAMES.index('difesa'))
        and prior['player_team_side'] == side
        and 0 < current_time-prior['last_timestamp_seconds'] <= SERVE_RESPONSE_WINDOW_SECONDS
    ]
    if raw_id != CLASS_NAMES.index('alzata') and recent_same_side_receive_dig and available[2]:
        adjusted[2] *= RECEIVE_DIG_TO_SET_FACTOR
        correctable_targets.add(2)
        rules.append('ricezione_difesa_precedente_stesso_lato_prior_morbido_alzata')

    recent_opposite_attacks = [
        prior for prior in reliable_priors
        if prior['class_id'] == CLASS_NAMES.index('attacco')
        and prior['player_team_side'] != side
        and prior.get('player_track_id') != episode.get('player_track_id')
        and 0 < current_time-prior['last_timestamp_seconds'] <= ATTACK_RESPONSE_WINDOW_SECONDS
    ]
    if raw_id != CLASS_NAMES.index('difesa') and recent_opposite_attacks and available[5]:
        adjusted[5] *= ATTACK_TO_DEFENSE_FACTOR
        correctable_targets.add(5)
        rules.append('attacco_precedente_lato_opposto_prior_morbido_difesa')

    if raw_id == 4 and available[4] and side:
        attack_contexts = reliable_priors + list(simultaneous_episodes)
        if any(
                other.get('episode_id') != episode.get('episode_id')
                and other.get('class_id') == CLASS_NAMES.index('attacco')
                and other.get('reliable')
                and other.get('peak_confidence', 0.) >= 0.60
                and other.get('player_track_id') is not None
                and other.get('player_track_id') != episode.get('player_track_id')
                and other.get('player_team_side')
                and other.get('player_team_side') != side
                and intervals_are_simultaneous(other)
                for other in attack_contexts):
            rules.append('attacco_opposto_quasi_simultaneo_indizio_muro; contatto_non_provato')

    result['context_score_adjusted'] = adjusted.tolist()
    result['context_rules_applied'] = rules
    if not enabled:
        result['correction_reason'] = 'correzione_disattivata'
        return result
    if receive_dig_decision is not None and receive_dig_decision.get('supported'):
        apply_receive_dig_decision(result, receive_dig_decision, extra_rules=rules)
        return result
    if receive_dig_decision is not None:
        rules.append(receive_dig_decision['rule'])
    reliable_episode = (episode.get('detection_count', 0) >= MIN_EPISODE_DETECTIONS_FOR_CORRECTION
        and episode.get('player_track_id') is not None
        and episode.get('association_statuses') == {'associata'}
        and episode.get('player_team_side') is not None)
    result['correction_eligible'] = bool(reliable_episode)
    if not reliable_episode:
        if receive_dig_decision is not None:
            apply_receive_dig_decision(result, receive_dig_decision, extra_rules=rules)
        else:
            result['correction_reason'] = 'episodio_non_affidabile_track_associazione_o_durata'
        return result
    if not rules or not correctable_targets:
        if receive_dig_decision is not None:
            apply_receive_dig_decision(result, receive_dig_decision, extra_rules=rules)
        else:
            result['correction_reason'] = 'nessun_prior_contestuale_sufficientemente_supportato'
        return result
    choices = [cid for cid in correctable_targets if available[cid]]
    if not choices:
        result['correction_reason'] = 'classe_alternativa_non_supportata_dagli_score_raw'
        return result
    candidate = max(choices, key=lambda cid: adjusted[cid])
    raw_gap = means[raw_id] - means[candidate]
    if (candidate != raw_id and adjusted[candidate] > adjusted[raw_id]
            and raw_gap <= CONTEXT_MAX_RAW_SCORE_GAP):
        result['class_contextual'] = CLASS_NAMES[candidate]
        result['class_corrected'] = CLASS_NAMES[candidate]
        result['correction_performed'] = True
        result['correction_reason'] = (
            f"{rules[0]}: score raw candidato disponibile, gap raw={raw_gap:.4f} "
            f"<= {CONTEXT_MAX_RAW_SCORE_GAP:.4f}; episodio affidabile multi-frame.")
    else:
        result['correction_reason'] = (
            f"fallback_raw_immutata: alternativa non supera il raw/gap massimo ({raw_gap:.4f}).")
    return result

def contextualize_episodes(episodes, frame_rows, enabled=True):
    ordered = sorted(episodes, key=lambda ep: (ep['first_timestamp_seconds'], ep['episode_id']))
    simultaneous_context = []
    for ep in ordered:
        reliable = (ep.get('detection_count', 0) >= MIN_EPISODE_DETECTIONS_FOR_CORRECTION
                    and ep.get('player_track_id') is not None
                    and ep.get('association_statuses') == {'associata'}
                    and ep.get('player_team_side') is not None)
        simultaneous_context.append({
            'episode_id': ep.get('episode_id'), 'class_id': int(ep['class_id']),
            'first_timestamp_seconds': float(ep['first_timestamp_seconds']),
            'last_timestamp_seconds': float(ep['last_timestamp_seconds']),
            'player_team_side': ep.get('player_team_side'),
            'player_track_id': ep.get('player_track_id'),
            'peak_confidence': float(ep.get('peak_confidence', 0.)),
            'reliable': reliable,
        })
    history, decisions = [], {}
    for episode in ordered:
        decision = contextualize_episode(
            episode, history, enabled=enabled,
            simultaneous_episodes=simultaneous_context)
        episode.update(decision)
        observed_ball_near = any(row['ball_position_type'] == 'osservata'
            and math.isfinite(float(row['ball_distance_px']))
            and float(row['ball_distance_px']) <= CONTACT_PROXIMITY_DIAGNOSTIC_PX
            for row in episode.get('frame_evidence', []))
        predicted_ball_near = any(row['ball_position_type'] == 'predetta'
            and math.isfinite(float(row['ball_distance_px']))
            and float(row['ball_distance_px']) <= CONTACT_PROXIMITY_DIAGNOSTIC_PX
            for row in episode.get('frame_evidence', []))
        if episode['class_raw'] == 'muro' or episode['class_contextual'] == 'muro':
            episode['contact_evidence_status'] = (
                'palla_osservata_vicina_candidato_contatto_non_confermato' if observed_ball_near else
                'solo_palla_kalman_vicina_contatto_non_confermato' if predicted_ball_near else
                'nessuna_evidenza_palla_vicina_muro_puo_essere_senza_contatto')
        decisions[episode['episode_id']] = episode
        reliable = (episode.get('detection_count', 0) >= MIN_EPISODE_DETECTIONS_FOR_CORRECTION
                    and episode.get('player_track_id') is not None
                    and episode.get('association_statuses') == {'associata'}
                    and episode.get('player_team_side') is not None)
        event_contact_evidence = episode.get('contact_evidence_status', 'non_applicabile')
        if observed_ball_near:
            event_contact_evidence = 'palla_osservata_vicina_candidato_contatto_non_confermato'
        elif predicted_ball_near:
            event_contact_evidence = 'solo_palla_kalman_vicina_contatto_non_confermato'
        elif episode['class_raw'] != 'muro':
            event_contact_evidence = 'contatto_non_verificato_nessuna_palla_osservata_vicina'
        history.append({'episode_id': episode['episode_id'], 'class_id': episode['class_id'],
            'first_timestamp_seconds': episode['first_timestamp_seconds'],
            'last_timestamp_seconds': episode['last_timestamp_seconds'],
            'player_team_side': episode.get('player_team_side'),
            'player_track_id': episode.get('player_track_id'),
            'peak_confidence': episode.get('peak_confidence', 0.),
            'contact_evidence_status': event_contact_evidence,
            'reliable': reliable})
    for row in frame_rows:
        decision = decisions[row['episode_id']]
        row['class_contextual'] = decision['class_contextual']
        row['class_corrected'] = decision['class_contextual']
        row['context_score_mode'] = decision['score_mode']
        row['context_raw_score_vector'] = (json.dumps(decision['raw_score_mean']) if decision['raw_score_mean'] is not None else '')
        row['context_adjusted_score_vector'] = (json.dumps(decision['context_score_adjusted'])
                                                 if decision['context_score_adjusted'] is not None else '')
        row['context_rule_applied'] = '|'.join(decision['context_rules_applied'])
        row['context_correction_reason'] = decision['correction_reason']
        row['context_correction_performed'] = decision['correction_performed']
        for class_id, name in enumerate(CLASS_NAMES):
            raw_scores = row.get('raw_class_scores')
            row[f'raw_score_{name}'] = float(raw_scores[class_id]) if raw_scores is not None else math.nan
            context_scores = decision['context_score_adjusted']
            row[f'context_score_{name}'] = float(context_scores[class_id]) if context_scores is not None else math.nan
    return list(decisions.values())

# Test unitari sintetici: nessun modello, video o scrittura.
_ORIGINAL_START_SECONDS_FOR_TESTS = START_SECONDS
START_SECONDS = 0.0
assert compute_frame_range(1.0, 2.0, 30.0, 300, 2) == (30, 60)
assert list(range(*compute_frame_range(1.0, 2.0, 30.0, 300, 2), 2)) == list(range(30,60,2))
assert compute_frame_range(1.01, 1.10, 30.0, 300, 1) == (31, 33)
def _synthetic_episode(class_id, scores, side='B_y_ge9', track=2, count=4):
    vectors = [list(scores) for _ in range(count)]
    return {'episode_id':'E','class_id':class_id,'class_name':CLASS_NAMES[class_id],
        'player_track_id':track,'player_team_side':side,'first_timestamp_seconds':2.0,
        'last_timestamp_seconds':2.2,'peak_confidence':float(scores[class_id]),
        'detection_count':count,'association_statuses':{'associata'},
        'score_mode_set':{'raw_head_six_class'},'raw_score_vectors':vectors}

_serve_prior = {'class_id':0,'last_timestamp_seconds':1.0,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.80,'reliable':True}
_close_receive = _synthetic_episode(2,[0.01,0.39,0.40,0.01,0.01,0.02])
_corrected = contextualize_episode(_close_receive,[_serve_prior])
assert _corrected['class_raw']=='alzata' and _corrected['class_contextual']=='ricezione'
assert _corrected['correction_performed'] and _corrected['raw_score_mean'][1] == 0.39
_disabled = contextualize_episode(_close_receive,[_serve_prior],enabled=False)
assert _disabled['class_contextual']=='alzata' and _disabled['raw_score_mean'] is not None
assert _disabled['context_score_adjusted']==_disabled['raw_score_mean']
_top_only = _synthetic_episode(2,[0.01,0.39,0.40,0.01,0.01,0.02])
_top_only['raw_score_vectors']=[];_top_only['score_mode_set']={'top_only'}
_fallback = contextualize_episode(_top_only,[_serve_prior])
assert _fallback['class_contextual']=='alzata' and _fallback['raw_score_mean'] is None
_no_serve = contextualize_episode(_close_receive,[])
assert _no_serve['class_contextual']=='alzata' and not _no_serve['correction_performed']
_ambiguous_raw_scores = [0.01,0.10,0.40,0.01,0.01,0.39]
_ambiguous_after_serve = _synthetic_episode(5,_ambiguous_raw_scores, count=2)
_ambiguous_after_serve['first_timestamp_seconds'] = START_SECONDS + 1.0
_ambiguous_after_serve['last_timestamp_seconds'] = START_SECONDS + 1.1
_detected_serve = {'episode_id':'S','class_id':0,
    'last_timestamp_seconds':START_SECONDS+0.2,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.8,'reliable':True}
_after_serve_result = contextualize_episode(_ambiguous_after_serve,[_detected_serve])
assert _after_serve_result['class_raw']=='difesa'
assert _after_serve_result['class_contextual']=='ricezione'
assert _after_serve_result['correction_performed']
assert 'raw_ricezione_difesa_dopo_battuta_affidabile_lato_opposto_entro_3s' in '|'.join(_after_serve_result['context_rules_applied'])

_no_serve_overwrite = _synthetic_episode(1,[0.01,0.40,0.01,0.01,0.01,0.10],count=2)
_no_serve_overwrite['first_timestamp_seconds'] = START_SECONDS + 3.0
_no_serve_overwrite['last_timestamp_seconds'] = START_SECONDS + 3.1
_no_serve_result = contextualize_episode(_no_serve_overwrite,[])
assert _no_serve_result['class_raw']=='ricezione'
assert _no_serve_result['class_contextual']=='ricezione'
assert not _no_serve_result['correction_performed']
assert 'mantenuta' in _no_serve_result['correction_reason']

_top_only_receive = _synthetic_episode(1,[0.0,0.0,0.0,0.0,0.0,0.0],count=1)
_top_only_receive['raw_score_vectors']=[]
_top_only_receive['score_mode_set']={'top_only'}
_top_only_result = contextualize_episode(_top_only_receive,[])
assert _top_only_result['class_contextual']=='ricezione'
assert _top_only_result['raw_score_mean'] is None
assert _top_only_result['context_score_adjusted'] is None

_unreliable_serve = dict(_detected_serve,reliable=False)
assert resolve_receive_dig_class(1,START_SECONDS+1.0,[_unreliable_serve],response_side='B_y_ge9',response_track_id=2,response_reliable=True)['target_id']==1
assert not resolve_receive_dig_class(1,START_SECONDS+1.0,[_unreliable_serve],response_side='B_y_ge9',response_track_id=2,response_reliable=True)['supported']

_serve_1172_unreliable = {
    'episode_id':'E000024','class_id':CLASS_NAMES.index('battuta'),
    'first_timestamp_seconds':1172.3333333333,
    'last_timestamp_seconds':1172.3333333333,
    'peak_confidence':0.41557177901268005,'reliable':False,
    'player_track_id':49,'detection_count':1}
_wall_1172 = {
    'episode_id':'E000025','class_id':CLASS_NAMES.index('muro'),
    'first_timestamp_seconds':1172.7666666667,
    'last_timestamp_seconds':1173.1666666667,
    'contact_evidence_status':'palla_osservata_vicina_candidato_contatto_non_confermato'}
_receive_at_1173_3 = _synthetic_episode(
    CLASS_NAMES.index('ricezione'),
    [0.00012871097149513844,0.6820528126996139,0.00029635497552872353,
     0.0000010260620973748918,0.000004448330008721308,0.00825635626799716],
    track=4,count=29)
_receive_at_1173_3['first_timestamp_seconds']=1172.8
_receive_at_1173_3['last_timestamp_seconds']=1174.066667
_receive_at_1173_3['peak_confidence']=0.886967
_touch_time_1173_306=1173.306
assert (_receive_at_1173_3['first_timestamp_seconds'] <= _touch_time_1173_306
        <= _receive_at_1173_3['last_timestamp_seconds'])
_receive_1173_3_result = contextualize_episode(
    _receive_at_1173_3,[_serve_1172_unreliable,_wall_1172])
assert _receive_1173_3_result['class_raw']=='ricezione'
assert _receive_1173_3_result['class_contextual']=='ricezione'
assert not _receive_1173_3_result['correction_performed']
assert not _receive_1173_3_result['correction_eligible']
assert 'battuta rilevata' in _receive_1173_3_result['correction_reason']
assert 'non affidabile' in _receive_1173_3_result['correction_reason']
assert contextualize_episode(
    _no_serve_overwrite,[],enabled=False)['class_contextual']=='ricezione'

_serve_sequence_event = {
    'episode_id':'S0','class_id':0,'first_timestamp_seconds':START_SECONDS+0.1,
    'last_timestamp_seconds':START_SECONDS+0.2,'peak_confidence':0.8,'reliable':True,
    'player_team_side':'A_y_lt9','player_track_id':1}
_first_receive = {
    'episode_id':'R1','class_id':1,'first_timestamp_seconds':START_SECONDS+0.5,
    'last_timestamp_seconds':START_SECONDS+0.6,'player_team_side':'B_y_ge9',
    'player_track_id':2,
    'contact_evidence_status':'palla_osservata_vicina_candidato_contatto_non_confermato'}
_first_decision = resolve_receive_dig_class(
    5,START_SECONDS+0.7,[_serve_sequence_event],
    response_side='B_y_ge9',response_track_id=2,response_reliable=True)
assert _first_decision['target_id']==1
_second_defense_decision = resolve_receive_dig_class(
    5,START_SECONDS+1.0,[_serve_sequence_event,_first_receive],
    response_side='B_y_ge9',response_track_id=2,response_reliable=True)
assert _second_defense_decision['target_id']==5
assert _second_defense_decision['intervening_episode_ids']==['R1']
assert 'contatto non affermato' in _second_defense_decision['reason']

_intermediate_block_without_contact = {
    'episode_id':'B1','class_id':4,'first_timestamp_seconds':START_SECONDS+0.4,
    'last_timestamp_seconds':START_SECONDS+0.5,
    'contact_evidence_status':'nessuna_evidenza_palla_vicina_muro_puo_essere_senza_contatto'}
_block_barrier_decision = resolve_receive_dig_class(
    1,START_SECONDS+0.8,[_serve_sequence_event,_intermediate_block_without_contact],
    response_side='B_y_ge9',response_track_id=2,response_reliable=True)
assert _block_barrier_decision['target_id']==1
assert not _block_barrier_decision['supported']
assert 'nessuna_evidenza' in _block_barrier_decision['reason']

_unobserved_intermediate = resolve_receive_dig_class(
    1,START_SECONDS+0.8,[_serve_sequence_event],
    response_side='B_y_ge9',response_track_id=2,response_reliable=True)
assert _unobserved_intermediate['target_id']==1
assert 'nessun episodio intermedio rilevato' in _unobserved_intermediate['reason']
_three_event_sequence = [_serve_sequence_event,_first_receive]
_first_context_episode = _synthetic_episode(
    5,[0.01,0.10,0.01,0.01,0.01,0.40],count=2)
_first_context_episode['first_timestamp_seconds']=START_SECONDS+0.5
_first_context_episode['last_timestamp_seconds']=START_SECONDS+0.6
_first_contextual = contextualize_episode(_first_context_episode,[_serve_sequence_event])
assert _first_contextual['class_raw']=='difesa'
assert _first_contextual['class_contextual']=='ricezione'
_first_receive['class_id']=5
_first_receive['class_contextual']='ricezione'
_second_context_episode = _synthetic_episode(
    5,[0.01,0.10,0.01,0.01,0.01,0.40],count=2)
_second_context_episode['first_timestamp_seconds']=START_SECONDS+0.7
_second_context_episode['last_timestamp_seconds']=START_SECONDS+0.8
_second_contextual = contextualize_episode(_second_context_episode,_three_event_sequence)
assert _first_contextual['class_contextual']=='ricezione'
assert _second_contextual['class_contextual']=='difesa'

def _sequence_episode(episode_id,class_id,first,last,scores):
    episode = _synthetic_episode(class_id,scores,count=3)
    episode.update({'episode_id':episode_id,'first_timestamp_seconds':first,
        'last_timestamp_seconds':last,'player_track_id':episode_id,
        'player_team_side':'A_y_lt9','association_statuses':{'associata'},
        'frame_evidence':[]})
    return episode

_serve_episode = _sequence_episode(
    'S1',0,START_SECONDS+0.1,START_SECONDS+0.2,[0.8,0.01,0.01,0.01,0.01,0.01])
_serve_episode['player_team_side']='A_y_lt9'
_receive_episode = _sequence_episode(
    'R1',5,START_SECONDS+0.5,START_SECONDS+0.6,[0.01,0.10,0.01,0.01,0.01,0.40])
_receive_episode['player_team_side']='B_y_ge9'
_following_defense_episode = _sequence_episode(
    'D1',5,START_SECONDS+0.8,START_SECONDS+0.9,[0.01,0.10,0.01,0.01,0.01,0.40])
_following_defense_episode['player_team_side']='B_y_ge9'
_sequence_result = contextualize_episodes(
    [_serve_episode,_receive_episode,_following_defense_episode],[])
assert [_ep['class_contextual'] for _ep in _sequence_result] == [
    'battuta','ricezione','difesa']

_set_intermediate = _sequence_episode(
    'A1',2,START_SECONDS+0.4,START_SECONDS+0.5,[0.01,0.01,0.80,0.01,0.01,0.01])
_after_unverified_set = _sequence_episode(
    'D2',5,START_SECONDS+0.8,START_SECONDS+0.9,[0.01,0.10,0.01,0.01,0.01,0.40])
_intermediate_result = contextualize_episodes(
    [_sequence_episode('S2',0,START_SECONDS+0.1,START_SECONDS+0.2,
        [0.8,0.01,0.01,0.01,0.01,0.01]),_set_intermediate,_after_unverified_set],[])
assert _intermediate_result[-1]['class_contextual']=='difesa'
assert 'contatto_non_verificato' in _intermediate_result[-1]['correction_reason']
_wide_gap = _synthetic_episode(2,[0.01,0.10,0.50,0.01,0.01,0.02])
assert contextualize_episode(_wide_gap,[_serve_prior])['class_contextual']=='alzata'
_short = _synthetic_episode(2,[0.01,0.39,0.40,0.01,0.01,0.02],count=1)
_short['raw_score_vectors']=[_short['raw_score_vectors'][0]];_short['detection_count']=1
assert contextualize_episode(_short,[_serve_prior])['class_contextual']=='alzata'
_ace_counterexample = contextualize_episode(_close_receive,[])
assert _ace_counterexample['class_contextual']=='alzata'
_different_side_attack = _synthetic_episode(3,[0.01,0.01,0.01,0.40,0.01,0.01],side='B_y_ge9')
_opposite_attack = {'class_id':3,'last_timestamp_seconds':1.8,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.8,'reliable':True}
assert contextualize_episode(_different_side_attack,[_opposite_attack])['class_contextual']=='attacco'
_same_side_attack = _synthetic_episode(3,[0.01,0.01,0.38,0.40,0.01,0.01],side='A_y_lt9')
_same_team_attack = {'class_id':3,'last_timestamp_seconds':1.8,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.8,'reliable':True}
_same_side_result = contextualize_episode(_same_side_attack,[_same_team_attack])
assert _same_side_result['class_contextual']=='attacco'
assert 'attacco_stesso_lato_ravvicinato' in '|'.join(_same_side_result['context_rules_applied'])
assert _same_side_result['context_score_adjusted'][3] < _same_side_result['raw_score_mean'][3]
_ambiguous_wall = _synthetic_episode(3,[0.01,0.01,0.01,0.40,0.39,0.01])
_opponent_attack = {'class_id':3,'last_timestamp_seconds':2.0,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.8,'reliable':True}
_wall_result = contextualize_episode(_ambiguous_wall,[_opponent_attack])
assert _wall_result['class_contextual']=='attacco'
assert _wall_result['class_contextual']=='attacco' and not _wall_result['context_rules_applied']
_jump_serve = _synthetic_episode(0,[0.01,0.01,0.01,0.01,0.01,0.01],side='A_y_lt9')
assert contextualize_episode(_jump_serve,[])['class_contextual']=='battuta'
assert math.isclose(box_iou([0,0,10,10], [0,0,10,10]), 1.0)
assert math.isclose(box_iou([0,0,10,10], [10,0,20,10]), 0.0)
assert make_episode_id({'class_id':3,'player_track_id':7,'bbox':[0,0,10,10]},
                       [{'episode_id':'E000001','class_id':3,'player_track_id':7,'last_frame':10,'last_bbox':[0,0,10,10]}],
                       20,30) == 'E000001'
assert make_episode_id({'class_id':3,'player_track_id':8,'bbox':[0,0,10,10]},
                       [{'episode_id':'E000001','class_id':3,'player_track_id':7,'last_frame':10,'last_bbox':[0,0,10,10]}],
                       20,30) == 'E000002'
base = {'class_name':'ricezione','timestamp_seconds':2.,'player_team_side':'B_y_ge9',
        'player_association':'associata','ball_position_type':'assente_o_lontana','confidence':.8}
response = attach_soft_context(dict(base), [{'class_name':'battuta','timestamp_seconds':1.,'player_team_side':'A_y_lt9'}], [])
assert response['soft_adjustment'] > 0 and response['class_corrected'] == 'ricezione'
ace_case = attach_soft_context(dict(base), [], [])
assert ace_case['class_corrected'] == 'ricezione' and 'ace' not in ace_case['rule_evidence']
same_side = attach_soft_context({'class_name':'attacco','timestamp_seconds':2.,'player_team_side':'A_y_lt9',
    'player_association':'associata','ball_position_type':'osservata','confidence':.8},
    [{'class_name':'attacco','timestamp_seconds':1.5,'player_team_side':'A_y_lt9'}], [])
other_side = attach_soft_context({'class_name':'attacco','timestamp_seconds':2.,'player_team_side':'B_y_ge9',
    'player_association':'associata','ball_position_type':'osservata','confidence':.8},
    [{'class_name':'attacco','timestamp_seconds':1.5,'player_team_side':'A_y_lt9'}], [])
assert same_side['soft_adjustment'] < 0 and other_side['soft_adjustment'] == 0
jump_serve = {'class_name':'battuta','timestamp_seconds':1.,'player_team_side':'A_y_lt9',
              'player_association':'associata','ball_position_type':'osservata','confidence':.9}
assert attach_soft_context(jump_serve, [], [])['class_corrected'] == 'battuta'
ambiguous_candidates = [dict(bbox_x1=0,bbox_y1=0,bbox_x2=20,bbox_y2=40,track_id=1,team_side='A'),
                        dict(bbox_x1=1,bbox_y1=0,bbox_x2=21,bbox_y2=40,track_id=2,team_side='B')]
ambiguous = associate_player([0,0,20,40], ambiguous_candidates)
assert ambiguous['player_association'] == 'ambigua' and ambiguous['player_track_id'] is None
predicted_ball = nearest_ball([0,0,20,20], [
    {'x_px':10,'y_px':10,'track_id':6,'position_type':'predetta','confidence':0.4}])
assert predicted_ball['ball_position_type'] == 'predetta'
observed_ball = nearest_ball([0,0,20,20], [
    {'x_px':10,'y_px':10,'track_id':6,'position_type':'osservata','confidence':0.9}])
assert observed_ball['ball_position_type'] == 'osservata'

_three_second_serve = dict(_serve_prior)
_three_second_serve['last_timestamp_seconds'] = START_SECONDS
_three_second_receive = _synthetic_episode(
    5,[0.01,0.39,0.01,0.01,0.01,0.40],side='B_y_ge9',track=2,count=3)
_three_second_receive['first_timestamp_seconds'] = START_SECONDS + 3.0
_three_second_receive['last_timestamp_seconds'] = START_SECONDS + 3.1
_serve_3s_decision = resolve_receive_dig_class(
    5,START_SECONDS+3.0,[_three_second_serve],
    response_side='B_y_ge9',response_track_id=2,response_reliable=True)
assert _serve_3s_decision['supported'] and _serve_3s_decision['target_id']==1
_too_late_decision = resolve_receive_dig_class(
    5,START_SECONDS+3.0001,[_three_second_serve],
    response_side='B_y_ge9',response_track_id=2,response_reliable=True)
assert not _too_late_decision['supported'] and _too_late_decision['target_id']==5
_same_side_decision = resolve_receive_dig_class(
    5,START_SECONDS+1.0,[_serve_prior],
    response_side='A_y_lt9',response_track_id=2,response_reliable=True)
assert not _same_side_decision['supported'] and _same_side_decision['target_id']==5
_missing_side_decision = resolve_receive_dig_class(
    5,START_SECONDS+1.0,[_serve_prior],
    response_side=None,response_track_id=2,response_reliable=False)
assert not _missing_side_decision['supported']

_same_side_set = _synthetic_episode(
    5,[0.01,0.01,0.39,0.01,0.01,0.40],side='A_y_lt9',track=2,count=4)
_same_side_receive_prior = {
    'class_id':1,'first_timestamp_seconds':1.0,'last_timestamp_seconds':1.1,
    'player_team_side':'A_y_lt9','player_track_id':1,
    'peak_confidence':0.8,'reliable':True}
_set_from_receive = contextualize_episode(_same_side_set,[_same_side_receive_prior])
assert _set_from_receive['class_contextual']=='alzata'
assert 'ricezione_difesa_precedente_stesso_lato_prior_morbido_alzata' in '|'.join(
    _set_from_receive['context_rules_applied'])

_opposite_defense = _synthetic_episode(
    1,[0.01,0.40,0.01,0.01,0.01,0.39],side='B_y_ge9',track=3,count=4)
_prior_opponent_attack = {
    'episode_id':'ATT_PRIOR','class_id':3,'first_timestamp_seconds':START_SECONDS+0.2,
    'last_timestamp_seconds':START_SECONDS+0.3,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.8,'reliable':True}
_attack_to_defense = contextualize_episode(
    _opposite_defense,[_prior_opponent_attack])
assert _attack_to_defense['class_contextual']=='difesa'
assert 'attacco_precedente_lato_opposto_prior_morbido_difesa' in '|'.join(
    _attack_to_defense['context_rules_applied'])

_attack_raw_block_candidate = _synthetic_episode(
    3,[0.01,0.01,0.01,0.40,0.38,0.01],side='B_y_ge9',track=2,count=4)
_opposing_attack_episode = {
    'episode_id':'ATT_SIM','class_id':3,'first_timestamp_seconds':2.0,
    'last_timestamp_seconds':2.2,'player_team_side':'A_y_lt9',
    'player_track_id':1,'peak_confidence':0.85,'reliable':True}
_attack_to_block = contextualize_episode(
    _attack_raw_block_candidate,[],simultaneous_episodes=[_opposing_attack_episode])
assert _attack_to_block['class_raw']=='attacco'
assert _attack_to_block['class_contextual']=='muro'
assert _attack_to_block['correction_performed']
assert 'attacco_opposto_quasi_simultaneo_prior_morbido_attacco_a_muro' in '|'.join(
    _attack_to_block['context_rules_applied'])
_weak_wall_candidate = _synthetic_episode(
    3,[0.01,0.01,0.01,0.40,0.04,0.01],side='B_y_ge9',track=2,count=4)
assert contextualize_episode(
    _weak_wall_candidate,[],simultaneous_episodes=[_opposing_attack_episode]
    )['class_contextual']=='attacco'
_same_side_attack_evidence = dict(_opposing_attack_episode,player_team_side='B_y_ge9')
assert contextualize_episode(
    _attack_raw_block_candidate,[],simultaneous_episodes=[_same_side_attack_evidence]
    )['class_contextual']=='attacco'

_wall_followed_by_no_side_constraint = _synthetic_episode(
    1,[0.01,0.40,0.01,0.01,0.01,0.02],side='B_y_ge9',track=2,count=4)
_prior_wall = {'class_id':4,'first_timestamp_seconds':1.0,'last_timestamp_seconds':1.2,
               'player_team_side':'A_y_lt9','player_track_id':1,
               'peak_confidence':0.8,'reliable':True}
_after_wall_decision = contextualize_episode(
    _wall_followed_by_no_side_constraint,[_prior_wall])
assert _after_wall_decision['class_contextual']=='ricezione'
assert not any('muro' in rule for rule in _after_wall_decision['context_rules_applied'])

START_SECONDS = _ORIGINAL_START_SECONDS_FOR_TESTS
print('Test CPU: matching, real-score relabel serve→ricezione, top-only fallback, disabled mode, score gap, episodio corto, ace/no-serve, attacco altro lato, muro senza geometria rete e battuta in salto superati.')

Test CPU: matching, real-score relabel serve→ricezione, top-only fallback, disabled mode, score gap, episodio corto, ace/no-serve, attacco altro lato, muro senza geometria rete e battuta in salto superati.


## 4. Controllo copertura e anteprima tracce

Costruisce indici solo per i frame richiesti e mostra esempi dei CSV esistenti senza ridisegnare/ricaricare modelli tracking. Le righe `predetta` della palla sono conservate esplicitamente come tali. Prima di attivare inferenza verificare che timestamp e identificativi corrispondano alla clip scelta.

In [4]:
clip_ball = ball_df[(ball_df.frame_index >= START_FRAME) & (ball_df.frame_index < END_FRAME_EXCLUSIVE)].copy()
clip_players = player_df[(player_df.frame_index >= START_FRAME) & (player_df.frame_index < END_FRAME_EXCLUSIVE)].copy()

def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in ('true', '1', 'yes')

player_by_frame = defaultdict(list)
for row in clip_players.to_dict('records'):
    if not as_bool(row['output_included']) or row['association_status'] != 'current_detection_associated_filtered_box':
        continue
    if pd.isna(row['track_id']):
        continue
    player_by_frame[int(row['frame_index'])].append(row)
ball_by_frame = defaultdict(list)
for row in clip_ball.to_dict('records'):
    ball_by_frame[int(row['frame_index'])].append(row)
requested_frames = list(range(START_FRAME, END_FRAME_EXCLUSIVE, FRAME_STRIDE))
# Adatta esplicitamente le chiavi del detector raw e conserva una breve storia causale.
_attach_soft_context_impl = attach_soft_context
_make_episode_id_impl = make_episode_id
_CONTEXT_HISTORY = []

def attach_soft_context(row, history, simultaneous):
    normalized = dict(row)
    normalized.setdefault('class_name', normalized.get('class_raw'))
    normalized.setdefault('confidence', normalized.get('confidence_raw'))
    normalized.setdefault('class_id', normalized.get('gesture_class_id'))
    prior = []
    for item in _CONTEXT_HISTORY:
        gap = normalized['timestamp_seconds'] - item['timestamp_seconds']
        if not 0 <= gap <= max(SERVE_RESPONSE_WINDOW_SECONDS, SAME_SIDE_ATTACK_WINDOW_SECONDS):
            continue
        same_actor_episode = (normalized.get('class_id') == item.get('class_id')
                              and normalized.get('player_track_id') is not None
                              and normalized.get('player_track_id') == item.get('player_track_id')
                              and gap <= EPISODE_GAP_SECONDS)
        if not same_actor_episode:
            prior.append(item)
    current = []
    for item in simultaneous:
        value = dict(item)
        value.setdefault('class_name', value.get('class_raw'))
        value.setdefault('class_id', value.get('gesture_class_id'))
        current.append(value)
    result = _attach_soft_context_impl(normalized, prior, current)
    _CONTEXT_HISTORY.append({'class_name': result['class_name'], 'class_id': result['class_id'],
        'timestamp_seconds': result['timestamp_seconds'], 'player_team_side': result.get('player_team_side'),
        'player_track_id': result.get('player_track_id'), 'bbox': result.get('bbox')})
    row.update(result)
    return row

def make_episode_id(row, open_episodes, frame_index, fps):
    normalized = dict(row)
    normalized.setdefault('class_id', normalized.get('gesture_class_id'))
    return _make_episode_id_impl(normalized, open_episodes, frame_index, fps)

_CONTEXT_HISTORY.clear()
same_time_serve = {'class_name':'battuta','class_raw':'battuta','class_id':0,'gesture_class_id':0,
    'timestamp_seconds':1.0,'confidence_raw':0.8,'player_team_side':'A_y_lt9',
    'player_track_id':1,'player_association':'associata','ball_position_type':'osservata'}
same_time_receive = {'class_name':'ricezione','class_raw':'ricezione','class_id':1,'gesture_class_id':1,
    'timestamp_seconds':1.0,'confidence_raw':0.8,'player_team_side':'B_y_ge9',
    'player_track_id':2,'player_association':'associata','ball_position_type':'osservata'}
attach_soft_context(dict(same_time_serve), [], [same_time_serve])
assert attach_soft_context(dict(same_time_receive), [], [same_time_receive])['soft_adjustment'] == 0
prior_serve = dict(same_time_serve, timestamp_seconds=2.0)
_CONTEXT_HISTORY.clear()
attach_soft_context(dict(prior_serve), [], [prior_serve])
later_receive = dict(same_time_receive, timestamp_seconds=3.0)
assert attach_soft_context(later_receive, [], [later_receive])['soft_adjustment'] > 0
_CONTEXT_HISTORY.clear()
coverage = {
    'video_path': str(VIDEO_PATH), 'video_fps': VIDEO_FPS, 'video_frame_count': VIDEO_FRAME_COUNT,
    'clip_interval_seconds': [START_SECONDS, END_SECONDS], 'frame_interval': [START_FRAME, END_FRAME_EXCLUSIVE],
    'stride': FRAME_STRIDE, 'expected_sample_frames': len(requested_frames),
    'frames_with_player_rows': sum(frame in player_by_frame for frame in requested_frames),
    'frames_with_ball_rows': sum(frame in ball_by_frame for frame in requested_frames),
    'ball_observed_rows': int(clip_ball.position_type.eq('osservata').sum()),
    'ball_kalman_predicted_rows': int(clip_ball.position_type.eq('predetta').sum()),
    'player_source_rows': len(clip_players), 'player_eligible_associated_rows': sum(map(len, player_by_frame.values())),
    'warnings': ['Frames senza righe traccia palla indicano assenza di osservazione/predizione, non prova di ace o fine copertura.']}
for label, count in (('giocatori', coverage['frames_with_player_rows']), ('palla', coverage['frames_with_ball_rows'])):
    if count < len(requested_frames):
        coverage['warnings'].append(f'Frame senza righe traccia {label}: {len(requested_frames)-count}/{len(requested_frames)}')
print(json.dumps(coverage, indent=2, ensure_ascii=False))
print('\nPalla:')
display(clip_ball.head(6))
print('\nGiocatori associati:')
display(clip_players[clip_players.output_included.astype(str).str.lower().isin(['true','1'])].head(6))

{
  "video_path": "C:\\Users\\SavarDar\\OneDrive - Electrolux\\Desktop\\Scout_Volley_CV&NLP\\Annotazioni\\Test_demo_ITA_DOM_Parigi2024_30fps.mp4",
  "video_fps": 30.0,
  "video_frame_count": 221230,
  "clip_interval_seconds": [
    680.0,
    7350.0
  ],
  "frame_interval": [
    20400,
    220500
  ],
  "stride": 1,
  "expected_sample_frames": 200100,
  "frames_with_player_rows": 198562,
  "frames_with_ball_rows": 103486,
  "ball_observed_rows": 85436,
  "ball_kalman_predicted_rows": 18050,
  "player_source_rows": 1921965,
  "player_eligible_associated_rows": 1921965,
  "warnings": [
    "Frames senza righe traccia palla indicano assenza di osservazione/predizione, non prova di ace o fine copertura.",
    "Frame senza righe traccia giocatori: 1538/200100",
    "Frame senza righe traccia palla: 96614/200100"
  ]
}

Palla:


,timestamp_seconds,frame_index,track_id,confidence,x_px,y_px,position_type,bbox_x1,bbox_y1,bbox_x2,bbox_y2,_source_csv
0,686.166667,20585,1,0.776534,12.185259,176.384384,osservata,2.161070,167.189194,22.209448,185.579575,C:\Users\SavarDar\OneDrive - Electrolux\Deskto...
1,686.200000,20586,1,0.822054,23.735062,177.088928,osservata,13.888020,167.613159,33.582104,186.564697,C:\Users\SavarDar\OneDrive - Electrolux\Deskto...
2,686.233333,20587,1,0.812787,35.305431,178.408142,osservata,25.911270,169.013397,44.699593,187.802887,C:\Users\SavarDar\OneDrive - Electrolux\Deskto...
3,686.266667,20588,1,0.784023,47.367001,180.530457,osservata,38.521255,170.719696,56.212746,190.341217,C:\Users\SavarDar\OneDrive - Electrolux\Deskto...
4,686.300000,20589,1,0.790229,58.589027,183.033386,osservata,49.045334,173.977448,68.132721,192.089325,C:\Users\SavarDar\OneDrive - Electrolux\Deskto...
5,686.333333,20590,1,0.810794,70.633181,186.407898,osservata,61.275867,177.602692,79.990494,195.213104,C:\Users\SavarDar\OneDrive - Electrolux\Deskto...



Giocatori associati:


,frame_index,timestamp_seconds,row_type,detection_index,class_id,class_name,confidence,bbox_x1,bbox_y1,bbox_x2,...,ocr_confidence,number_confirmed,number_confirmed_confidence,number_read_status,number_last_valid_read_seconds,number_confirmed_at_seconds,number_vote_count,number_read_reason,number_crop_stddev,jersey_crop_path
0,20400,680.0,detection_yolo,0,0,player,0.876196,352.137939,503.004028,417.234436,...,NaN,NaN,NaN,unread,NaN,NaN,0,no_numeric_token,69.411262,jersey_crops\frame_020400_id_1.png
1,20400,680.0,detection_yolo,1,0,player,0.852090,609.460571,391.626282,680.239380,...,97.637632,NaN,NaN,candidate_unconfirmed,680.0,NaN,1,numeric_candidate,53.499652,jersey_crops\frame_020400_id_2.png
2,20400,680.0,detection_yolo,2,0,player,0.850862,819.955444,334.313049,875.804932,...,NaN,NaN,NaN,unread,NaN,NaN,0,no_numeric_token,55.436624,jersey_crops\frame_020400_id_3.png
3,20400,680.0,detection_yolo,3,0,player,0.843158,572.008789,296.374451,616.750244,...,NaN,NaN,NaN,unread,NaN,NaN,0,no_numeric_token,48.312684,jersey_crops\frame_020400_id_4.png
4,20400,680.0,detection_yolo,4,0,player,0.839478,887.387695,398.179138,964.858032,...,NaN,NaN,NaN,unread,NaN,NaN,0,no_numeric_token,40.549045,jersey_crops\frame_020400_id_5.png
5,20400,680.0,detection_yolo,5,0,player,0.835964,252.436096,328.379395,311.758484,...,NaN,NaN,NaN,unread,NaN,NaN,0,no_numeric_token,57.773613,jersey_crops\frame_020400_id_6.png


## 5. Inferenza video opt-in ed export contestuale

La prima passata elabora solo `range(ceil(start*FPS), ceil(end*FPS), stride)`, raccoglie raw detection e score reali. La seconda passata annota il video opzionale con la decisione per episodio (puo usare frame futuri, quindi e offline). Il video viene verificato dopo la scrittura; se i codec non sono disponibili o il file non e valido, il report contiene un avviso e i CSV restano disponibili. Timestamp/video output restano coerenti: FPS sorgente/stride e intervallo temporale registrato. Niente tracking/OCR o modello palla/giocatori viene rilanciato.

Un episodio viene corretto solo se tutti i requisiti documentati sono soddisfatti; il vettore ripesato non e calibrato e non crea una classe assente dall'head. Top-only o vettore incompleto = raw invariato, score alternativi vuoti/esplicitamente incompleti. Nessun full-match run e stato richiesto; la finestra predefinita e un secondo.

Per ciascun valore di `OPPONENT`, l'export scrive `proposte_<PARTITA>.csv` e `embedding_<PARTITA>.csv` nella directory univoca della run. Con un track giocatore affidabile, le detection dello stesso track vengono raggruppate indipendentemente dalla classe raw, usando gap temporali e picchi di confidence; track diversi restano distinti anche nello stesso frame. Senza track il fallback resta per episodio di classe/IoU, per non fondere spazialmente giocatori distinti. Ogni gruppo produce un evento stimato con l'ID della detection al massimo di confidence, comune ai due file; score, coordinate e embedding sono di quel frame e la classe/probabilità restano quelle raw di quella detection. Tutte le detection, incluse quelle false, sono assegnate a un gruppo. Due tocchi vicini dello stesso track possono ancora fondersi se i picchi cadono sotto la distanza minima o manca un avvallamento di confidence sufficiente; track instabili/frammentati possono invece dividere un tocco. `frame_event_map_<PARTITA>.csv` permette di risalire da ogni detection frame al gruppo, agli episodi raw sorgente e al picco; parametri e copertura sono nel report. Lo scout non determina gruppi o picchi. `conf_tocco` e' confidence del detector al frame scelto, non probabilita calibrata di contatto. L'embedding e' l'input alla convoluzione finale della testa di classificazione, letto alla cella spaziale dell'anchor mantenuto dalla NMS: dimensione originale, senza PCA o riduzione; se manca resta vuoto. `palla_x,palla_y` usano solo punti palla osservati nello stesso frame, associati entro la soglia pixel al centro della detection e proiettati sul piano del campo con l'omografia confermata; Kalman e proiezioni fuori campo sono vuoti. Per una palla in volo e' una stima dell'intersezione del raggio immagine col piano, non una misura certa della posizione fisica a terra. Il notebook non rilancia training o tracking e non esegue cross-validation.

In [5]:
def group_touch_proposals(detections, proposals, max_gap, min_peak_distance, min_prominence):
    """Group tracked-player detections across classes by gap and confidence peaks; never use scout."""
    if (not all(math.isfinite(value) for value in (max_gap, min_peak_distance, min_prominence))
            or max_gap < 0 or min_peak_distance < 0 or not 0 <= min_prominence <= 1):
        raise ValueError('Parametri di raggruppamento tocchi non validi.')
    by_id = {row['id']: row for row in proposals}
    if len(by_id) != len(proposals) or len({row['id'] for row in detections}) != len(detections):
        raise ValueError('ID detection duplicati nel raggruppamento.')
    if set(by_id) != {row['id'] for row in detections}:
        raise ValueError('ID detection e proposte frame-level non allineati.')
    streams_by_key = defaultdict(list)
    for detection in detections:
        proposal = by_id[detection['id']]
        if (int(proposal['fotogramma']) != int(detection['frame_index'])
                or not math.isclose(float(proposal['tempo_s']), float(detection['timestamp_seconds']), abs_tol=1e-8)):
            raise ValueError('Frame o tempo della proposta non coincide con la detection.')
        confidence = float(proposal['conf_tocco'])
        if not math.isfinite(confidence) or not 0 <= confidence <= 1:
            raise ValueError('Confidence della detection non valida.')
        track_id = detection.get('player_track_id')
        try:
            has_track = track_id is not None and math.isfinite(float(track_id))
        except (TypeError, ValueError):
            has_track = track_id is not None and bool(str(track_id).strip())
        grouping_key = ('track', str(track_id)) if has_track else ('episode_fallback', str(detection['episode_id']))
        streams_by_key[grouping_key].append(detection)
    groups = []
    for grouping_key, episode_rows in streams_by_key.items():
        ordered = sorted(episode_rows, key=lambda row: (row['timestamp_seconds'], row['id']))
        runs = []
        for row in ordered:
            if not runs or row['timestamp_seconds'] - runs[-1][-1]['timestamp_seconds'] > max_gap + 1e-9:
                runs.append([])
            runs[-1].append(row)
        for run in runs:
            # One score per sampled frame for peak finding; all detections remain in the audit.
            frame_best = {}
            for row in run:
                frame = int(row['frame_index'])
                if frame not in frame_best or by_id[row['id']]['conf_tocco'] > by_id[frame_best[frame]['id']]['conf_tocco']:
                    frame_best[frame] = row
            series = sorted(frame_best.values(), key=lambda row: (row['timestamp_seconds'], row['id']))
            scores = [float(by_id[row['id']]['conf_tocco']) for row in series]
            peaks = []
            i = 0
            while i < len(scores):
                j = i
                while j + 1 < len(scores) and scores[j + 1] == scores[i]:
                    j += 1
                if (i == 0 or scores[i] > scores[i - 1]) and (j == len(scores) - 1 or scores[j] > scores[j + 1]):
                    peaks.append((i + j) // 2)
                i = j + 1
            eligible = []
            for k, peak in enumerate(peaks):
                drops = []
                left = peaks[k - 1] if k else 0
                right = peaks[k + 1] if k + 1 < len(peaks) else len(scores) - 1
                if left < peak:
                    drops.append(scores[peak] - min(scores[left:peak + 1]))
                if peak < right:
                    drops.append(scores[peak] - min(scores[peak:right + 1]))
                if not drops or min(drops) + 1e-12 >= min_prominence:
                    eligible.append(peak)
            strongest = max(peaks, key=lambda index: (scores[index], -series[index]['timestamp_seconds']))
            if strongest not in eligible:
                eligible.append(strongest)
            selected = []
            peak_buckets = defaultdict(list)
            for peak in sorted(eligible, key=lambda index: (-scores[index], series[index]['timestamp_seconds'])):
                time = series[peak]['timestamp_seconds']
                bucket = math.floor(time / min_peak_distance) if min_peak_distance > 0 else 0
                nearby = (other for key in (bucket - 1, bucket, bucket + 1) for other in peak_buckets[key])
                if all(abs(time - series[other]['timestamp_seconds']) + 1e-9 >= min_peak_distance for other in nearby):
                    selected.append(peak)
                    peak_buckets[bucket].append(peak)
            selected.sort(key=lambda index: series[index]['timestamp_seconds'])
            boundaries = [(series[a]['timestamp_seconds'] + series[b]['timestamp_seconds']) / 2
                          for a, b in zip(selected, selected[1:])]
            segments = [[] for _ in selected]
            segment_index = 0
            for row in run:
                while segment_index < len(boundaries) and row['timestamp_seconds'] > boundaries[segment_index]:
                    segment_index += 1
                segments[segment_index].append(row)
            for segment in segments:
                peak = min(segment, key=lambda row: (-float(by_id[row['id']]['conf_tocco']), row['timestamp_seconds'], row['id']))
                groups.append((peak, segment, grouping_key))
    groups.sort(key=lambda group: (group[0]['timestamp_seconds'], group[0]['id']))
    event_proposals, audit = [], []
    for peak, segment, grouping_key in groups:
        event_id = peak['id']
        event_proposals.append(dict(by_id[event_id]))
        start, end = segment[0]['timestamp_seconds'], segment[-1]['timestamp_seconds']
        frame_count = len({row['frame_index'] for row in segment})
        source_episode_ids = sorted({str(row['episode_id']) for row in segment})
        for row in segment:
            audit.append({'id': event_id, 'detection_id': row['id'], 'episode_id': row['episode_id'],
                          'source_episode_ids': '|'.join(source_episode_ids),
                          'grouping_key': '|'.join(grouping_key),
                          'fotogramma': row['frame_index'], 'tempo_s': row['timestamp_seconds'],
                          'confidence_frame': by_id[row['id']]['conf_tocco'],
                          'class_raw': row['class_raw'], 'is_peak': row['id'] == event_id,
                          'group_start_s': start, 'group_end_s': end,
                          'group_detection_count': len(segment), 'group_distinct_frames': frame_count})
    if len(audit) != len(detections) or len({row['detection_id'] for row in audit}) != len(detections):
        raise RuntimeError('Il raggruppamento non copre ogni detection esattamente una volta.')
    return event_proposals, audit

def _synthetic_touch_group_tests():
    detection_rows, frame_rows = [], []
    for detection_id, frame, score, episode in (
            (1, 0, .4, 'player1'), (2, 3, .92, 'player1'), (3, 6, .3, 'player1'),
            (4, 9, .3, 'player1'), (5, 12, .97, 'player1'), (6, 15, .4, 'player1'),
            (7, 21, .58, 'false_positive'), (8, 12, .87, 'player2')):
        time = frame / 30.
        class_name = ('battuta', 'alzata', 'attacco', 'attacco', 'difesa', 'ricezione', 'muro', 'muro')[detection_id - 1]
        track_id = (1, 1, 1, 1, 1, 1, 3, 2)[detection_id - 1]
        detection_rows.append({'id': detection_id, 'frame_index': frame, 'timestamp_seconds': time,
                               'episode_id': episode, 'class_raw': class_name, 'player_track_id': track_id})
        frame_rows.append({'id': detection_id, 'fotogramma': frame, 'tempo_s': time,
                           'conf_tocco': score, 'palla_x': detection_id})
    events, audit = group_touch_proposals(detection_rows, frame_rows, .2, .3, .05)
    assert [row['id'] for row in events] == [2, 5, 8, 7]
    assert events[1]['palla_x'] == 5
    assert len(audit) == len(detection_rows) and len({row['detection_id'] for row in audit}) == len(audit)
    assert {row['id'] for row in audit if row['is_peak']} == {2, 5, 7, 8}
    assert sum(row['id'] == 2 for row in audit) == 3 and sum(row['id'] == 5 for row in audit) == 3
    assert {row['class_raw'] for row in audit if row['id'] == 2} == {'battuta', 'alzata', 'attacco'}
    assert {row['id'] for row in audit if row['fotogramma'] == 12} >= {5, 8}
    assert {row['grouping_key'] for row in audit if row['id'] in (5, 8)} == {'track|1', 'track|2'}
    close_frames = [0, 3, 6, 12, 15, 18]
    close_scores = [.3, .92, .3, .3, .90, .3]
    close_detections, close_proposals = [], []
    for detection_id, (frame, score) in enumerate(zip(close_frames, close_scores), 20):
        time = frame / 30.
        close_detections.append({'id': detection_id, 'frame_index': frame, 'timestamp_seconds': time,
                                 'episode_id': f'class-{detection_id}', 'class_raw': 'alzata' if detection_id < 24 else 'attacco',
                                 'player_track_id': 9})
        close_proposals.append({'id': detection_id, 'fotogramma': frame, 'tempo_s': time, 'conf_tocco': score})
    close_events, close_audit = group_touch_proposals(close_detections, close_proposals, .2, .3, .05)
    assert [row['id'] for row in close_events] == [21, 24]
    assert group_touch_proposals([], [], .2, .3, .05) == ([], [])

_synthetic_touch_group_tests()

def open_context_video_writer(run_dir, fps, frame_size):
    failures = []
    for codec, suffix in (('mp4v', '.mp4'), ('MJPG', '.avi')):
        candidate = run_dir / ('video_gesti_contestuali' + suffix)
        if candidate.exists():
            raise FileExistsError(f'Video esistente, non sovrascritto: {candidate}')
        attempt = None
        opened = False
        try:
            attempt = cv2.VideoWriter(str(candidate), cv2.VideoWriter_fourcc(*codec), fps, frame_size)
            opened = attempt.isOpened()
            if opened:
                return attempt, candidate, None
            failures.append(f'{codec}: apertura fallita')
        except cv2.error as exc:
            failures.append(f'{codec}: {exc}')
        finally:
            if attempt is not None and not opened:
                attempt.release()
        if candidate.exists():
            candidate.unlink()
    return None, None, 'Codec video contestuale non disponibile (' + '; '.join(failures) + ').'

def validate_context_video(path, expected_frames, expected_fps):
    video = None
    try:
        video = cv2.VideoCapture(str(path))
        if not video.isOpened():
            return None, None, f'Video contestuale non riapribile: {path}'
        frame_count = float(video.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(video.get(cv2.CAP_PROP_FPS))
        readable, _ = video.read()
        if (not readable or not math.isfinite(frame_count) or frame_count != expected_frames
                or not math.isfinite(fps) or fps <= 0 or not math.isclose(fps, expected_fps, rel_tol=0.01)):
            return None, None, f'Video contestuale non coerente: frames={frame_count}/{expected_frames}, fps={fps}.'
        return int(frame_count), fps, None
    except cv2.error as exc:
        return None, None, f'Verifica video contestuale fallita: {exc}'
    finally:
        if video is not None:
            video.release()

def report_optional_video_warning(message, coverage):
    warnings.warn(message)
    coverage.setdefault('warnings', []).append(message)

INFERENCE_COMPLETED = False
if not RUN_INFERENCE:
    print('Inferenza disattivata. Verificare preflight, tracce e intervallo breve; poi impostare RUN_INFERENCE=True.')
else:
    if sha256_file(GESTURE_CHECKPOINT) != CHECKPOINT_SHA256:
        raise RuntimeError('Checkpoint cambiato dopo preflight; riavviare dal preflight.')
    if RUN_DIR.exists():
        raise FileExistsError(f'Run esistente, non sovrascritto: {RUN_DIR}')
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    DETECTIONS_CSV = RUN_DIR / 'detections_frame.csv'
    EVENTS_CSV = RUN_DIR / 'episodi_gesto.csv'
    CONFIG_PATH = RUN_DIR / 'configurazione_run.json'
    REPORT_PATH = RUN_DIR / 'copertura_limiti.json'
    PREVIEW_PATH = RUN_DIR / 'anteprima.jpg'
    detection_rows, episodes = [], []
    proposal_rows = []
    preclass_features = {}
    open_episodes = []
    PREDICTOR = None
    _CONTEXT_HISTORY.clear()
    writer = None
    output_video_path = None
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    if not cap.isOpened():
        raise OSError(f'Video non apribile: {VIDEO_PATH}')
    if not cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME):
        cap.release()
        raise OSError(f'Impossibile posizionarsi al frame iniziale {START_FRAME}.')
    processed_frames = 0
    sampled_frame_indices = []
    try:
        for frame_index in requested_frames:
            if frame_index != START_FRAME:
                # Legge il frame precedente e avanza fino all'indice richiesto.
                for _ in range(FRAME_STRIDE):
                    ok, frame = cap.read()
                if not ok:
                    break
            else:
                ok, frame = cap.read()
            if not ok or frame is None:
                raise RuntimeError(f'Lettura interrotta al frame {frame_index}.')
            pos_after = float(cap.get(cv2.CAP_PROP_POS_FRAMES))
            if math.isfinite(pos_after) and abs(pos_after - (frame_index + 1)) > 1.1:
                raise RuntimeError(f'OpenCV frame-index incoerente: atteso {frame_index}, posizione dopo read={pos_after}.')
            timestamp = frame_index / VIDEO_FPS
            if not (START_SECONDS - 0.51/VIDEO_FPS <= timestamp < END_SECONDS + 0.01/VIDEO_FPS):
                raise RuntimeError(f'Frame {frame_index} fuori dall’intervallo configurato: {timestamp}.')
            player_candidates = player_by_frame.get(frame_index, [])
            ball_candidates = ball_by_frame.get(frame_index, [])
            model_candidates, frame_score_mode = predict_frame_candidates(frame)
            frame_detections = []
            for detection_index, candidate in enumerate(model_candidates):
                box = list(map(float, candidate['bbox']))
                class_id = int(candidate['class_id'])
                confidence = float(candidate['confidence'])
                touch_id = len(detection_rows) + len(frame_detections) + 1
                feature_vector = candidate.get('embedding_feature')
                if feature_vector is not None:
                    preclass_features[touch_id] = feature_vector
                if not 0 <= class_id < len(CLASS_NAMES):
                    raise ValueError(f'ID gesto fuori mapping: {class_id}')
                player_context = associate_player(box, player_candidates)
                ball_context = nearest_ball(box, ball_candidates)
                row = {'frame_index': frame_index, 'timestamp_seconds': timestamp,
                       'id': touch_id, 'detection_index': detection_index, 'gesture_class_id': class_id,
                       'class_id': class_id, 'class_name': CLASS_NAMES[class_id],
                       'class_raw': CLASS_NAMES[class_id], 'class_corrected': CLASS_NAMES[class_id],
                       'confidence_raw': confidence, 'raw_class_scores': candidate['raw_class_scores'],
                       'score_source_mode': candidate['score_mode'], 'bbox_x1': box[0], 'bbox_y1': box[1],
                       'bbox_x2': box[2], 'bbox_y2': box[3], 'bbox': box,
                       **player_context, **ball_context}
                raw_scores = candidate['raw_class_scores']
                probabilities = normalize_raw_class_scores(raw_scores, len(CLASS_NAMES))
                ball_x_m, ball_y_m = observed_ball_ground_position(
                    box, ball_candidates, CAMERA_H_IMAGE_TO_WORLD)
                proposal_rows.append({
                    'id': touch_id, 'fotogramma': frame_index, 'tempo_s': timestamp, 'squadra': '',
                    'p_battuta': '' if probabilities is None else float(probabilities[0]),
                    'p_ricezione': '' if probabilities is None else float(probabilities[1]),
                    'p_alzata': '' if probabilities is None else float(probabilities[2]),
                    'p_attacco': '' if probabilities is None else float(probabilities[3]),
                    'p_muro': '' if probabilities is None else float(probabilities[4]),
                    'p_difesa': '' if probabilities is None else float(probabilities[5]),
                    'palla_x': '' if ball_x_m is None else ball_x_m,
                    'palla_y': '' if ball_y_m is None else ball_y_m,
                    'giocatore_x': player_context['player_x_m'] if math.isfinite(player_context['player_x_m']) else '',
                    'giocatore_y': player_context['player_y_m'] if math.isfinite(player_context['player_y_m']) else '',
                    'conf_tocco': confidence,
                })
                frame_detections.append(row)
            simultaneous = list(frame_detections)
            for row in frame_detections:
                attach_soft_context(row, episodes, simultaneous)
                episode_id = make_episode_id(row, open_episodes, frame_index, VIDEO_FPS)
                row['episode_id'] = episode_id
                row['episode_start'] = not any(ep['episode_id'] == episode_id for ep in open_episodes)
                row['sequence_context'] = row['rule_evidence']
                detection_rows.append({key: value for key, value in row.items() if key != 'bbox'})
                matching = next((ep for ep in open_episodes if ep['episode_id'] == episode_id), None)
                if matching is None:
                    matching = {'episode_id': episode_id, 'class_id': row['gesture_class_id'],
                                'class_name': row['class_raw'], 'class_corrected': row['class_corrected'],
                                'player_track_id': row['player_track_id'], 'player_team_side': row['player_team_side'],
                                'first_frame': frame_index, 'last_frame': frame_index,
                                'first_timestamp_seconds': timestamp, 'last_timestamp_seconds': timestamp,
                                'bbox': list(row['bbox']), 'last_bbox': list(row['bbox']), 'peak_confidence': row['confidence_raw'],
                                'detection_count': 0, 'frames': set(), 'rule_evidence': set(),
                                'soft_adjustment_sum': 0.0, 'ball_evidence_types': set(),
                                'association_statuses': set(), 'raw_score_vectors': [],
                                'score_mode_set': set(), 'frame_evidence': []}
                    open_episodes.append(matching)
                matching['last_frame'] = frame_index
                matching['last_timestamp_seconds'] = timestamp
                matching['last_bbox'] = list(row['bbox'])
                matching['bbox'] = [min(matching['bbox'][0], box[0]), min(matching['bbox'][1], box[1]),
                                    max(matching['bbox'][2], box[2]), max(matching['bbox'][3], box[3])]
                matching['peak_confidence'] = max(matching['peak_confidence'], row['confidence_raw'])
                matching['detection_count'] += 1
                matching['frames'].add(frame_index)
                matching['rule_evidence'].update(row['rule_evidence'].split('|'))
                matching['soft_adjustment_sum'] += row['soft_adjustment']
                matching['ball_evidence_types'].add(row['ball_position_type'])
                matching['association_statuses'].add(row['player_association'])
                matching['score_mode_set'].add(row['score_source_mode'])
                if row['raw_class_scores'] is not None:
                    matching['raw_score_vectors'].append(row['raw_class_scores'])
                matching['frame_evidence'].append({'ball_position_type': row['ball_position_type'],
                                                   'ball_distance_px': row['ball_distance_px']})
            processed_frames += 1
            sampled_frame_indices.append(frame_index)
            if processed_frames % 30 == 0:
                print(f'Frame {processed_frames}/{len(requested_frames)} | t={timestamp:.3f}s | detections={len(frame_detections)}')
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    if processed_frames == 0:
        raise RuntimeError('Nessun frame video elaborato.')
    if sampled_frame_indices != requested_frames[:processed_frames]:
        raise RuntimeError('Sequenza frame campionata diversa da quella prevista.')

    contextual_episodes = contextualize_episodes(
        open_episodes, detection_rows, enabled=ENABLE_CONTEXTUAL_CLASS_CORRECTION)
    episode_by_id = {episode['episode_id']: episode for episode in contextual_episodes}

    # Re-render in a second pass so retrospective episode labels appear in the video too.
    output_video_path = None
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    if not cap.isOpened() or not cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME):
        cap.release()
        raise OSError('Impossibile riaprire/posizionarsi sul video per annotare le classi contestuali.')
    annotated_writer, output_video_path, video_warning = open_context_video_writer(
        RUN_DIR, VIDEO_FPS / FRAME_STRIDE, (VIDEO_WIDTH, VIDEO_HEIGHT))
    if video_warning:
        report_optional_video_warning(video_warning, coverage)
    detections_by_frame = defaultdict(list)
    for row in detection_rows:
        detections_by_frame[int(row['frame_index'])].append(row)
    video_render_complete = False
    try:
        for frame_index in sampled_frame_indices:
            if frame_index == START_FRAME:
                ok, source_frame = cap.read()
            else:
                for _ in range(FRAME_STRIDE):
                    ok, source_frame = cap.read()
            if not ok or source_frame is None:
                raise RuntimeError(f'Seconda lettura video interrotta al frame {frame_index}.')
            rendered = source_frame.copy()
            for row in detections_by_frame.get(frame_index, []):
                x1,y1,x2,y2 = map(int, np.rint([row['bbox_x1'],row['bbox_y1'],row['bbox_x2'],row['bbox_y2']]))
                class_id = CLASS_NAMES.index(row['class_contextual'])
                color = [(0,220,220),(255,160,20),(180,80,255),(0,0,255),(255,80,80),(60,220,60)][class_id]
                cv2.rectangle(rendered, (x1,y1), (x2,y2), color, 2)
                player_text = f" ID{row['player_track_id']}" if pd.notna(row['player_track_id']) else ''
                label = f"{row['class_raw']}→{row['class_contextual']} {row['confidence_raw']:.2f}{player_text}"
                cv2.putText(rendered, label, (x1,max(18,y1-6)), cv2.FONT_HERSHEY_SIMPLEX,.55,color,2,cv2.LINE_AA)
            timestamp = frame_index / VIDEO_FPS
            cv2.rectangle(rendered,(0,0),(min(VIDEO_WIDTH-1,520),34),(0,0,0),-1)
            cv2.putText(rendered,f't={timestamp:.3f}s frame={frame_index} gestures={len(detections_by_frame.get(frame_index,[]))}',
                        (8,24),cv2.FONT_HERSHEY_SIMPLEX,.62,(255,255,255),1,cv2.LINE_AA)
            if frame_index == sampled_frame_indices[0] and not cv2.imwrite(str(PREVIEW_PATH),rendered):
                raise OSError(f'Impossibile scrivere anteprima {PREVIEW_PATH}')
            if annotated_writer is None:
                break
            try:
                annotated_writer.write(rendered)
            except cv2.error as exc:
                video_warning = f'Scrittura video contestuale fallita: {exc}'
                report_optional_video_warning(video_warning, coverage)
                annotated_writer.release()
                annotated_writer = None
                if output_video_path.exists():
                    output_video_path.unlink()
                output_video_path = None
                break
        video_render_complete = True
    finally:
        cap.release()
        if annotated_writer is not None:
            annotated_writer.release()
        if not video_render_complete and output_video_path is not None and output_video_path.exists():
            output_video_path.unlink()
    detection_columns = ['id','frame_index','timestamp_seconds','detection_index','gesture_class_id','class_raw',
        'class_corrected','class_contextual','confidence_raw','score_source_mode','raw_class_scores_json',
        'context_score_mode','context_raw_score_vector','context_adjusted_score_vector','context_rule_applied',
        'context_correction_reason','context_correction_performed','raw_score_battuta','raw_score_ricezione',
        'raw_score_alzata','raw_score_attacco','raw_score_muro','raw_score_difesa','context_score_battuta',
        'context_score_ricezione','context_score_alzata','context_score_attacco','context_score_muro','context_score_difesa',
        'soft_adjustment','ranking_score_diagnostic_only','correction_reason','rule_evidence','sequence_context','bbox_x1','bbox_y1','bbox_x2','bbox_y2','episode_id','episode_start',
        'player_track_id','player_team_side','player_association','player_iou','player_center_distance_px','player_score',
        'ball_track_id','ball_position_type','ball_distance_px','ball_confidence']
    for row in detection_rows:
        row['raw_class_scores_json'] = json.dumps(row['raw_class_scores']) if row.get('raw_class_scores') is not None else ''
    pd.DataFrame(detection_rows, columns=detection_columns).to_csv(DETECTIONS_CSV, index=False, encoding='utf-8-sig')
    if EMBEDDING_EXPORT_ENABLED:
        frame_proposal_count = len(proposal_rows)
        proposal_rows, event_audit_rows = group_touch_proposals(
            detection_rows, proposal_rows, TOUCH_GROUP_MAX_GAP_SECONDS,
            TOUCH_PEAK_MIN_DISTANCE_SECONDS, TOUCH_PEAK_MIN_PROMINENCE)
        audit_path = RUN_DIR / f'frame_event_map_{OPPONENT}.csv'
        audit_columns = ['id', 'detection_id', 'episode_id', 'source_episode_ids', 'grouping_key', 'fotogramma', 'tempo_s',
                         'confidence_frame', 'class_raw', 'is_peak', 'group_start_s', 'group_end_s',
                         'group_detection_count', 'group_distinct_frames']
        pd.DataFrame(event_audit_rows, columns=audit_columns).to_csv(audit_path, index=False, encoding='utf-8-sig')
        coverage['proposal_grouping'] = {
            'method': 'heuristic: same tracked player across raw classes, split on detection gaps and confidence peaks',
            'class_independent_for_tracked_players': True,
            'class_independent_for_trackless_fallback': False,
            'simultaneous_player_isolation': 'separate groups for distinct player track IDs',
            'uses_scout': False,
            'trackless_episode_gap_seconds': EPISODE_GAP_SECONDS,
            'episode_trackless_iou_min': 0.5,
            'max_detection_gap_seconds': TOUCH_GROUP_MAX_GAP_SECONDS,
            'min_seed_peak_distance_seconds': TOUCH_PEAK_MIN_DISTANCE_SECONDS,
            'min_peak_prominence': TOUCH_PEAK_MIN_PROMINENCE,
            'peak_prominence_definition': 'minimum confidence drop to adjacent local-peak valley or run edge; highest peak always retained',
            'group_boundary': 'midpoint in time between retained peaks',
            'event_frame_selection': 'maximum confidence inside each group; ties use earliest time then lowest detection ID; event frame can differ from seed peak',
            'frame_detection_rows': frame_proposal_count,
            'grouped_detection_rows': len(event_audit_rows),
            'event_rows': len(proposal_rows),
            'unique_frames_with_detections': len({row['fotogramma'] for row in event_audit_rows}),
            'groups_with_multiple_frames': sum(row['is_peak'] and row['group_distinct_frames'] > 1 for row in event_audit_rows),
            'largest_group_frames': max((row['group_distinct_frames'] for row in event_audit_rows), default=0),
            'frame_event_map_csv': str(audit_path),
            'limitations': 'estimated events, not verified contacts: close touches on one track can merge when peaks are less than the minimum distance or lack a confidence valley; trackless groups remain class/episode-specific to avoid unsafe spatial merging',
        }
        proposal_columns = ['id','fotogramma','tempo_s','squadra',
            'p_battuta','p_ricezione','p_alzata','p_attacco','p_muro','p_difesa',
            'conf_tocco','palla_x','palla_y','giocatore_x','giocatore_y']
        proposals_path = RUN_DIR / f'proposte_{OPPONENT}.csv'
        pd.DataFrame(proposal_rows, columns=proposal_columns).to_csv(
            proposals_path, index=False, encoding='utf-8-sig')
        detect_module = (PREDICTOR.model.model.model[-1] if PREDICTOR is not None
                         else MODEL.model.model[-1])
        embedding_dimensions = [int(classifier[-1].in_channels) for classifier in detect_module.cv3]
        if not embedding_dimensions or len(set(embedding_dimensions)) != 1:
            raise ValueError(f'Dimensione pre-classificatore non uniforme tra scale: {embedding_dimensions}')
        embedding_dim = embedding_dimensions[0]
        embedding_columns = ['id'] + [f'e{i}' for i in range(embedding_dim)]
        embedding_path = RUN_DIR / f'embedding_{OPPONENT}.csv'
        embedding_rows = []
        for row in proposal_rows:
            vector = preclass_features.get(row['id'])
            embedding_row = {'id': row['id']}
            if vector is None or vector.shape != (embedding_dim,) or not np.isfinite(vector).all():
                embedding_row.update({column: '' for column in embedding_columns[1:]})
            else:
                embedding_row.update({f'e{i}': f'{float(vector[i]):.4f}'
                                      for i in range(embedding_dim)})
            embedding_rows.append(embedding_row)
        if [row['id'] for row in embedding_rows] != [row['id'] for row in proposal_rows]:
            raise RuntimeError('Gli ID dei CSV proposte ed embedding non sono allineati.')
        pd.DataFrame(embedding_rows, columns=embedding_columns).to_csv(
            embedding_path, index=False, encoding='utf-8-sig')
        embedding_missing = sum(
            1 for row in embedding_rows
            if any(row[column] == '' for column in embedding_columns[1:]))
        embedding_status = (
            f'Per-detection pre-classifier features at retained NMS anchor; '
            f'{len(embedding_rows) - embedding_missing}/{len(embedding_rows)} vectors present; '
            f'dimension={embedding_dim}; no reduction applied.')
        coverage['embedding_status'] = embedding_status
        coverage['proposals_csv'] = str(proposals_path)
        coverage['embedding_csv'] = str(embedding_path)
        coverage['embedding_dimension'] = embedding_dim
        coverage['embedding_feature_source'] = 'input to final convolution of the winning class head at the NMS-selected anchor'
        coverage['embedding_missing_rows'] = embedding_missing
        coverage['proposal_probability_rows_missing'] = sum(
            1 for row in proposal_rows if row['p_battuta'] == '')
        coverage['proposal_probability_source'] = 'raw six-class sigmoid scores normalized to sum one; no contextual sequence correction'
        coverage['conf_tocco_source'] = 'peak-frame YOLO winning-class confidence; not a calibrated event/contact probability'
        coverage['team_source'] = 'not inferable from geometric tracker side; exported blank'
        coverage['ball_coordinates_source'] = (
            'same-frame observed ball pixel only, nearest to gesture-box center within '
            f'{BALL_ASSOCIATION_MAX_PX:g}px; projected with the confirmed image-to-court homography')
        coverage['ball_coordinates_observed_only'] = True
        coverage['ball_coordinates_axes'] = 'palla_x=court length 0..18m; palla_y=court width 0..9m'
        coverage['ball_coordinates_projection_caveat'] = (
            'homography intersects the observed image ray with the court plane; for a ball in flight '
            'this is an estimate, not a measured physical ground position')
    for episode in open_episodes:
        episodes.append({'episode_id': episode['episode_id'], 'class_raw': episode['class_raw'],
            'class_corrected': episode['class_contextual'], 'class_contextual': episode['class_contextual'],
            'score_mode': episode['score_mode'], 'raw_score_vector_mean_json': json.dumps(episode['raw_score_mean']) if episode['raw_score_mean'] is not None else '',
            'context_score_vector_json': json.dumps(episode['context_score_adjusted']) if episode['context_score_adjusted'] is not None else '',
            'context_rules_applied': '|'.join(episode['context_rules_applied']),
            'correction_reason': episode['correction_reason'], 'correction_performed': episode['correction_performed'],
            'correction_eligible': episode['correction_eligible'],
            'contact_evidence_status': episode['contact_evidence_status'], 'player_track_id': episode['player_track_id'],
            'player_team_side_geometric': episode['player_team_side'], 'first_frame': episode['first_frame'],
            'last_frame': episode['last_frame'], 'first_timestamp_seconds': episode['first_timestamp_seconds'],
            'last_timestamp_seconds': episode['last_timestamp_seconds'],
            'duration_seconds': episode['last_timestamp_seconds']-episode['first_timestamp_seconds'],
            'bbox_x1': episode['bbox'][0], 'bbox_y1': episode['bbox'][1],
            'bbox_x2': episode['bbox'][2], 'bbox_y2': episode['bbox'][3],
            'peak_confidence_raw': episode['peak_confidence'], 'detection_count': episode['detection_count'],
            'distinct_frames': len(episode['frames']), 'mean_soft_adjustment_diagnostic_only': episode['soft_adjustment_sum']/episode['detection_count'],
            'ball_evidence_types': '|'.join(sorted(episode['ball_evidence_types'])),
            'player_association_statuses': '|'.join(sorted(episode['association_statuses'])),
            'rule_evidence': '|'.join(sorted(episode['rule_evidence'])),
            'note': 'Score non calibrati; classe contestuale opzionale, contatto non confermato.'})
    event_columns = ['episode_id','class_raw','class_corrected','player_track_id','player_team_side_geometric',
        'first_frame','last_frame','first_timestamp_seconds','last_timestamp_seconds','duration_seconds',
        'bbox_x1','bbox_y1','bbox_x2','bbox_y2','peak_confidence_raw','detection_count','distinct_frames',
        'mean_soft_adjustment_diagnostic_only','ball_evidence_types','player_association_statuses','rule_evidence','note','score_mode','raw_score_vector_mean_json',
        'context_score_vector_json','context_rules_applied','correction_reason','correction_performed',
        'correction_eligible','contact_evidence_status']
    pd.DataFrame(episodes, columns=event_columns).to_csv(EVENTS_CSV, index=False, encoding='utf-8-sig')

    out_frames = out_fps = None
    if output_video_path is not None:
        out_frames, out_fps, video_warning = validate_context_video(
            output_video_path, processed_frames, VIDEO_FPS / FRAME_STRIDE)
        if video_warning:
            report_optional_video_warning(video_warning, coverage)
            if output_video_path.exists():
                output_video_path.unlink()
            output_video_path = None
    coverage.update({'processed_frames': processed_frames, 'output_fps': out_fps,
        'output_playback_duration_seconds': out_frames/out_fps if out_fps is not None else None,
        'source_interval_duration_seconds': END_SECONDS-START_SECONDS,
        'first_sampled_frame': sampled_frame_indices[0], 'last_sampled_frame': sampled_frame_indices[-1],
        'first_sampled_timestamp_seconds': sampled_frame_indices[0]/VIDEO_FPS,
        'last_sampled_timestamp_seconds': sampled_frame_indices[-1]/VIDEO_FPS,
        'gesture_detection_rows': len(detection_rows), 'gesture_episodes': len(episodes),
        'contextual_corrections': sum(ep['correction_performed'] for ep in contextual_episodes),
        'raw_head_score_rows': sum(row['score_source_mode']=='raw_head_six_class' for row in detection_rows),
        'top_only_fallback_rows': sum(row['score_source_mode']=='top_only' for row in detection_rows),
        'unassociated_gesture_rows': sum(row['player_association'] == 'non_associata' for row in detection_rows),
        'ambiguous_player_associations': sum(row['player_association'] == 'ambigua' for row in detection_rows),
        'kalman_ball_context_rows': sum(row['ball_position_type'] == 'predetta' for row in detection_rows),
        'observed_ball_context_rows': sum(row['ball_position_type'] == 'osservata' for row in detection_rows),
        'detection_rows_without_ball_context': sum(row['ball_position_type'] == 'assente_o_lontana' for row in detection_rows),
        'video_output_path': str(output_video_path) if output_video_path is not None else None,
        'video_output_warning': video_warning, 'detections_csv': str(DETECTIONS_CSV), 'episodes_csv': str(EVENTS_CSV)})
    config = {'created_utc': datetime.now(timezone.utc).isoformat(), 'video': str(VIDEO_PATH),
        'checkpoint': str(GESTURE_CHECKPOINT), 'checkpoint_sha256': CHECKPOINT_SHA256,
        'class_names': CLASS_NAMES, 'ball_csv_paths': [str(p) for p in BALL_CSV_PATHS], 'ball_config_paths': [str(p) for p in BALL_CONFIG_PATHS], 'ball_segments_audit': BALL_SEGMENT_AUDIT, 'ball_segment_boundary_audit': BALL_SEGMENT_BOUNDARY_AUDIT, 'ball_exact_duplicate_rows_dropped': BALL_EXACT_DUPLICATE_ROWS_DROPPED,
        'player_csv': str(PLAYER_CSV), 'player_config': str(PLAYER_CONFIG),
        'start_seconds': START_SECONDS, 'end_seconds_exclusive': END_SECONDS, 'source_fps': VIDEO_FPS,
        'frame_stride': FRAME_STRIDE, 'output_fps': VIDEO_FPS/FRAME_STRIDE, 'image_size': IMAGE_SIZE,
        'confidence_threshold': DETECTION_CONFIDENCE, 'nms_iou': NMS_IOU, 'device': str(DEVICE),
        'ultralytics': ultralytics.__version__, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
        'association': {'player_iou_min':PLAYER_IOU_MIN,'player_center_max_heights':PLAYER_CENTER_MAX_HEIGHTS,
                        'ambiguity_ratio':PLAYER_AMBIGUITY_RATIO,'ball_max_distance_px':BALL_ASSOCIATION_MAX_PX},
        'contextual_class_correction_enabled': ENABLE_CONTEXTUAL_CLASS_CORRECTION,
        'score_vector_mode': 'raw_head_six_class' if RAW_SCORE_VECTOR_SUPPORTED else 'top_only',
        'score_vector_fallback_reason': RAW_SCORE_FALLBACK_REASON,
        'class_score_floor': CONTEXT_CANDIDATE_SCORE_FLOOR,
        'max_raw_score_gap_for_correction': CONTEXT_MAX_RAW_SCORE_GAP,
        'minimum_episode_detections_for_correction': MIN_EPISODE_DETECTIONS_FOR_CORRECTION,
        'heuristics': {'episode_gap_seconds':EPISODE_GAP_SECONDS,'serve_response_window_seconds':SERVE_RESPONSE_WINDOW_SECONDS,'attack_response_window_seconds':ATTACK_RESPONSE_WINDOW_SECONDS,'receive_dig_to_set_factor':RECEIVE_DIG_TO_SET_FACTOR,'attack_to_defense_factor':ATTACK_TO_DEFENSE_FACTOR,'attack_to_block_factor':ATTACK_TO_BLOCK_FACTOR,
                       'same_side_attack_window_seconds':SAME_SIDE_ATTACK_WINDOW_SECONDS,
                       'simultaneous_window_seconds':SIMULTANEOUS_WINDOW_SECONDS,'soft_rule_weight':SOFT_RULE_WEIGHT,
                       'class_correction':'optional prior reweighting of real raw-head six-class scores; never alters raw'}, 'coverage': coverage,
        'limits': ['lato geometrico non equivale a squadra/rotazione', 'prossimita non prova contatto',
                   'palla predetta Kalman non e osservazione', 'eventuali sequenze mancanti non provano ace o errore',
                   'offline, nessuna promessa real-time; episodi usano storia temporale']}
    CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    REPORT_PATH.write_text(json.dumps(coverage, ensure_ascii=False, indent=2), encoding='utf-8')
    INFERENCE_COMPLETED = True
    print('Run:', RUN_DIR)
    print('Video annotato:', output_video_path)
    print('Detection frame-per-frame:', DETECTIONS_CSV)
    print('Episodi deduplicati:', EVENTS_CSV)
    print('Report copertura/limiti:', REPORT_PATH)

Frame 30/200100 | t=680.967s | detections=0
Frame 60/200100 | t=681.967s | detections=0
Frame 90/200100 | t=682.967s | detections=0
Frame 120/200100 | t=683.967s | detections=0
Frame 150/200100 | t=684.967s | detections=0
Frame 180/200100 | t=685.967s | detections=0
Frame 210/200100 | t=686.967s | detections=0
Frame 240/200100 | t=687.967s | detections=0
Frame 270/200100 | t=688.967s | detections=0
Frame 300/200100 | t=689.967s | detections=0
Frame 330/200100 | t=690.967s | detections=0
Frame 360/200100 | t=691.967s | detections=1
Frame 390/200100 | t=692.967s | detections=0
Frame 420/200100 | t=693.967s | detections=0
Frame 450/200100 | t=694.967s | detections=1
Frame 480/200100 | t=695.967s | detections=0
Frame 510/200100 | t=696.967s | detections=0
Frame 540/200100 | t=697.967s | detections=0
Frame 570/200100 | t=698.967s | detections=1
Frame 600/200100 | t=699.967s | detections=1
Frame 630/200100 | t=700.967s | detections=1
Frame 660/200100 | t=701.967s | detections=0
Frame 690/200

KeyboardInterrupt: 

## 6. Valutazione opzionale su tocchi annotati


Gli episodi automatici vengono deduplicati per `episode_id`; il timestamp rappresentativo e il frame della detection con confidenza raw piu alta. Il matching temporale uno-a-uno conserva l'ordine temporale, usa le tolleranze per classe scout e massimizza il numero di coppie; a parita di coppie privilegia i muri scout abbinati a muri contestuali (o raw se la classe contestuale manca), poi minimizza lo scarto temporale. Usa quindi la classe scout e puo favorire artificialmente l'accuratezza dei muri: il report include anche il confronto senza priorita di classe. Le stesse coppie sono usate per raw e contestuale. Tocchi allo stesso frame restano distinti.

Sono esportate tre viste: (1) precision/recall/F1 di **eventi candidati di contatto** abbinati temporalmente, indipendentemente dalla classe; (2) accuratezza, matrice di confusione e precision/recall/F1 per classe soltanto sulle coppie temporalmente abbinate, con n e copertura dei tocchi annotati; (3) metrica congiunta tempo+classe, in cui una coppia con label sbagliata conta come FP e FN e gli eventi/tocchi non abbinati sono penalizzati. La classificazione include anche episodi senza correzione contestuale; il report espone quota corretta, invariata e modificata.

Un gesto muro non prova un tocco. I muri restano candidati al matching indipendentemente dall'evidenza palla; il campo ball_warning distingue palla osservata lontana, solo Kalman, assente e assenza di evidenza. La prossimita o la distanza non cambia classe o sopprime la predizione. Il report conserva il conteggio degli episodi che la logica legacy avrebbe escluso e quantifica quanti ora vengono abbinati correttamente o restano errori. Se mancano annotazioni valutabili nella clip, le metriche restano nulle con stato esplicito, non diventano accuratezza perfetta. Sono metriche della sola clip; possibili sovrapposizioni tra match e dataset esterno di training non sono verificate.


La valutazione, quando eseguita, salva `touch_evaluation_report.json`, `metrics.csv`, `per_class_metrics.csv`, matrici raw/contestuale, candidati e matching in una directory univoca `*_touch_eval_*`; non modifica annotazioni o output del run.

### Report aggiuntivo per scambio
Oltre alle metriche sull'intera clip, il notebook raggruppa i tocchi scout per video/partita, set e `scambio`, e valuta i candidati nelle finestre che vanno dal primo all'ultimo tocco con margine configurabile (`RALLY_WINDOW_MARGIN_SECONDS`, default 0,5 s per lato). Scout e finestre sono usati solo dopo l'inferenza, per la valutazione: non filtrano frame o predizioni. Il matching temporale e uno-a-uno e usa la stessa tolleranza e funzione del report full-clip. Le finestre sono semiaperte e limitate alla clip; se si sovrappongono, ciascun candidato e assegnato una sola volta alla finestra il cui intervallo non esteso e piu vicino (spareggio deterministico per chiave video/set/scambio). Eventi fuori da tutte le finestre sono esclusi solo da questo secondo report e restano nel report full-clip.
I gruppi/finestra troncati ai bordi sono marcati come contesto parziale: il bordo della clip non e un vero inizio o fine scambio. Sono esportati report e finestre per scambio, matching, matrici raw/contestuale e metriche aggregate.

Il matching full-clip e quello per scambio usano lo stesso matching uno-a-uno e la stessa coppia per raw e contestuale. La tolleranza dipende dalla classe ground truth: battuta `SERVE_MATCH_TOLERANCE_SECONDS`, altri gesti `MATCH_TOLERANCE_SECONDS`; in casi di finestre sovrapposte un candidato resta assegnato al massimo a un tocco.

In [ ]:
if not INFERENCE_COMPLETED:
    print("Valutazione non eseguita: completa prima l'inferenza, oppure configura esplicitamente gli output di un run esistente.")
else:
    import numpy as np
    ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET = GROUND_TRUTH_TEAM_SIDE_BY_SET

    EVALUATION_DIR = OUTPUT_ROOT / f"{VIDEO_PATH.stem}_touch_eval_{RUN_ID}"
    if not math.isfinite(MATCH_TOLERANCE_SECONDS) or MATCH_TOLERANCE_SECONDS < 0:
        raise ValueError("MATCH_TOLERANCE_SECONDS deve essere finita e >= 0.")
    if Path(TOUCH_CSV).resolve() in (Path(DETECTIONS_CSV).resolve(), Path(EVENTS_CSV).resolve()):
        raise ValueError("Il CSV ground truth non puo coincidere con un output di inferenza.")

    def _label(value):
        if value is None or pd.isna(value):
            return None
        text = str(value).strip().lower()
        return text if text in CLASS_NAMES else None

    def _finite(value):
        try:
            number = float(value)
            return number if math.isfinite(number) else None
        except (TypeError, ValueError):
            return None

    def _ratio(numerator, denominator):
        return float(numerator / denominator) if denominator else None

    def _f1(precision, recall):
        if precision is None or recall is None or precision + recall == 0:
            return None
        return float(2 * precision * recall / (precision + recall))

    def _flag(value):
        if pd.isna(value):
            return None
        text = str(value).strip().lower()
        if text in ("0", "false", "no", ""):
            return False
        if text in ("1", "true", "yes"):
            return True
        return None

    def _wall_contact_candidate(raw_class, context_class, contact_status):
        is_wall = raw_class == "muro" or context_class == "muro"
        if not is_wall:
            return True, "evento_gesto_non_muro_candidato_contatto"
        return True, "muro_predetto_candidato_match_indipendente_dalla_palla"

    def _legacy_wall_contact_candidate(raw_class, context_class, contact_status):
        is_wall = raw_class == "muro" or context_class == "muro"
        if not is_wall:
            return True, "evento_gesto_non_muro_candidato_contatto"
        status = str(contact_status or "")
        if status.startswith("palla_osservata_vicina"):
            return True, "muro_palla_osservata_vicina_contatto_non_confermato"
        if status.startswith("solo_palla_kalman_vicina"):
            return False, "muro_solo_kalman_escluso_contatto_non_confermato"
        if status.startswith("nessuna_evidenza_palla_vicina"):
            return False, "muro_senza_evidenza_palla_vicina_escluso"
        return False, "muro_stato_contatto_mancante_escluso"

    def _ball_uncertainty_warning(ball_position_type, ball_distance_px,
                                   ball_confidence=None, ball_evidence_types=None):
        position = str(ball_position_type or "").strip().lower()
        distance = _finite(ball_distance_px)
        evidence = str(ball_evidence_types or "").strip().lower()
        if position == "osservata":
            if distance is not None and distance > CONTACT_PROXIMITY_DIAGNOSTIC_PX:
                return "palla_osservata_potenzialmente_lontana", distance
            if distance is not None:
                return "palla_osservata_vicina", distance
            return "palla_osservata_distanza_sconosciuta", None
        if position == "predetta" or "predetta" in evidence:
            return "solo_palla_kalman_incerta", distance
        if position in ("assente_o_lontana", ""):
            if position == "assente_o_lontana":
                return "palla_assente_o_lontana_senza_evidenza_vicina", distance
            return "nessuna_evidenza_palla", None

    def _one_to_one_time_match(ground_truth, predictions, tolerance, prefer_wall_class=True):
        def tolerance_for(gt):
            value = (tolerance.get(gt.get("label"), MATCH_TOLERANCE_SECONDS)
                     if isinstance(tolerance, dict) else tolerance)
            value = float(value)
            if not math.isfinite(value) or value < 0:
                raise ValueError(f"Tolleranza temporale non valida per {gt.get('label')}: {value}")
            return value

        gt_order = sorted(range(len(ground_truth)),
                          key=lambda i: (ground_truth[i]["time"], ground_truth[i]["id"]))
        pred_order = sorted(range(len(predictions)),
                            key=lambda i: (predictions[i]["time"], predictions[i]["id"]))
        n, m = len(gt_order), len(pred_order)
        count = np.zeros((n + 1, m + 1), dtype=np.int32)
        walls = np.zeros((n + 1, m + 1), dtype=np.int32)
        cost = np.zeros((n + 1, m + 1), dtype=np.float64)
        action = np.zeros((n + 1, m + 1), dtype=np.uint8)
        action[1:, 0] = 1
        action[0, 1:] = 2
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                options = [
                    (int(count[i - 1, j]), int(walls[i - 1, j]), float(cost[i - 1, j]), 1),
                    (int(count[i, j - 1]), int(walls[i, j - 1]), float(cost[i, j - 1]), 2),
                ]
                gi, pj = gt_order[i - 1], pred_order[j - 1]
                delta = abs(ground_truth[gi]["time"] - predictions[pj]["time"])
                if delta <= tolerance_for(ground_truth[gi]):
                    pred_class = predictions[pj].get("class_contextual") or predictions[pj].get("class_raw")
                    wall_match = int(prefer_wall_class and ground_truth[gi].get("label") == "muro"
                                     and pred_class == "muro")
                    options.append((int(count[i - 1, j - 1]) + 1,
                                    int(walls[i - 1, j - 1]) + wall_match,
                                    float(cost[i - 1, j - 1]) + delta, 3))
                best = max(options, key=lambda x: (x[0], x[1], -x[2], x[3]))
                count[i, j], walls[i, j], cost[i, j], action[i, j] = best
        pairs = []
        i, j = n, m
        while i or j:
            step = int(action[i, j])
            if step == 3:
                pairs.append((gt_order[i - 1], pred_order[j - 1]))
                i, j = i - 1, j - 1
            elif step == 1:
                i -= 1
            elif step == 2:
                j -= 1
            else:
                raise RuntimeError("Percorso di matching temporale non valido.")
        return list(reversed(pairs))


    # Il test usa tolleranze proprie, indipendenti dalla configurazione dell'utente.
    _test_class_tolerances = {
        "battuta": 2.0, "ricezione": 0.5, "alzata": 0.5,
        "attacco": 0.5, "muro": 0.5, "difesa": 0.5,
    }
    _class_tolerance_gt = [
        {"time": 0.0, "id": "serve-gt", "label": "battuta"},
        {"time": 10.0, "id": "attack-gt", "label": "attacco"},
    ]
    _class_tolerance_predictions = [
        {"time": 2.0, "id": "serve-pred"},
        {"time": 10.5, "id": "attack-pred"},
    ]
    assert _one_to_one_time_match(
        _class_tolerance_gt, _class_tolerance_predictions,
        _test_class_tolerances) == [(0, 0), (1, 1)]
    _outside_class_tolerance_predictions = [
        {"time": 2.00001, "id": "serve-late"},
        {"time": 10.50001, "id": "attack-late"},
    ]
    assert _one_to_one_time_match(
        _class_tolerance_gt, _outside_class_tolerance_predictions,
        _test_class_tolerances) == []
    _contested_serve_gt = [
        {"time": 0.0, "id": "serve-1", "label": "battuta"},
        {"time": 1.0, "id": "serve-2", "label": "battuta"},
    ]
    _contested_serve_pred = [{"time": 0.9, "id": "one-candidate"}]
    assert len(_one_to_one_time_match(
        _contested_serve_gt, _contested_serve_pred,
        _test_class_tolerances)) == 1

    def _class_report(pairs, ground_truth, predictions, label_field):
        predicted_names = CLASS_NAMES + ["non_classificata"]
        matrix = pd.DataFrame(0, index=CLASS_NAMES, columns=predicted_names, dtype=int)
        correct = 0
        unchanged = []
        for gi, pi in pairs:
            actual = ground_truth[gi]["label"]
            predicted = predictions[pi].get(label_field)
            if predicted not in CLASS_NAMES:
                predicted = "non_classificata"
            matrix.loc[actual, predicted] += 1
            correct += int(actual == predicted)
            if predictions[pi].get("class_raw") == predictions[pi].get("class_contextual"):
                unchanged.append(actual == predicted)
        class_rows = []
        for name in CLASS_NAMES:
            tp = int(matrix.loc[name, name])
            support = int(matrix.loc[name].sum())
            predicted_count = int(matrix[name].sum())
            precision, recall = _ratio(tp, predicted_count), _ratio(tp, support)
            class_rows.append({
                "class": name, "n_ground_truth_matched": support,
                "n_predictions_matched": predicted_count, "tp": tp,
                "fp": predicted_count - tp, "fn": support - tp,
                "precision": precision, "recall": recall, "f1": _f1(precision, recall),
            })
        return {
            "n_temporally_matched": len(pairs),
            "accuracy_on_temporal_matches": _ratio(correct, len(pairs)),
            "n_class_correct": correct,
            "unchanged_context_fraction_on_matches": _ratio(len(unchanged), len(pairs)),
            "accuracy_when_context_unchanged": _ratio(sum(unchanged), len(unchanged)),
        }, matrix, pd.DataFrame(class_rows)

    def _rally_key(row):
        fields = (row.get("video"), row.get("set"), row.get("rally"))
        if any(value is None or pd.isna(value) or not str(value).strip()
               for value in fields):
            return None
        return tuple(str(value).strip() for value in fields)

    def _build_rally_windows(video_rows, ground_truth, fps, clip_start, clip_end, margin):
        if not math.isfinite(float(margin)) or margin < 0:
            raise ValueError("RALLY_WINDOW_MARGIN_SECONDS deve essere finito e >= 0.")
        if not (math.isfinite(float(clip_start)) and math.isfinite(float(clip_end))
                and 0 <= clip_start < clip_end):
            raise ValueError("Intervallo clip non valido per le finestre scambio.")
        eligible_rows = {}
        for row in video_rows.to_dict("records"):
            key = _rally_key({
                "video": row.get("video"), "set": row.get("set"),
                "rally": row.get("scambio"),
            })
            if (key is None or not bool(row.get("_aligned"))
                    or pd.isna(row.get("_doubt")) or bool(row.get("_doubt"))
                    or _label(row.get("gesto")) not in CLASS_NAMES):
                continue
            frame = _finite(row.get("_frame"))
            if frame is None:
                continue
            row_id = row.get("_source_row")
            if row_id is None or pd.isna(row_id):
                continue
            eligible_rows.setdefault(key, []).append({
                "id": int(row_id), "time": float(frame) / float(fps),
            })

        ground_truth_by_key = {}
        for gt in ground_truth:
            key = _rally_key(gt)
            if key is not None:
                ground_truth_by_key.setdefault(key, []).append(gt)

        windows = []
        keys_with_window = set()
        for key in sorted(eligible_rows):
            touches = eligible_rows[key]
            first = min(row["time"] for row in touches)
            last = max(row["time"] for row in touches)
            untrimmed_start, untrimmed_end = first - margin, last + margin
            if untrimmed_end <= clip_start or untrimmed_start >= clip_end:
                continue
            start, end = max(clip_start, untrimmed_start), min(clip_end, untrimmed_end)
            if not start < end:
                continue
            touch_times = [row["time"] for row in touches]
            before_count = sum(time < clip_start for time in touch_times)
            after_count = sum(time >= clip_end for time in touch_times)
            clipped_before, clipped_after = untrimmed_start < clip_start, untrimmed_end > clip_end
            keys_with_window.add(key)
            windows.append({
                "key": key, "video": key[0], "set": key[1], "rally": key[2],
                "first_touch_seconds_full_rally": first,
                "last_touch_seconds_full_rally": last,
                "window_start_seconds_unclipped": untrimmed_start,
                "window_end_seconds_unclipped": untrimmed_end,
                "window_start_seconds": start, "window_end_seconds_exclusive": end,
                "n_eligible_touches_full_rally": len(touches),
                "n_eligible_touches_before_clip": before_count,
                "n_eligible_touches_after_clip": after_count,
                "n_eligible_touches_in_clip": len(ground_truth_by_key.get(key, [])),
                "window_clipped_at_clip_start": clipped_before,
                "window_clipped_at_clip_end": clipped_after,
                "incomplete_at_clip_border": bool(
                    before_count or after_count or clipped_before or clipped_after),
                "clip_border_is_true_rally_boundary": False,
                "n_candidate_window_hits_before_overlap_resolution": 0,
                "n_candidates_assigned": 0,
                "n_candidate_assignments_resolved_by_overlap": 0,
            })

        gt_without_key = sum(_rally_key(gt) is None for gt in ground_truth)
        gt_without_window = sum(
            _rally_key(gt) is not None and _rally_key(gt) not in keys_with_window
            for gt in ground_truth
        )
        return windows, ground_truth_by_key, {
            "n_ground_truth_without_match_set_rally_key": int(gt_without_key),
            "n_ground_truth_without_intersecting_rally_window": int(gt_without_window),
        }

    def _assign_candidates_to_rally_windows(windows, candidates):
        by_key = {window["key"]: [] for window in windows}
        outside = []
        assignment_rows = []
        window_by_key = {window["key"]: window for window in windows}
        for index, candidate in enumerate(candidates):
            time = _finite(candidate.get("time"))
            if time is None:
                outside.append(index)
                continue
            hits = [
                window for window in windows
                if window["window_start_seconds"] <= time
                < window["window_end_seconds_exclusive"]
            ]
            for window in hits:
                window["n_candidate_window_hits_before_overlap_resolution"] += 1
            if not hits:
                outside.append(index)
                continue

            def assignment_order(window):
                first = window["first_touch_seconds_full_rally"]
                last = window["last_touch_seconds_full_rally"]
                core_distance = max(first - time, 0.0, time - last)
                return core_distance, window["key"]

            chosen = min(hits, key=assignment_order)
            by_key[chosen["key"]].append(index)
            chosen["n_candidates_assigned"] += 1
            if len(hits) > 1:
                chosen["n_candidate_assignments_resolved_by_overlap"] += 1
            assignment_rows.append({
                "candidate_index": index,
                "episode_id": candidate.get("episode_id", candidate.get("id")),
                "video": chosen["video"], "set": chosen["set"], "rally": chosen["rally"],
                "candidate_time_seconds": time,
                "n_windows_containing_candidate": len(hits),
                "assigned_by_overlap_tiebreak": len(hits) > 1,
            })

        assigned_indices = [index for values in by_key.values() for index in values]
        if len(assigned_indices) != len(set(assigned_indices)):
            raise RuntimeError("Un candidato e stato assegnato a piu finestre scambio.")
        return by_key, outside, assignment_rows

    def _evaluate_rally_windows(video_rows, ground_truth, candidates, fps,
                                clip_start, clip_end, margin, tolerance):
        windows, ground_truth_by_key, exclusions = _build_rally_windows(
            video_rows, ground_truth, fps, clip_start, clip_end, margin)
        candidates_by_key, outside_indices, assignment_rows = (
            _assign_candidates_to_rally_windows(windows, candidates))
        flat_gt, flat_pred, flat_pairs = [], [], []
        metric_rows, match_rows = [], []
        for window in windows:
            key = window["key"]
            gt_group = sorted(
                ground_truth_by_key.get(key, []),
                key=lambda row: (row["time"], row["id"]))
            pred_indices = candidates_by_key[key]
            pred_group = [candidates[index] for index in pred_indices]
            local_pairs = _one_to_one_time_match(gt_group, pred_group, tolerance)
            gt_offset, pred_offset = len(flat_gt), len(flat_pred)
            flat_gt.extend(gt_group)
            flat_pred.extend(pred_group)
            flat_pairs.extend(
                (gt_offset + gi, pred_offset + pi) for gi, pi in local_pairs)
            raw_report, _, _ = _class_report(
                local_pairs, gt_group, pred_group, "class_raw")
            contextual_report, _, _ = _class_report(
                local_pairs, gt_group, pred_group, "class_contextual")
            matched_gt = {gi for gi, _ in local_pairs}
            matched_pred = {pi for _, pi in local_pairs}
            n_gt, n_pred, n_match = len(gt_group), len(pred_group), len(local_pairs)
            for gi, pi in local_pairs:
                gt, pred = gt_group[gi], pred_group[pi]
                match_rows.append({
                    "status": "temporal_match", "video": window["video"],
                    "set": window["set"], "rally": window["rally"],
                    "annotation_id": gt["id"], "annotation_time_seconds": gt["time"],
                    "gesture_truth": gt["label"], "episode_id": pred.get("episode_id"),
                    "candidate_time_seconds": pred["time"],
                    "time_delta_seconds": pred["time"] - gt["time"],
                    "class_raw": pred.get("class_raw"),
                    "class_contextual": pred.get("class_contextual"),
                    "raw_class_correct": pred.get("class_raw") == gt["label"],
                    "context_class_correct": pred.get("class_contextual") == gt["label"],
                    "ball_warning": pred.get("ball_warning"),
                })
            for gi, gt in enumerate(gt_group):
                if gi not in matched_gt:
                    match_rows.append({
                        "status": "missed_annotated_touch", "video": window["video"],
                        "set": window["set"], "rally": window["rally"],
                        "annotation_id": gt["id"], "annotation_time_seconds": gt["time"],
                        "gesture_truth": gt["label"],
                    })
            for pi, pred in enumerate(pred_group):
                if pi not in matched_pred:
                    match_rows.append({
                        "status": "unmatched_contact_candidate", "video": window["video"],
                        "set": window["set"], "rally": window["rally"],
                        "episode_id": pred.get("episode_id"),
                        "candidate_time_seconds": pred["time"],
                        "class_raw": pred.get("class_raw"),
                        "class_contextual": pred.get("class_contextual"),
                    })

            raw_correct = raw_report["n_class_correct"]
            context_correct = contextual_report["n_class_correct"]
            metric_rows.append({
                **{k: v for k, v in window.items() if k != "key"},
                "n_ground_truth_touches": n_gt, "n_contact_candidates": n_pred,
                "n_temporal_matches": n_match,
                "n_unmatched_candidates": n_pred - n_match,
                "n_missed_touches": n_gt - n_match,
                "raw_class_correct_on_matches": raw_correct,
                "raw_accuracy_on_matches": raw_report["accuracy_on_temporal_matches"],
                "contextual_class_correct_on_matches": context_correct,
                "contextual_accuracy_on_matches":
                    contextual_report["accuracy_on_temporal_matches"],
                "contact_precision": _ratio(n_match, n_pred),
                "contact_recall": _ratio(n_match, n_gt),
                "contact_f1": _f1(_ratio(n_match, n_pred), _ratio(n_match, n_gt)),
                "joint_raw_tp": raw_correct,
                "joint_raw_fp": n_pred - raw_correct,
                "joint_raw_fn": n_gt - raw_correct,
                "joint_raw_f1": _f1(
                    _ratio(raw_correct, n_pred), _ratio(raw_correct, n_gt)),
                "joint_contextual_tp": context_correct,
                "joint_contextual_fp": n_pred - context_correct,
                "joint_contextual_fn": n_gt - context_correct,
                "joint_contextual_f1": _f1(
                    _ratio(context_correct, n_pred), _ratio(context_correct, n_gt)),
            })

        raw_summary, raw_matrix, raw_classes = _class_report(
            flat_pairs, flat_gt, flat_pred, "class_raw")
        context_summary, context_matrix, context_classes = _class_report(
            flat_pairs, flat_gt, flat_pred, "class_contextual")
        n_gt, n_pred, n_match = len(flat_gt), len(flat_pred), len(flat_pairs)
        raw_correct = raw_summary["n_class_correct"]
        context_correct = context_summary["n_class_correct"]
        return {
            "windows": windows,
            "metrics_by_rally": metric_rows,
            "match_rows": match_rows,
            "candidate_assignments": assignment_rows,
            "raw_confusion": raw_matrix,
            "contextual_confusion": context_matrix,
            "raw_per_class": raw_classes,
            "contextual_per_class": context_classes,
            "summary": {
                "status": "evaluated" if n_gt else "no_evaluable_rally_touches",
                "ground_truth_role": "evaluation only; not used for inference or prediction filtering",
                "window_definition": "first/last aligned non-doubtful known-label touch +/- margin; half-open and clipped to clip interval",
                "clip_interval_seconds_half_open": [float(clip_start), float(clip_end)],
                "rally_window_margin_seconds_each_side": float(margin),
                "match_tolerance_seconds": MATCH_TOLERANCE_SECONDS,
                "match_tolerance_seconds_by_ground_truth_class": {key: float(value) for key, value in tolerance.items()},
                "n_rallies_with_intersecting_windows": len(windows),
                "n_incomplete_at_clip_border": sum(
                    bool(window["incomplete_at_clip_border"]) for window in windows),
                "clip_border_is_true_rally_boundary": False,
                "n_ground_truth_touches_in_rally_report": n_gt,
                "n_contact_candidates_in_windows_unique": n_pred,
                "n_contact_candidates_outside_rally_windows_excluded_from_this_report":
                    len(outside_indices),
                "n_candidate_window_hits_resolved_by_overlap": sum(
                    row["assigned_by_overlap_tiebreak"] for row in assignment_rows),
                "n_candidate_assignments_unique": len(assignment_rows),
                "candidate_assignments_have_duplicates": False,
                "n_temporal_matches": n_match,
                "n_unmatched_candidates": n_pred - n_match,
                "n_missed_annotated_touches": n_gt - n_match,
                "contact_candidate_detection": {
                    "tp": n_match, "fp": n_pred - n_match, "fn": n_gt - n_match,
                    "precision": _ratio(n_match, n_pred),
                    "recall": _ratio(n_match, n_gt),
                    "f1": _f1(_ratio(n_match, n_pred), _ratio(n_match, n_gt)),
                },
                "classification_raw_on_temporal_matches": raw_summary,
                "classification_contextual_on_temporal_matches": context_summary,
                "joint_time_and_class_raw": {
                    "tp": raw_correct, "fp": n_pred - raw_correct,
                    "fn": n_gt - raw_correct,
                    "precision": _ratio(raw_correct, n_pred),
                    "recall": _ratio(raw_correct, n_gt),
                    "f1": _f1(_ratio(raw_correct, n_pred), _ratio(raw_correct, n_gt)),
                },
                "joint_time_and_class_contextual": {
                    "tp": context_correct, "fp": n_pred - context_correct,
                    "fn": n_gt - context_correct,
                    "precision": _ratio(context_correct, n_pred),
                    "recall": _ratio(context_correct, n_gt),
                    "f1": _f1(
                        _ratio(context_correct, n_pred), _ratio(context_correct, n_gt)),
                },
                **exclusions,
                "limitations": [
                    "Rally IDs and touch times come from scout for evaluation only.",
                    "The first/last annotated touch plus margin is an evaluation window, not a proven real rally boundary.",
                    "Windows clipped at clip edges have incomplete context; clip edges are not rally starts or ends.",
                    "Predictions outside the union of rally windows are excluded only here; full-clip metrics remain unchanged.",
                    "A candidate in overlapping windows is assigned once to the nearest unpadded rally touch interval; ties use lexical video/set/rally key.",
                ],
            },
        }

    def _synthetic_rally_window_tests():
        rows = pd.DataFrame([
            {"video": "m.mp4", "set": 1, "scambio": 1, "_aligned": True,
             "_doubt": False, "gesto": "battuta", "_frame": 30, "_source_row": 2},
            {"video": "m.mp4", "set": 1, "scambio": 1, "_aligned": True,
             "_doubt": False, "gesto": "attacco", "_frame": 60, "_source_row": 3},
            {"video": "m.mp4", "set": 1, "scambio": 2, "_aligned": True,
             "_doubt": False, "gesto": "ricezione", "_frame": 78, "_source_row": 4},
            {"video": "m.mp4", "set": 1, "scambio": 2, "_aligned": True,
             "_doubt": False, "gesto": "difesa", "_frame": 90, "_source_row": 5},
            {"video": "m.mp4", "set": 1, "scambio": 0, "_aligned": True,
             "_doubt": False, "gesto": "battuta", "_frame": 0, "_source_row": 6},
        ])
        gt = [
            {"id": 2, "video": "m.mp4", "set": 1, "rally": 1,
             "time": 1.0, "label": "battuta"},
            {"id": 3, "video": "m.mp4", "set": 1, "rally": 1,
             "time": 2.0, "label": "attacco"},
            {"id": 4, "video": "m.mp4", "set": 1, "rally": 2,
             "time": 2.6, "label": "ricezione"},
            {"id": 5, "video": "m.mp4", "set": 1, "rally": 2,
             "time": 3.0, "label": "difesa"},
        ]
        windows, _, _ = _build_rally_windows(rows, gt, 30., 0., 4., 0.5)
        assigned, outside, audit = _assign_candidates_to_rally_windows(
            windows, [
                {"id": "p1", "episode_id": "p1", "time": 2.1},
                {"id": "p2", "episode_id": "p2", "time": 2.7},
                {"id": "p3", "episode_id": "p3", "time": 3.5},
            ])
        assert len(windows) == 3
        assert len(assigned[("m.mp4", "1", "1")]) == 1
        assert len(assigned[("m.mp4", "1", "2")]) == 1
        assert len(outside) == 1
        assert len(audit) == 2 and len({row["episode_id"] for row in audit}) == 2
        assert sum(window["incomplete_at_clip_border"] for window in windows) == 1
        assert all(not window["clip_border_is_true_rally_boundary"] for window in windows)

    _synthetic_rally_window_tests()

    def _synthetic_evaluation_tests():
        synthetic_gt = [
            {"id": 0, "time": 10.00, "label": "attacco"},
            {"id": 1, "time": 10.12, "label": "muro"},
        ]
        synthetic_pred = [
            {"id": "p0", "time": 10.03, "class_raw": "attacco",
             "class_contextual": "attacco"},
            {"id": "p1", "time": 10.13, "class_raw": "muro",
             "class_contextual": "muro"},
        ]
        close_pairs = _one_to_one_time_match(synthetic_gt, synthetic_pred, 0.15)
        assert close_pairs == [(0, 0), (1, 1)]
        fpfn_gt = [{"id": 0, "time": 1.0, "label": "battuta"},
                   {"id": 1, "time": 3.0, "label": "ricezione"}]
        fpfn_pred = [{"id": "p0", "time": 1.1, "class_raw": "ricezione",
                      "class_contextual": "battuta"},
                     {"id": "p1", "time": 2.0, "class_raw": "alzata",
                      "class_contextual": "alzata"}]
        fpfn_pairs = _one_to_one_time_match(fpfn_gt, fpfn_pred, 0.2)
        assert len(fpfn_pairs) == 1 and fpfn_pairs == [(0, 0)]
        raw, _, _ = _class_report(fpfn_pairs, fpfn_gt, fpfn_pred, "class_raw")
        context, _, _ = _class_report(fpfn_pairs, fpfn_gt, fpfn_pred, "class_contextual")
        assert raw["accuracy_on_temporal_matches"] == 0.0
        assert context["accuracy_on_temporal_matches"] == 1.0
        temporal_tp = len(fpfn_pairs)
        assert temporal_tp == 1
        assert len(fpfn_pred) - temporal_tp == 1
        assert len(fpfn_gt) - temporal_tp == 1
        assert (raw["n_class_correct"], len(fpfn_pred) - raw["n_class_correct"],
                len(fpfn_gt) - raw["n_class_correct"]) == (0, 2, 2)
        assert (context["n_class_correct"], len(fpfn_pred) - context["n_class_correct"],
                len(fpfn_gt) - context["n_class_correct"]) == (1, 1, 1)
        assert _one_to_one_time_match([], fpfn_pred, 0.2) == []
        empty, _, _ = _class_report([], [], fpfn_pred, "class_raw")
        assert empty["accuracy_on_temporal_matches"] is None
        assert _wall_contact_candidate("muro", "muro",
               "nessuna_evidenza_palla_vicina_muro_puo_essere_senza_contatto") == (
                   True, "muro_predetto_candidato_match_indipendente_dalla_palla")
        assert _wall_contact_candidate("muro", "muro",
               "solo_palla_kalman_vicina_contatto_non_confermato") == (
                   True, "muro_predetto_candidato_match_indipendente_dalla_palla")
        assert _wall_contact_candidate("muro", "muro",
               "palla_osservata_vicina_candidato_contatto_non_confermato") == (
                   True, "muro_predetto_candidato_match_indipendente_dalla_palla")
        assert _ball_uncertainty_warning("", None)[0] == "nessuna_evidenza_palla"
        assert _ball_uncertainty_warning("predetta", 80.0)[0] == "solo_palla_kalman_incerta"
        assert _ball_uncertainty_warning("osservata", 200.0)[0] == "palla_osservata_potenzialmente_lontana"
        wall_gt = [{"id": 0, "time": 5.0, "label": "muro"}]
        wall_and_near_pred = [
            {"id": "attack", "time": 4.93, "class_raw": "attacco", "class_contextual": "attacco"},
            {"id": "defense", "time": 5.01, "class_raw": "difesa", "class_contextual": "difesa"},
            {"id": "wall", "time": 5.04, "class_raw": "muro", "class_contextual": "muro"}]
        assert _one_to_one_time_match(wall_gt, wall_and_near_pred, 0.5) == [(0, 2)]
        assert _one_to_one_time_match(wall_gt, wall_and_near_pred, 0.5, prefer_wall_class=False) == [(0, 1)]
        assert _one_to_one_time_match(wall_gt, wall_and_near_pred[:2], 0.5) == [(0, 1)]
        assert _one_to_one_time_match(wall_gt, wall_and_near_pred, 0.02) == [(0, 1)]
        contested_gt = [{"id": 0, "time": 5.0, "label": "muro"},
                        {"id": 1, "time": 5.1, "label": "attacco"}]
        contested_pred = [{"id": "only", "time": 5.05, "class_contextual": "muro"}]
        assert _one_to_one_time_match(contested_gt, contested_pred, 0.5) == [(0, 0)]
        cardinality_pred = [{"id": "attack", "time": 5.0, "class_contextual": "attacco"},
                            {"id": "wall", "time": 5.1, "class_contextual": "muro"}]
        assert _one_to_one_time_match(contested_gt, cardinality_pred, 0.5) == [(0, 0), (1, 1)]
        assert _ball_uncertainty_warning("", None)[0] != _ball_uncertainty_warning("osservata", 200.0)[0]

    def _synthetic_class_tolerance_tests():
        class_tolerance = {"battuta": 2.0, "attacco": 0.5}
        serve_gt = [{"id": 0, "time": 0.0, "label": "battuta"}]
        exact_boundary = [{"id": "p0", "time": 2.0,
                           "class_raw": "battuta", "class_contextual": "battuta"}]
        assert _one_to_one_time_match(
            serve_gt, exact_boundary, class_tolerance) == [(0, 0)]
        just_outside = [{"id": "p0", "time": 2.0001,
                         "class_raw": "battuta", "class_contextual": "battuta"}]
        assert _one_to_one_time_match(serve_gt, just_outside, class_tolerance) == []

        attack_gt = [{"id": 0, "time": 0.0, "label": "attacco"}]
        attack_inside = [{"id": "p0", "time": 0.5,
                          "class_raw": "muro", "class_contextual": "attacco"}]
        attack_outside = [{"id": "p0", "time": 0.5001,
                           "class_raw": "muro", "class_contextual": "attacco"}]
        assert _one_to_one_time_match(
            attack_gt, attack_inside, class_tolerance) == [(0, 0)]
        assert _one_to_one_time_match(
            attack_gt, attack_outside, class_tolerance) == []

        overlap_gt = [
            {"id": 0, "time": 0.0, "label": "battuta"},
            {"id": 1, "time": 1.8, "label": "attacco"},
        ]
        overlap_pred = [{"id": "shared", "time": 1.7,
                         "class_raw": "battuta", "class_contextual": "muro"}]
        shared_pairs = _one_to_one_time_match(
            overlap_gt, overlap_pred, class_tolerance)
        assert len(shared_pairs) == 1
        raw, _, _ = _class_report(shared_pairs, overlap_gt, overlap_pred, "class_raw")
        contextual, _, _ = _class_report(
            shared_pairs, overlap_gt, overlap_pred, "class_contextual")
        assert raw["n_temporally_matched"] == contextual["n_temporally_matched"] == 1
        assert sum(pi == 0 for _, pi in shared_pairs) == 1

    _synthetic_evaluation_tests()
    _synthetic_class_tolerance_tests()

    def _camera_half_contact_report(side, ground_truth, contact_predictions, pairs):
        side_gt_indices = {
            gi for gi, gt in enumerate(ground_truth) if gt.get("camera_half") == side
        }
        matched_prediction_indices = {pi for _, pi in pairs}
        matched_gt_pairs = [(gi, pi) for gi, pi in pairs if gi in side_gt_indices]
        unmatched_prediction_indices = set(range(len(contact_predictions))) - matched_prediction_indices
        side_false_positives = [
            pi for pi in unmatched_prediction_indices
            if contact_predictions[pi].get("camera_half_prediction") == side
        ]
        unknown_side_false_positives = [
            pi for pi in unmatched_prediction_indices
            if contact_predictions[pi].get("camera_half_prediction") not in ("near", "far")
        ]
        true_positive = len(matched_gt_pairs)
        false_negative = len(side_gt_indices) - true_positive
        side_precision = (
            _ratio(true_positive, true_positive + len(side_false_positives))
            if not unknown_side_false_positives else None
        )
        return {
            "camera_half": side, "n_ground_truth_touches": len(side_gt_indices),
            "tp": true_positive, "fn": false_negative,
            "fp_assigned_to_side": len(side_false_positives),
            "unassigned_unmatched_predictions": len(unknown_side_false_positives),
            "precision": side_precision,
            "recall": _ratio(true_positive, len(side_gt_indices)),
            "f1": _f1(side_precision, _ratio(true_positive, len(side_gt_indices))),
            "matched_pairs": matched_gt_pairs,
            "status": (
                "not_measurable_no_ground_truth_side_mapping"
                if not side_gt_indices and not ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET
                else "not_measurable_no_ground_truth_events_for_side"
                if not side_gt_indices
                else "prediction_side_incomplete"
                if unknown_side_false_positives
                else "evaluated"
            ),
        }

    def _synthetic_camera_half_tests():
        truth = [
            {"camera_half": "near", "label": "attacco"},
            {"camera_half": "far", "label": "difesa"},
        ]
        predicted = [
            {"camera_half_prediction": "near"},
            {"camera_half_prediction": "far"},
            {"camera_half_prediction": "near"},
        ]
        pairs = [(0, 0), (1, 1)]
        near = _camera_half_contact_report("near", truth, predicted, pairs)
        far = _camera_half_contact_report("far", truth, predicted, pairs)
        assert (near["tp"], near["fp_assigned_to_side"], near["fn"]) == (1, 1, 0)
        assert near["precision"] == 0.5 and near["recall"] == 1.0
        assert far["tp"] == 1 and far["fp_assigned_to_side"] == 0
        predicted.append({"camera_half_prediction": None})
        incomplete = _camera_half_contact_report("near", truth, predicted, pairs)
        assert incomplete["precision"] is None
        assert incomplete["unassigned_unmatched_predictions"] == 1
        configured_mapping = dict(GROUND_TRUTH_TEAM_SIDE_BY_SET)
        GROUND_TRUTH_TEAM_SIDE_BY_SET.clear()
        unconfigured = _camera_half_contact_report("near", [
            {"camera_half": None, "label": "attacco"}], predicted, [])
        GROUND_TRUTH_TEAM_SIDE_BY_SET.update(configured_mapping)
        assert unconfigured["status"] == "not_measurable_no_ground_truth_side_mapping"

    _synthetic_camera_half_tests()
    if not isinstance(GROUND_TRUTH_TEAM_SIDE_BY_SET, dict):
        raise TypeError("GROUND_TRUTH_TEAM_SIDE_BY_SET deve essere un dizionario.")
    if not isinstance(PREDICTION_TRACK_SIDE_TO_CAMERA_HALF, dict):
        raise TypeError("PREDICTION_TRACK_SIDE_TO_CAMERA_HALF deve essere un dizionario.")
    for mapping_name, mapping in (
        ("GROUND_TRUTH_TEAM_SIDE_BY_SET", GROUND_TRUTH_TEAM_SIDE_BY_SET),
        ("PREDICTION_TRACK_SIDE_TO_CAMERA_HALF", PREDICTION_TRACK_SIDE_TO_CAMERA_HALF),
    ):
        for key, value in mapping.items():
            if mapping_name == "GROUND_TRUTH_TEAM_SIDE_BY_SET":
                if not isinstance(value, dict):
                    raise TypeError(f"{mapping_name}[{key!r}] deve mappare squadre in near/far.")
                values = value.values()
            else:
                values = [value]
            if any(side not in ("near", "far") for side in values):
                raise ValueError(f"{mapping_name} ammette solo valori 'near' o 'far'.")

    def _calibrated_track_side_map():
        calibration_path = Path(CAMERA_CALIBRATION_JSON)
        player_config_path = Path(SIDE_MAPPING_PLAYER_CONFIG)
        if not calibration_path.is_file() or not player_config_path.is_file():
            raise FileNotFoundError(
                f"Calibrazione o configurazione traccia lato mancante: "
                f"{calibration_path}, {player_config_path}")
        calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
        player_side_config = json.loads(player_config_path.read_text(encoding="utf-8"))
        settings = player_side_config.get("settings", {})
        if calibration.get("confirmed_by_user") is not True:
            raise ValueError("La calibrazione campo non risulta confermata dall'utente.")
        if Path(calibration.get("video", "")).resolve() != VIDEO_PATH.resolve():
            raise ValueError("La calibrazione non appartiene esattamente al video selezionato.")
        if Path(settings.get("video", "")).resolve() != VIDEO_PATH.resolve():
            raise ValueError("La traccia player usata per gli ancoraggi appartiene a un altro video.")
        if player_side_config.get("status") != "complete":
            raise ValueError("La traccia player completa non risulta terminata.")
        if player_side_config.get("calibration_sha256") != sha256_file(calibration_path):
            raise ValueError("La traccia player e la calibrazione hanno hash non corrispondenti.")
        if not math.isclose(float(player_side_config.get("fps", 0)), VIDEO_FPS, abs_tol=0.01):
            raise ValueError("FPS della traccia player completa diverso dal video.")
        side_definition = str(player_side_config.get("side_definition", ""))
        if "A: Y<9" not in side_definition or "B: Y>=9" not in side_definition:
            raise ValueError(f"Definizione geometrica dei lati inattesa: {side_definition}")

        matrix = np.asarray(calibration.get("H_image_to_world"), dtype=float)
        if matrix.shape != (3, 3) or not np.isfinite(matrix).all():
            raise ValueError("Omografia immagine→campo non valida.")
        corners = calibration.get("corners_image_px", [])
        corner_order = calibration.get("corner_order", [])
        if len(corners) != len(corner_order):
            raise ValueError("Corner e nomi dei corner della calibrazione non concordano.")

        def world_y_for(label_fragment):
            matches = [index for index, label in enumerate(corner_order)
                       if label_fragment in str(label).lower()]
            if not matches:
                raise ValueError(f"Corner di calibrazione assente: {label_fragment}")
            point = np.asarray([*corners[matches[0]], 1.0], dtype=float)
            projected = matrix @ point
            if abs(projected[2]) < 1e-9:
                raise ValueError("Proiezione omografica non valida sul corner.")
            return float(projected[1] / projected[2])

        far_y = world_y_for("fondo lontano")
        near_y = world_y_for("fondo vicino")
        if not (far_y < 9.0 < near_y):
            raise ValueError(
                f"Orientamento omografia incompatibile con A/B: far_y={far_y}, near_y={near_y}.")
        return {
            "A_y_lt9": "far",
            "B_y_ge9": "near",
        }, {
            "calibration_path": str(calibration_path.resolve()),
            "calibration_sha256": sha256_file(calibration_path),
            "confirmed_by_user": True,
            "far_baseline_world_y_m": far_y,
            "near_baseline_world_y_m": near_y,
            "track_side_definition": side_definition,
        }, player_side_config

    def _select_stable_baseline_track(candidates, serve_time, side_to_half):
        if candidates.empty:
            return {
                "status": "no_geometric_candidate",
                "candidate_track_count": 0,
            }
        qualified = []
        for track_id, group in candidates.groupby("track_id", sort=False):
            frame_count = int(group["frame_index"].nunique())
            time_start = float(group["timestamp_seconds"].min())
            time_end = float(group["timestamp_seconds"].max())
            y_min, y_max = float(group["floor_y_m"].min()), float(group["floor_y_m"].max())
            track_sides = group["team_side"].dropna().astype(str).unique()
            last_age = float(serve_time - time_end)
            if (frame_count < SIDE_ANCHOR_MIN_FRAMES
                    or time_end - time_start < SIDE_ANCHOR_MIN_DURATION_SECONDS
                    or y_max - y_min > SIDE_ANCHOR_MAX_FOOT_Y_RANGE_METERS
                    or last_age < -1e-6
                    or last_age > SIDE_ANCHOR_MAX_LAST_OBSERVATION_AGE_SECONDS
                    or len(track_sides) != 1
                    or str(track_sides[0]) not in side_to_half):
                continue
            numbers = sorted({
                str(value) for value in group["number_confirmed"].dropna().tolist()
            })
            qualified.append({
                "track_id": str(track_id),
                "team_side_geometric": str(track_sides[0]),
                "camera_half_from_calibration": side_to_half[str(track_sides[0])],
                "frames": frame_count,
                "duration_seconds": time_end - time_start,
                "last_observation_age_seconds": last_age,
                "floor_y_min_m": y_min,
                "floor_y_max_m": y_max,
                "recognized_numbers_evidence_only": "|".join(numbers),
            })
        if len(qualified) == 1:
            return {
                "status": "unique_stable_baseline_track",
                "candidate_track_count": int(candidates["track_id"].nunique()),
                **qualified[0],
            }
        return {
            "status": (
                "ambiguous_multiple_stable_tracks" if len(qualified) > 1
                else "no_stable_pre_serve_track"
            ),
            "candidate_track_count": int(candidates["track_id"].nunique()),
            "stable_track_count": len(qualified),
        }

    def _aggregate_serve_side_anchors(anchor_rows, teams_by_set, min_anchors):
        mapping, set_reports = {}, {}
        for set_id, teams in sorted(teams_by_set.items()):
            teams = sorted(set(str(team) for team in teams if str(team).strip()))
            report = {"teams": teams, "anchors_by_team": {}, "status": None, "mapping": {}}
            if len(teams) != 2:
                report["status"] = "unknown_unexpected_team_count"
                set_reports[set_id] = report
                continue
            rows_by_team = {
                team: [row for row in anchor_rows
                       if str(row.get("set")) == set_id and str(row.get("team")) == team
                       and row.get("status") == "unique_stable_baseline_track"]
                for team in teams
            }
            observed_sides = {}
            for team in teams:
                sides = {row["team_side_geometric"] for row in rows_by_team[team]}
                report["anchors_by_team"][team] = {
                    "count": len(rows_by_team[team]),
                    "side_counts": dict(Counter(
                        row["team_side_geometric"] for row in rows_by_team[team])),
                }
                if len(sides) > 1:
                    report["status"] = "unknown_conflicting_serve_anchors"
                    break
                if sides:
                    observed_sides[team] = next(iter(sides))
            if report["status"] is not None:
                set_reports[set_id] = report
                continue
            strong = {
                team: side for team, side in observed_sides.items()
                if len(rows_by_team[team]) >= min_anchors
            }
            if not strong:
                report["status"] = "unknown_insufficient_repeated_serve_anchors"
                set_reports[set_id] = report
                continue
            if len(set(strong.values())) > 1:
                report["status"] = "unknown_conflicting_team_sides"
                set_reports[set_id] = report
                continue
            if len(strong) == 2:
                first, second = teams
                if strong[first] == strong[second]:
                    report["status"] = "unknown_teams_not_on_opposite_halves"
                    set_reports[set_id] = report
                    continue
                team_geometry = dict(strong)
            else:
                known_team, known_side = next(iter(strong.items()))
                other_team = next(team for team in teams if team != known_team)
                opposite = "B_y_ge9" if known_side == "A_y_lt9" else "A_y_lt9"
                if other_team in observed_sides and observed_sides[other_team] != opposite:
                    report["status"] = "unknown_opposing_team_anchor_conflict"
                    set_reports[set_id] = report
                    continue
                team_geometry = {known_team: known_side, other_team: opposite}
            report["status"] = "derived_from_repeated_stable_serve_anchors"
            report["mapping"] = team_geometry
            mapping[set_id] = team_geometry
            set_reports[set_id] = report
        return mapping, set_reports

    def _synthetic_side_anchor_tests():
        side_to_half = {"A_y_lt9": "far", "B_y_ge9": "near"}
        stable_rows = [
            {"track_id": 7, "frame_index": frame, "timestamp_seconds": 9.80 + 0.04 * frame,
             "floor_x_m": 5.0, "floor_y_m": 18.20 + 0.01 * frame,
             "team_side": "B_y_ge9", "number_confirmed": "4"}
            for frame in range(5)
        ]
        stable = _select_stable_baseline_track(
            pd.DataFrame(stable_rows), 10.0, side_to_half)
        assert stable["status"] == "unique_stable_baseline_track"
        assert stable["camera_half_from_calibration"] == "near"
        ambiguous = _select_stable_baseline_track(
            pd.DataFrame(stable_rows + [
                {**row, "track_id": 8, "floor_x_m": 6.0} for row in stable_rows
            ]), 10.0, side_to_half)
        assert ambiguous["status"] == "ambiguous_multiple_stable_tracks"
        jumping_rows = [
            {**row, "floor_y_m": 18.10 if row["frame_index"] < 3 else 18.90}
            for row in stable_rows
        ]
        jumping = _select_stable_baseline_track(
            pd.DataFrame(jumping_rows), 10.0, side_to_half)
        assert jumping["status"] == "no_stable_pre_serve_track"

        anchors = [
            {"set": "1", "team": "ITA", "status": "unique_stable_baseline_track",
             "team_side_geometric": "A_y_lt9"},
            {"set": "1", "team": "ITA", "status": "unique_stable_baseline_track",
             "team_side_geometric": "A_y_lt9"},
        ]
        mapping, reports = _aggregate_serve_side_anchors(
            anchors, {"1": ["ITA", "POL"]}, 2)
        assert reports["1"]["status"] == "derived_from_repeated_stable_serve_anchors"
        assert mapping == {"1": {"ITA": "A_y_lt9", "POL": "B_y_ge9"}}
        _, insufficient = _aggregate_serve_side_anchors(
            anchors[:1], {"1": ["ITA", "POL"]}, 2)
        assert insufficient["1"]["status"] == "unknown_insufficient_repeated_serve_anchors"
        conflicting = anchors + [
            {"set": "1", "team": "ITA", "status": "unique_stable_baseline_track",
             "team_side_geometric": "B_y_ge9"}
        ]
        conflict_mapping, conflict_reports = _aggregate_serve_side_anchors(
            conflicting, {"1": ["ITA", "POL"]}, 2)
        assert not conflict_mapping
        assert conflict_reports["1"]["status"] == "unknown_conflicting_serve_anchors"

    _synthetic_side_anchor_tests()

    def _derive_camera_half_mapping(video_rows):
        side_to_half, calibration_report, side_player_config = _calibrated_track_side_map()
        player_csv = Path(SIDE_MAPPING_PLAYER_CSV)
        if not player_csv.is_file():
            raise FileNotFoundError(f"Traccia player completa per mapping lato mancante: {player_csv}")
        settings = side_player_config.get("settings", {})
        if Path(settings.get("video", "")).resolve() != VIDEO_PATH.resolve():
            raise ValueError("Video della traccia player di mapping non corrispondente.")
        serve_rows = video_rows[
            video_rows["gesto"].map(_label).eq("battuta")
            & video_rows["_doubt"].eq(False)
            & video_rows["_aligned"]
        ].copy()
        serve_rows = serve_rows[
            serve_rows["_frame"].notna() & serve_rows["_annotated_time"].notna()
            & serve_rows["set"].notna() & serve_rows["squadra"].notna()
        ].copy()
        serve_rows["_set_key"] = serve_rows["set"].astype(str).str.strip()
        serve_rows["_team_key"] = serve_rows["squadra"].astype(str).str.strip()
        team_rows = video_rows[
            video_rows["set"].notna() & video_rows["squadra"].notna()
        ].copy()
        team_rows["_set_key"] = team_rows["set"].astype(str).str.strip()
        team_rows["_team_key"] = team_rows["squadra"].astype(str).str.strip()
        teams_by_set = {
            str(set_id): sorted(set(group["_team_key"]))
            for set_id, group in team_rows.groupby("_set_key", sort=True)
        }
        required = {
            "timestamp_seconds", "frame_index", "track_id", "floor_x_m",
            "floor_y_m", "team_side", "number_confirmed", "in_court",
        }
        usecols = sorted(required)
        windows = sorted(
            (float(row["_annotated_time"]) - SIDE_ANCHOR_LOOKBACK_SECONDS,
             float(row["_annotated_time"]))
            for _, row in serve_rows.iterrows()
        )
        merged_windows = []
        for start, end in windows:
            if merged_windows and start <= merged_windows[-1][1]:
                merged_windows[-1][1] = max(merged_windows[-1][1], end)
            else:
                merged_windows.append([start, end])

        player_config_start = settings.get("start")
        player_config_end = settings.get("end")
        player_rows = []
        max_alignment_error = 0.0
        if merged_windows:
            with pd.read_csv(player_csv, usecols=usecols, low_memory=False,
                             chunksize=200000) as reader:
                for chunk in reader:
                    for column in ("timestamp_seconds", "frame_index", "track_id",
                                   "floor_x_m", "floor_y_m"):
                        chunk[column] = pd.to_numeric(chunk[column], errors="coerce")
                    if chunk[["timestamp_seconds", "frame_index", "track_id",
                              "floor_x_m", "floor_y_m"]].isna().any().any():
                        raise ValueError(f"Valori player non numerici/non validi in {player_csv}.")
                    alignment_error = (
                        chunk["timestamp_seconds"].to_numpy() * VIDEO_FPS
                        - chunk["frame_index"].to_numpy()
                    )
                    if alignment_error.size:
                        max_alignment_error = max(
                            max_alignment_error, float(np.max(np.abs(alignment_error))))
                    if max_alignment_error > 0.51:
                        raise ValueError("Timestamp/frame del CSV player completo non allineati.")
                    times = chunk["timestamp_seconds"].to_numpy(dtype=float)
                    starts = np.asarray([window[0] for window in merged_windows], dtype=float)
                    ends = np.asarray([window[1] for window in merged_windows], dtype=float)
                    indices = np.searchsorted(starts, times, side="right") - 1
                    in_serve_window = indices >= 0
                    valid_indices = np.flatnonzero(in_serve_window)
                    in_serve_window[valid_indices] &= (
                        times[valid_indices] <= ends[indices[valid_indices]] + 1e-9
                    )
                    inside_values = (
                        chunk["in_court"].astype(str).str.strip().str.lower()
                        .isin(("true", "1", "yes"))
                    )
                    x = chunk["floor_x_m"].to_numpy(dtype=float)
                    y = chunk["floor_y_m"].to_numpy(dtype=float)
                    near_baseline = (
                        ((y >= -0.8) & (y < 0.0))
                        | ((y > 18.0) & (y <= 18.8))
                    )
                    select = (
                        in_serve_window & ~inside_values.to_numpy()
                        & np.isfinite(x) & np.isfinite(y)
                        & (x >= -1.0) & (x <= 10.0) & near_baseline
                    )
                    if select.any():
                        player_rows.append(chunk.loc[select].copy())
        baseline_rows = (
            pd.concat(player_rows, ignore_index=True)
            if player_rows else pd.DataFrame(columns=usecols)
        )

        anchor_rows = []
        config_start = float(player_config_start) if player_config_start is not None else -math.inf
        config_end = float(player_config_end) if player_config_end is not None else math.inf
        for _, serve in serve_rows.iterrows():
            serve_time = float(serve["_annotated_time"])
            start_time = serve_time - SIDE_ANCHOR_LOOKBACK_SECONDS
            base = {
                "annotation_row": int(serve["_source_row"]),
                "set": str(serve["_set_key"]),
                "team": str(serve["_team_key"]),
                "serve_frame": int(serve["_frame"]),
                "serve_time_seconds": serve_time,
                "lookback_start_seconds": start_time,
                "lookback_end_seconds_exclusive": serve_time,
            }
            if start_time < config_start or serve_time > config_end:
                anchor_rows.append({**base, "status": "player_csv_lookback_not_covered"})
                continue
            if baseline_rows.empty:
                candidates = baseline_rows
            else:
                candidates = baseline_rows[
                    baseline_rows["timestamp_seconds"].between(
                        start_time, serve_time, inclusive="both")
                ]
            selected = _select_stable_baseline_track(
                candidates, serve_time, side_to_half)
            anchor_rows.append({**base, **selected})

        geometry_by_set, set_reports = _aggregate_serve_side_anchors(
            anchor_rows, teams_by_set, SIDE_ANCHOR_MIN_REPEATED_SERVES_PER_TEAM_SET)
        camera_by_set = {
            set_id: {team: side_to_half[side] for team, side in team_sides.items()}
            for set_id, team_sides in geometry_by_set.items()
        }
        report = {
            "method": "scout_serve_team_plus_unique_stable_pre_serve_baseline_track",
            "evaluation_only": True,
            "gesture_detector_or_associated_gesture_player_used_for_mapping": False,
            "lookback_seconds": SIDE_ANCHOR_LOOKBACK_SECONDS,
            "minimum_anchor_frames": SIDE_ANCHOR_MIN_FRAMES,
            "minimum_anchor_duration_seconds": SIDE_ANCHOR_MIN_DURATION_SECONDS,
            "maximum_projected_foot_y_range_m": SIDE_ANCHOR_MAX_FOOT_Y_RANGE_METERS,
            "maximum_age_of_last_track_row_seconds": SIDE_ANCHOR_MAX_LAST_OBSERVATION_AGE_SECONDS,
            "minimum_repeated_serve_anchors_per_team_set":
                SIDE_ANCHOR_MIN_REPEATED_SERVES_PER_TEAM_SET,
            "player_csv": str(player_csv.resolve()),
            "player_csv_sha256": sha256_file(player_csv),
            "player_csv_frame_timestamp_max_error_frames": max_alignment_error,
            "calibration": calibration_report,
            "track_side_to_camera_half": side_to_half,
            "mapping_by_set_team": camera_by_set,
            "per_set": set_reports,
            "n_non_doubtful_aligned_serve_rows": len(serve_rows),
            "n_unique_stable_baseline_anchors": sum(
                row.get("status") == "unique_stable_baseline_track" for row in anchor_rows),
            "n_unresolved_serve_anchors": sum(
                row.get("status") != "unique_stable_baseline_track" for row in anchor_rows),
        }
        return camera_by_set, anchor_rows, report, side_to_half

    touch_path = Path(TOUCH_CSV)
    if not touch_path.is_file():
        raise FileNotFoundError(f"CSV tocchi ground truth non trovato: {touch_path}")
    touch_source_hash = sha256_file(touch_path)
    if EVALUATION_DIR.exists():
        raise FileExistsError(f"Directory valutazione gia esistente: {EVALUATION_DIR}")
    touches = pd.read_csv(touch_path, encoding="utf-8-sig")
    required_touch_columns = {"video", "fotogramma", "tempo_s", "gesto", "dubbio"}
    missing_columns = required_touch_columns - set(touches.columns)
    if missing_columns:
        raise ValueError(f"Schema CSV tocchi non valido; colonne mancanti: {sorted(missing_columns)}")
    touches["_source_row"] = np.arange(len(touches), dtype=int) + 2
    touches["_frame"] = pd.to_numeric(touches["fotogramma"], errors="coerce")
    touches["_annotated_time"] = pd.to_numeric(touches["tempo_s"], errors="coerce")
    video_rows = touches[touches["video"].astype(str).str.strip().str.lower()
                         == SCOUT_VIDEO_NAME.lower()].copy()
    if video_rows.empty:
        raise ValueError(
            f"Nessuna annotazione per {VIDEO_PATH.name} in {touch_path.name}; "
            "verificare la selezione match/video. Nessuna metrica viene calcolata."
        )
    video_rows["_frame_time"] = video_rows["_frame"] / VIDEO_FPS
    video_rows["_alignment_error"] = (
        video_rows["_frame_time"] - video_rows["_annotated_time"]
    ).abs()
    alignment_tolerance = 0.5 / VIDEO_FPS + 0.00051
    video_rows["_aligned"] = (
        video_rows["_frame"].notna() & video_rows["_annotated_time"].notna()
        & (video_rows["_alignment_error"] <= alignment_tolerance)
    )
    video_rows["_doubt"] = video_rows["dubbio"].map(_flag)
    CALIBRATED_TRACK_SIDE_TO_CAMERA_HALF, CALIBRATION_SIDE_REPORT, _side_player_config = (
        _calibrated_track_side_map())
    if GROUND_TRUTH_TEAM_SIDE_BY_SET:
        ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET = {
            str(set_id): {str(team): side for team, side in teams.items()}
            for set_id, teams in GROUND_TRUTH_TEAM_SIDE_BY_SET.items()
        }
        SIDE_ANCHOR_ROWS = []
        SIDE_MAPPING_REPORT = {
            "method": "explicit_manual_team_side_by_set",
            "evaluation_only": True,
            "mapping_by_set_team": ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET,
            "calibration": CALIBRATION_SIDE_REPORT,
        }
    else:
        (ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET, SIDE_ANCHOR_ROWS,
         SIDE_MAPPING_REPORT, CALIBRATED_TRACK_SIDE_TO_CAMERA_HALF) = (
            _derive_camera_half_mapping(video_rows))
    if not PREDICTION_TRACK_SIDE_TO_CAMERA_HALF:
        PREDICTION_TRACK_SIDE_TO_CAMERA_HALF = dict(CALIBRATED_TRACK_SIDE_TO_CAMERA_HALF)
    in_clip = (
        video_rows["_frame"].ge(START_FRAME)
        & video_rows["_frame"].lt(END_FRAME_EXCLUSIVE)
    )
    clip_rows = video_rows[in_clip].copy()
    clip_rows["_label"] = clip_rows["gesto"].map(_label)
    aligned_rows = clip_rows[clip_rows["_aligned"]].copy()
    eligible_rows = aligned_rows[
        aligned_rows["_doubt"].eq(False) & aligned_rows["_label"].notna()
    ].copy()
    ground_truth = []
    for row in eligible_rows.to_dict("records"):
        frame_index = int(row["_frame"])
        ground_truth.append({
            "id": int(row["_source_row"]), "video": str(row["video"]).strip(),
            "frame": frame_index, "time": frame_index / VIDEO_FPS,
            "annotated_time": float(row["_annotated_time"]),
            "label": row["_label"], "team": str(row["squadra"]).strip() if "squadra" in row else None,
            "set": row.get("set"), "rally": row.get("scambio"),
            "camera_half": ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET.get(
                str(row.get("set", "")).strip(), {}).get(
                    str(row.get("squadra", "")).strip()) if "squadra" in row else None,
            "quality": row.get("valutazione"),
        })

    detections = pd.read_csv(DETECTIONS_CSV, encoding="utf-8-sig")
    episodes_df = pd.read_csv(EVENTS_CSV, encoding="utf-8-sig")
    required_detection_columns = {"episode_id", "frame_index", "timestamp_seconds",
                                  "confidence_raw", "class_raw", "class_contextual"}
    if required_detection_columns - set(detections.columns):
        raise ValueError("CSV detection privo dei campi necessari per validare timing e classi.")
    detections["_frame_numeric"] = pd.to_numeric(detections["frame_index"], errors="coerce")
    detections["_time_numeric"] = pd.to_numeric(detections["timestamp_seconds"], errors="coerce")
    expected_sampled_frame = (
        detections["_frame_numeric"].ge(START_FRAME)
        & detections["_frame_numeric"].lt(END_FRAME_EXCLUSIVE)
        & ((detections["_frame_numeric"] - START_FRAME) % FRAME_STRIDE == 0)
    )
    if not expected_sampled_frame.all():
        raise ValueError(
            "Il CSV detection include frame fuori dall'intervallo campionato configurato; "
            "verificare che gli output appartengano al run/video selezionato."
        )
    detection_alignment_error = (
        detections["_frame_numeric"] / VIDEO_FPS - detections["_time_numeric"]
    ).abs()
    bad_detection_times = (
        detections["_frame_numeric"].isna() | detections["_time_numeric"].isna()
        | (detection_alignment_error > alignment_tolerance)
    )
    if bad_detection_times.any():
        raise ValueError(
            f"{int(bad_detection_times.sum())} detection hanno frame/timestamp incoerenti; "
            "valutazione interrotta per evitare associazioni temporali errate."
        )
    episode_by_id = {
        str(row["episode_id"]): row for row in episodes_df.to_dict("records")
        if row.get("episode_id") is not None and not pd.isna(row.get("episode_id"))
    }
    predictions = []
    invalid_episode_rows = int(detections["episode_id"].isna().sum())
    for episode_id, group in detections.groupby("episode_id", sort=False, dropna=True):
        episode_id = str(episode_id)
        if episode_id not in episode_by_id:
            invalid_episode_rows += len(group)
            continue
        group = group.assign(
            _confidence=pd.to_numeric(group["confidence_raw"], errors="coerce")
        ).sort_values(["_confidence", "_frame_numeric"], ascending=[False, True])
        if group["_confidence"].notna().sum() == 0:
            invalid_episode_rows += len(group)
            continue
        peak = group.iloc[0]
        event = episode_by_id[episode_id]
        raw = _label(event.get("class_raw")) or _label(peak.get("class_raw"))
        context = (_label(event.get("class_corrected"))
                   or _label(event.get("class_contextual"))
                   or _label(peak.get("class_contextual")))
        contact_status = str(event.get("contact_evidence_status") or "")
        ball_position = str(peak.get("ball_position_type") or "")
        ball_distance = _finite(peak.get("ball_distance_px"))
        ball_confidence = _finite(peak.get("ball_confidence"))
        ball_warning, warning_distance = _ball_uncertainty_warning(
            ball_position, ball_distance, ball_confidence,
            '|'.join(str(value) for value in str(event.get("ball_evidence_types") or "").split('|') if value))
        included, exclusion = _wall_contact_candidate(raw, context, contact_status)
        frame_index = int(peak["_frame_numeric"])
        predictions.append({
            "id": episode_id, "episode_id": episode_id,
            "frame": frame_index, "time": frame_index / VIDEO_FPS,
            "class_raw": raw, "class_contextual": context,
            "confidence_raw": _finite(peak["confidence_raw"]),
            "contact_evidence_status": contact_status,
            "ball_position_type": ball_position,
            "ball_distance_px": ball_distance,
            "ball_confidence": ball_confidence,
            "ball_warning": ball_warning,
            "contact_candidate_included": included,
            "candidate_status": exclusion,
            "player_team_side_geometric": event.get("player_team_side_geometric"),
            "camera_half_prediction": PREDICTION_TRACK_SIDE_TO_CAMERA_HALF.get(
                str(event.get("player_team_side_geometric") or "")),
        })
    contact_predictions = predictions
    pairs = _one_to_one_time_match(ground_truth, contact_predictions, MATCH_TOLERANCE_BY_CLASS)
    unbiased_pairs = _one_to_one_time_match(
        ground_truth, contact_predictions, MATCH_TOLERANCE_BY_CLASS, prefer_wall_class=False)
    rally_window_result = _evaluate_rally_windows(
        video_rows, ground_truth, contact_predictions, VIDEO_FPS,
        START_SECONDS, END_SECONDS, RALLY_WINDOW_MARGIN_SECONDS,
        MATCH_TOLERANCE_BY_CLASS)
    matched_gt = {gi for gi, _ in pairs}
    matched_pred = {pi for _, pi in pairs}

    match_rows = []
    for gi, pi in pairs:
        gt, pred = ground_truth[gi], contact_predictions[pi]
        match_rows.append({
            "status": "temporal_match",
            "annotation_row": gt["id"], "annotation_frame": gt["frame"],
            "annotation_time_s": gt["annotated_time"], "canonical_touch_time_s": gt["time"],
            "gesture_truth": gt["label"], "team": gt["team"], "set": gt["set"],
            "camera_half_ground_truth": gt.get("camera_half"),
            "rally": gt["rally"], "gesture_quality": gt["quality"],
            "episode_id": pred["episode_id"], "event_frame": pred["frame"],
            "event_time_s": pred["time"], "time_delta_s_event_minus_touch": pred["time"] - gt["time"],
            "abs_time_delta_s": abs(pred["time"] - gt["time"]),
            "class_raw": pred["class_raw"], "class_contextual": pred["class_contextual"],
            "raw_class_correct": pred["class_raw"] == gt["label"],
            "context_class_correct": pred["class_contextual"] == gt["label"],
            "context_class_changed": pred["class_raw"] != pred["class_contextual"],
            "contact_evidence_status": pred["contact_evidence_status"],
            "ball_warning": pred.get("ball_warning"),
            "ball_position_type": pred.get("ball_position_type"),
            "ball_distance_px": pred.get("ball_distance_px"),
        })
    for gi, gt in enumerate(ground_truth):
        if gi not in matched_gt:
            match_rows.append({
                "status": "missed_annotated_touch", "annotation_row": gt["id"],
                "annotation_frame": gt["frame"], "annotation_time_s": gt["annotated_time"],
                "canonical_touch_time_s": gt["time"], "gesture_truth": gt["label"],
                "team": gt["team"], "set": gt["set"],
                "camera_half_ground_truth": gt.get("camera_half"),
                "rally": gt["rally"], "gesture_quality": gt["quality"],
            })
    for pi, pred in enumerate(contact_predictions):
        if pi not in matched_pred:
            match_rows.append({
                "status": "unmatched_contact_candidate", "episode_id": pred["episode_id"],
                "event_frame": pred["frame"], "event_time_s": pred["time"],
                "class_raw": pred["class_raw"], "class_contextual": pred["class_contextual"],
                "contact_evidence_status": pred["contact_evidence_status"],
            })
    for pred in predictions:
        if not pred["contact_candidate_included"]:
            match_rows.append({
                "status": pred["candidate_status"], "episode_id": pred["episode_id"],
                "event_frame": pred["frame"], "event_time_s": pred["time"],
                "class_raw": pred["class_raw"], "class_contextual": pred["class_contextual"],
                "contact_evidence_status": pred["contact_evidence_status"],
            })

    EVALUATION_DIR.mkdir(parents=True, exist_ok=False)
    pd.DataFrame(SIDE_ANCHOR_ROWS).to_csv(
        EVALUATION_DIR / "serve_side_anchors.csv", index=False, encoding="utf-8-sig")
    evaluation_valid = len(ground_truth) > 0
    metric_rows, confusion_files, class_rows_all = [], {}, []
    if evaluation_valid:
        contact_tp = len(pairs)
        contact_precision = _ratio(contact_tp, len(contact_predictions))
        contact_recall = _ratio(contact_tp, len(ground_truth))
        contact_report = {
            "metric": "contact_candidate_detection",
            "description": "one-to-one temporal event match, regardless of gesture label",
            "tp": contact_tp, "fp": len(contact_predictions) - contact_tp,
            "fn": len(ground_truth) - contact_tp,
            "precision": contact_precision, "recall": contact_recall,
            "f1": _f1(contact_precision, contact_recall),
            "n_ground_truth_touches": len(ground_truth),
            "n_contact_candidate_events": len(contact_predictions),
            "n_temporally_matched": len(pairs),
        }
        metric_rows.append(contact_report)
        for method, field in (("raw", "class_raw"), ("contextual", "class_contextual")):
            class_report, matrix, class_metrics = _class_report(
                pairs, ground_truth, contact_predictions, field)
            matrix.index.name = "gesture_truth"
            matrix.to_csv(EVALUATION_DIR / f"confusion_{method}.csv", encoding="utf-8-sig")
            confusion_files[method] = f"confusion_{method}.csv"
            class_report.update({
                "metric": f"classification_{method}_on_temporal_matches",
                "description": "classification only on temporally matched pairs; same pairs for raw/contextual",
                "matching_coverage_of_eligible_touches": _ratio(len(pairs), len(ground_truth)),
                "n_eligible_touches": len(ground_truth),
            })
            metric_rows.append(class_report)
            class_metrics.insert(0, "method", method)
            class_rows_all.append(class_metrics)
            correct = class_report["n_class_correct"]
            joint_fp, joint_fn = len(contact_predictions) - correct, len(ground_truth) - correct
            joint_precision = _ratio(correct, correct + joint_fp)
            joint_recall = _ratio(correct, correct + joint_fn)
            metric_rows.append({
                "metric": f"joint_time_and_class_{method}",
                "description": "only a temporal match with correct label is TP; wrong-class match is both FP and FN",
                "tp": correct, "fp": joint_fp, "fn": joint_fn,
                "precision": joint_precision, "recall": joint_recall,
                "f1": _f1(joint_precision, joint_recall),
                "n_ground_truth_touches": len(ground_truth),
                "n_contact_candidate_events": len(contact_predictions),
            })
        changed_count = sum(p["class_raw"] != p["class_contextual"] for p in contact_predictions)
        matched_changed = sum(
            contact_predictions[pi]["class_raw"] != contact_predictions[pi]["class_contextual"]
            for _, pi in pairs)
        for row in metric_rows:
            if row["metric"] == "classification_contextual_on_temporal_matches":
                row["changed_context_fraction_all_candidate_events"] = _ratio(
                    changed_count, len(contact_predictions))
                row["changed_context_fraction_on_matched_events"] = _ratio(
                    matched_changed, len(pairs))
                row["unchanged_context_fraction_all_candidate_events"] = _ratio(
                    len(contact_predictions) - changed_count, len(contact_predictions))
        report_status = "evaluated"
    else:
        report_status = "no_evaluable_touch_annotations_in_clip"
        metric_rows = [{
            "metric": name, "status": report_status,
            "precision": None, "recall": None, "f1": None,
            "accuracy_on_temporal_matches": None,
        } for name in (
            "contact_candidate_detection", "classification_raw_on_temporal_matches",
            "classification_contextual_on_temporal_matches",
            "joint_time_and_class_raw", "joint_time_and_class_contextual"
        )]
        print("Nessuna annotazione non-dubbia/allineata con classe nota nella clip: "
              "nessuna metrica di accuratezza viene calcolata.")

    side_metric_rows = [dict(row, camera_half="overall") for row in metric_rows]
    side_class_rows = []
    if evaluation_valid:
        for side in ("near", "far"):
            contact_side = _camera_half_contact_report(
                side, ground_truth, contact_predictions, pairs)
            side_metric_rows.append({
                key: value for key, value in contact_side.items() if key != "matched_pairs"
            } | {"metric": "contact_candidate_detection_by_camera_half"})
            side_pairs = contact_side["matched_pairs"]
            side_gt_count = contact_side["n_ground_truth_touches"]
            for method, field in (("raw", "class_raw"), ("contextual", "class_contextual")):
                side_report, side_matrix, side_per_class = _class_report(
                    side_pairs, ground_truth, contact_predictions, field)
                side_matrix.index.name = "gesture_truth"
                matrix_name = f"confusion_{method}_{side}.csv"
                side_matrix.to_csv(EVALUATION_DIR / matrix_name, encoding="utf-8-sig")
                confusion_files[f"{method}_{side}"] = matrix_name
                side_report.update({
                    "camera_half": side, "status": contact_side["status"],
                    "metric": f"classification_{method}_on_temporal_matches_by_camera_half",
                    "n_eligible_touches": side_gt_count,
                    "matching_coverage_of_eligible_touches": _ratio(
                        len(side_pairs), side_gt_count),
                })
                side_metric_rows.append(side_report)
                side_per_class.insert(0, "method", method)
                side_per_class.insert(1, "camera_half", side)
                side_class_rows.append(side_per_class)
                correct_side = side_report["n_class_correct"]
                joint_fp_known = (
                    len(side_pairs) - correct_side + contact_side["fp_assigned_to_side"]
                )
                joint_precision_side = (
                    _ratio(correct_side, correct_side + joint_fp_known)
                    if contact_side["unassigned_unmatched_predictions"] == 0 else None
                )
                joint_recall_side = _ratio(correct_side, side_gt_count)
                side_metric_rows.append({
                    "camera_half": side,
                    "status": contact_side["status"],
                    "metric": f"joint_time_and_class_{method}_by_camera_half",
                    "tp": correct_side, "fp": joint_fp_known,
                    "fn": side_gt_count - correct_side,
                    "precision": joint_precision_side, "recall": joint_recall_side,
                    "f1": _f1(joint_precision_side, joint_recall_side),
                    "n_ground_truth_touches": side_gt_count,
                    "matching_coverage_of_eligible_touches": _ratio(len(side_pairs), side_gt_count),
                    "unassigned_unmatched_predictions": contact_side["unassigned_unmatched_predictions"],
                })
    else:
        for side in ("near", "far"):
            for metric_name in (
                "contact_candidate_detection_by_camera_half",
                "classification_raw_on_temporal_matches_by_camera_half",
                "classification_contextual_on_temporal_matches_by_camera_half",
                "joint_time_and_class_raw_by_camera_half",
                "joint_time_and_class_contextual_by_camera_half",
            ):
                side_metric_rows.append({
                    "camera_half": side, "metric": metric_name,
                    "status": "no_evaluable_touch_annotations_in_clip",
                    "precision": None, "recall": None, "f1": None,
                })

    pd.DataFrame(match_rows).to_csv(
        EVALUATION_DIR / "event_matches.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(predictions).to_csv(
        EVALUATION_DIR / "candidate_events.csv", index=False, encoding="utf-8-sig")
    if class_rows_all:
        pd.concat(class_rows_all, ignore_index=True).to_csv(
            EVALUATION_DIR / "per_class_metrics.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(metric_rows).to_csv(
        EVALUATION_DIR / "metrics.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(side_metric_rows).to_csv(
        EVALUATION_DIR / "metrics_by_camera_half.csv", index=False, encoding="utf-8-sig")
    if side_class_rows:
        pd.concat(side_class_rows, ignore_index=True).to_csv(
            EVALUATION_DIR / "per_class_metrics_by_camera_half.csv",
            index=False, encoding="utf-8-sig")

    rally_window_result["raw_confusion"].index.name = "gesture_truth"
    rally_window_result["contextual_confusion"].index.name = "gesture_truth"
    rally_window_result["raw_confusion"].to_csv(
        EVALUATION_DIR / "rally_confusion_raw.csv", encoding="utf-8-sig")
    rally_window_result["contextual_confusion"].to_csv(
        EVALUATION_DIR / "rally_confusion_contextual.csv", encoding="utf-8-sig")
    pd.DataFrame(rally_window_result["windows"]).drop(
        columns=["key"], errors="ignore").to_csv(
            EVALUATION_DIR / "rally_windows.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(rally_window_result["metrics_by_rally"]).to_csv(
        EVALUATION_DIR / "rally_window_metrics.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(rally_window_result["match_rows"]).to_csv(
        EVALUATION_DIR / "rally_window_matches.csv", index=False, encoding="utf-8-sig")
    pd.concat([
        rally_window_result["raw_per_class"].assign(method="raw"),
        rally_window_result["contextual_per_class"].assign(method="contextual"),
    ], ignore_index=True).to_csv(
        EVALUATION_DIR / "rally_per_class_metrics.csv",
        index=False, encoding="utf-8-sig")
    with (EVALUATION_DIR / "rally_window_report.json").open(
            "w", encoding="utf-8") as stream:
        json.dump(rally_window_result["summary"], stream, ensure_ascii=False,
                  indent=2, allow_nan=False)

    max_alignment_error = _finite(video_rows["_alignment_error"].max())
    report = {
        "status": report_status,
        "evaluation_scope": "single configured video clip; not cross-validation",
        "ground_truth_role": "evaluation only; never supplied to prediction or contextual priors",
        "matching_policy": "maximum pairs, then contextual wall matches, then minimum absolute time difference; chronological order preserved",
        "matching_class_bias_warning": "Wall labels are used to select pairs; classification accuracy can be inflated. Compare the class-blind temporal baseline.",
        "class_blind_temporal_baseline": {
            "raw": _class_report(unbiased_pairs, ground_truth, contact_predictions, "class_raw")[0],
            "contextual": _class_report(unbiased_pairs, ground_truth, contact_predictions, "class_contextual")[0],
        },
        "touch_csv": str(touch_path.resolve()),
        "touch_csv_sha256": touch_source_hash,
        "touch_csv_rows_for_video": int(len(video_rows)),
        "video": str(VIDEO_PATH.resolve()),
        "video_basename_match": SCOUT_VIDEO_NAME,
        "video_basename_actual": VIDEO_PATH.name,
        "video_basename_alias_confirmed_by_user": SCOUT_VIDEO_NAME != VIDEO_PATH.name,
        "video_fps": float(VIDEO_FPS),
        "interval_seconds_half_open": [float(START_SECONDS), float(END_SECONDS)],
        "interval_frame_half_open": [int(START_FRAME), int(END_FRAME_EXCLUSIVE)],
        "frame_sampling_stride": int(FRAME_STRIDE),
        "match_tolerance_seconds": float(MATCH_TOLERANCE_SECONDS),
        "match_tolerance_seconds_by_ground_truth_class": {
            key: float(value) for key, value in MATCH_TOLERANCE_BY_CLASS.items()
        },
        "annotation_frame_time_alignment_tolerance_seconds": alignment_tolerance,
        "annotation_max_abs_frame_time_error_seconds": max_alignment_error,
        "annotation_rows_in_interval": int(len(clip_rows)),
        "annotation_rows_annotation_time_boundary_disagreement": int(
            (clip_rows["_annotated_time"].lt(START_SECONDS)
             | clip_rows["_annotated_time"].ge(END_SECONDS)).sum()),
        "annotation_rows_frame_time_misaligned_excluded": int((~clip_rows["_aligned"]).sum()),
        "annotation_rows_doubtful_excluded": int(clip_rows["_doubt"].ne(False).sum()),
        "annotation_rows_unknown_class_excluded": int(clip_rows["_label"].isna().sum()),
        "annotation_rows_eligible": int(len(ground_truth)),
        "annotation_rows_with_unknown_camera_half": sum(
            gt.get("camera_half") not in ("near", "far") for gt in ground_truth),
        "ground_truth_team_side_by_set": ACTIVE_GROUND_TRUTH_TEAM_SIDE_BY_SET,
        "camera_half_mapping": SIDE_MAPPING_REPORT,
        "camera_half_mapping_unknown_eligible_touches": sum(
            gt.get("camera_half") not in ("near", "far") for gt in ground_truth),
        "prediction_track_side_to_camera_half": PREDICTION_TRACK_SIDE_TO_CAMERA_HALF,
        "annotation_quality_values_kept_not_used_as_labels": (
            sorted(str(x) for x in clip_rows["valutazione"].dropna().unique())
            if "valutazione" in clip_rows else []
        ),
        "n_all_gesture_episodes": int(len(predictions)),
        "n_contact_candidate_events": int(len(contact_predictions)),
        "n_wall_predictions": sum(1 for p in predictions if p["class_raw"] == "muro" or p["class_contextual"] == "muro"),
        "n_wall_predictions_legacy_excluded": sum(1 for p in predictions if not _legacy_wall_contact_candidate(p["class_raw"], p["class_contextual"], p["contact_evidence_status"])[0]),
        "n_wall_legacy_excluded_matched_correct": sum(1 for gi, pi in pairs if (contact_predictions[pi]["class_raw"] == "muro" or contact_predictions[pi]["class_contextual"] == "muro") and ground_truth[gi]["label"] == "muro" and not _legacy_wall_contact_candidate(contact_predictions[pi]["class_raw"], contact_predictions[pi]["class_contextual"], contact_predictions[pi]["contact_evidence_status"])[0]),
        "n_wall_legacy_excluded_matched_error": sum(1 for gi, pi in pairs if (contact_predictions[pi]["class_raw"] == "muro" or contact_predictions[pi]["class_contextual"] == "muro") and ground_truth[gi]["label"] != "muro" and not _legacy_wall_contact_candidate(contact_predictions[pi]["class_raw"], contact_predictions[pi]["class_contextual"], contact_predictions[pi]["contact_evidence_status"])[0]),
        "n_wall_legacy_excluded_unmatched": sum(1 for pi, p in enumerate(contact_predictions) if pi not in matched_pred and (p["class_raw"] == "muro" or p["class_contextual"] == "muro") and not _legacy_wall_contact_candidate(p["class_raw"], p["class_contextual"], p["contact_evidence_status"])[0]),
        "n_ball_warning_far_observed": sum(1 for p in predictions if p.get("ball_warning") == "palla_osservata_potenzialmente_lontana"),
        "n_ball_warning_only_kalman": sum(1 for p in predictions if p.get("ball_warning") == "solo_palla_kalman_incerta"),
        "n_ball_warning_no_evidence": sum(1 for p in predictions if p.get("ball_warning") in ("nessuna_evidenza_palla", "palla_assente_o_lontana_senza_evidenza_vicina")),
        "n_invalid_episode_detections_excluded": int(invalid_episode_rows),
        "n_temporal_matches": int(len(pairs)),
        "n_missed_annotated_touches": int(len(ground_truth) - len(matched_gt)),
        "n_unmatched_contact_candidates": int(len(contact_predictions) - len(matched_pred)),
        "metrics": metric_rows,
        "confusion_matrices": confusion_files,
        "outputs": {
            "matches": "event_matches.csv",
            "candidate_events": "candidate_events.csv",
            "metrics": "metrics.csv",
            "metrics_by_camera_half": "metrics_by_camera_half.csv",
            "per_class_metrics": "per_class_metrics.csv" if class_rows_all else None,
            "per_class_metrics_by_camera_half": (
                "per_class_metrics_by_camera_half.csv" if side_class_rows else None),
            "serve_side_anchors": "serve_side_anchors.csv",
            "rally_window_report": "rally_window_report.json",
            "rally_windows": "rally_windows.csv",
            "rally_window_metrics": "rally_window_metrics.csv",
            "rally_window_matches": "rally_window_matches.csv",
            "rally_confusion_raw": "rally_confusion_raw.csv",
            "rally_confusion_contextual": "rally_confusion_contextual.csv",
            "rally_per_class_metrics": "rally_per_class_metrics.csv",
        },
        "rally_window_evaluation": rally_window_result["summary"],
        "limits": [
            "Event candidates are not proof of ball contact; a nearby observed ball is still unconfirmed.",
            "Walls are included regardless of ball evidence; warning does not prove or disprove contact.",
            "Scout touch annotations omit gestures without contact, including some blocks; this is not complete action recognition evaluation.",
            "The touch CSV has team but no camera-half field; mapping uses repeated, stable pre-serve player-foot anchors and the confirmed calibration.",
            "The player linked to a gesture prediction is never used as ground-truth side evidence.",
            "Jumping or unstable/missing baseline tracks are not anchors; uncertain sets remain unassigned.",
            "Near/far contact precision is null when unmatched candidate events have no independently mapped tracker side.",
            "Potential overlap between the external training dataset and this match is unverified.",
            "Results apply only to this clip and tolerance; no cross-validation or generalization estimate.",
        ],
    }
    with (EVALUATION_DIR / "touch_evaluation_report.json").open("w", encoding="utf-8") as stream:
        json.dump(report, stream, ensure_ascii=False, indent=2, allow_nan=False)
    print("Test sintetici: matching tocchi, FP/FN, classe, contesto, ancore lato stabili/ambigue "
          "e metriche near/far superati.")
    print("Valutazione:", report_status, "| tocchi", len(ground_truth),
          "| candidati contatto", len(contact_predictions), "| match", len(pairs))
    print("Output valutazione:", EVALUATION_DIR)
    if evaluation_valid:
        print(pd.DataFrame(metric_rows).to_string(index=False))


## 6. Valutazione opzionale su tocchi annotati


Il matching mantiene l'ordine temporale, massimizza le coppie, privilegia a parita i muri contestuali abbinati a muri scout, poi minimizza lo scarto. Questo usa le etichette scout e puo gonfiare le metriche: confrontare il baseline temporale senza priorita di classe nel report. Le stesse coppie sono usate per raw e contestuale.

Sono esportate tre viste: (1) precision/recall/F1 di **eventi candidati di contatto** abbinati temporalmente, indipendentemente dalla classe; (2) accuratezza, matrice di confusione e precision/recall/F1 per classe soltanto sulle coppie temporalmente abbinate, con n e copertura dei tocchi annotati; (3) metrica congiunta tempo+classe, in cui una coppia con label sbagliata conta come FP e FN e gli eventi/tocchi non abbinati sono penalizzati. La classificazione include anche episodi senza correzione contestuale; il report espone quota corretta, invariata e modificata.

Un gesto muro non prova un tocco. I muri restano candidati al matching indipendentemente dall'evidenza palla; ball_warning distingue palla osservata lontana, solo Kalman, assente e assenza di evidenza. La distanza non cambia classe o sopprime la predizione. Il report confronta la logica legacy con la nuova valutazione. Se mancano annotazioni valutabili nella clip, le metriche restano nulle con stato esplicito, non diventano accuratezza perfetta.


La valutazione, quando eseguita, salva `touch_evaluation_report.json`, `metrics.csv`, `per_class_metrics.csv`, matrici raw/contestuale, candidati e matching in una directory univoca `*_touch_eval_*`; non modifica annotazioni o output del run.

In [ ]:
if INFERENCE_COMPLETED:
    from IPython.display import display, Image as DisplayImage, Video
    display(DisplayImage(filename=str(PREVIEW_PATH), width=960))
    detection_preview = pd.read_csv(DETECTIONS_CSV)
    episode_preview = pd.read_csv(EVENTS_CSV)
    print('Detection frame-per-frame:', len(detection_preview))
    display(detection_preview.head(12))
    print('Episodi deduplicati:', len(episode_preview))
    display(episode_preview.head(12))
    print('Report limiti/copertura:')
    display(pd.DataFrame([coverage]).T.rename(columns={0:'valore'}))
    if output_video_path is not None:
        display(Video(filename=str(output_video_path), embed=False, html_attributes='controls'))
    else:
        print('Video contestuale non disponibile; vedere il report per il motivo.')
else:
    print('Anteprima non disponibile perche RUN_INFERENCE=False; nessun output video e stato creato.')